In [ ]:
# ================================================================
# T1.4 — XAI SANITY CHECKS
# CELL 1 — CONFIGURATION
# ================================================================

import os
import sys
import json
import time
import shutil
import hashlib
import importlib.util

from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as TF


# ------------------------------------------------
# Kaggle paths
# ------------------------------------------------

ROOT = Path("/kaggle")

INPUT = ROOT / "input"
WORK = ROOT / "working"
TEMP = ROOT / "temp"

PROJECT = WORK / "isic_final_project"
TRAINED_MODELS = WORK / "trained_models_readonly"

CACHE = TEMP / "isic_cache"

RESULTS = WORK / "t1_4_sanity_results"

RESULTS.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------
# Device
# ------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("T1.4 XAI SANITY CHECKS")
print("=" * 70)

print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU count:",
        torch.cuda.device_count()
    )

print(
    "INPUT:",
    INPUT
)

print(
    "WORK:",
    WORK
)

print(
    "TEMP:",
    TEMP
)

print(
    "RESULTS:",
    RESULTS
)

In [ ]:
# ================================================================
# CELL 2 — LOCATE LOCKED PROJECT
# ================================================================

from collections import deque


def find_by_markers(
    root,
    required_subdirs,
    max_depth=6
):

    root = Path(root)

    if not root.exists():
        return None

    required = set(
        required_subdirs
    )

    q = deque(
        [(root, 0)]
    )

    while q:

        d, depth = q.popleft()

        try:

            entries = list(
                os.scandir(d)
            )

        except (
            PermissionError,
            FileNotFoundError,
            NotADirectoryError
        ):

            continue

        names = {
            e.name
            for e in entries
            if e.is_dir()
        }

        if required <= names:

            return d

        if depth < max_depth:

            for e in entries:

                if (
                    e.is_dir()
                    and not e.name.startswith(".")
                ):

                    q.append(
                        (
                            Path(e.path),
                            depth + 1
                        )
                    )

    return None


PROJECT_MARKERS = {
    "step0I_locked_clean_population",
    "step1F_lesion_level_rho_assignments",
}


project_source = find_by_markers(
    INPUT,
    PROJECT_MARKERS
)

print(
    "Project source:",
    project_source
)

assert project_source is not None, (
    "Locked project not found. "
    "Check that the project dataset was added."
)

In [ ]:
# ================================================================
# CELL 3 — RESTORE LOCKED PROJECT
# ================================================================

if not PROJECT.exists():

    print(
        "Copying locked project..."
    )

    shutil.copytree(
        project_source,
        PROJECT
    )

else:

    print(
        "Project already restored:"
    )

    print(
        PROJECT
    )


assert (
    PROJECT
    / "step2W_analysis_code_freeze"
    / "analysis_pipeline.py"
).exists()

print(
    "✓ frozen analysis_pipeline.py found"
)

In [ ]:
# ================================================================
# CELL 4 — VERIFY FROZEN ANALYSIS CODE
# ================================================================

def sha_file(
    path,
    chunk=1 << 22
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(chunk),
            b""
        ):

            h.update(block)

    return h.hexdigest()


ANALYSIS_MODULE = (
    PROJECT
    / "step2W_analysis_code_freeze"
    / "analysis_pipeline.py"
)

HASH_RECORD = (
    PROJECT
    / "step2W_analysis_code_freeze"
    / "ANALYSIS_CODE_FINAL_HASH.json"
)


assert ANALYSIS_MODULE.exists()
assert HASH_RECORD.exists()


expected_sha = json.load(
    open(HASH_RECORD)
)["sha256"]


actual_sha = sha_file(
    ANALYSIS_MODULE
)


print(
    "Expected SHA256:",
    expected_sha
)

print(
    "Actual SHA256:",
    actual_sha
)


assert actual_sha == expected_sha, (
    "FROZEN ANALYSIS CODE HASH MISMATCH!"
)

print(
    "✓ analysis_pipeline.py hash verified"
)

In [ ]:
# ================================================================
# CELL 5 — LOAD FROZEN PRODUCTION PIPELINE
# ================================================================

spec = importlib.util.spec_from_file_location(
    "analysis_pipeline",
    ANALYSIS_MODULE
)

ap = importlib.util.module_from_spec(
    spec
)

spec.loader.exec_module(ap)

print(
    "✓ frozen analysis_pipeline imported"
)

print(
    "ig_enrichment:",
    ap.ig_enrichment
)

In [ ]:
# ================================================================
# CELL 6 — Inspect frozen production XAI configuration
# ================================================================

print("=== FROZEN PRODUCTION XAI CONFIG ===")

for name in [
    "IG_STEPS",
    "IG_RETRY_STEPS",
    "IG_SIGMA",
    "IG_TOL_REL",
    "IG_TAU_ABS",
    "XAI_EPS",
    "_MEAN",
    "_STD",
]:
    print(f"{name:20s} =", getattr(ap, name, "<NOT FOUND>"))

print("\n=== ig_enrichment signature ===")
import inspect
print(inspect.signature(ap.ig_enrichment))

print("\n=== ig_enrichment source ===")
print(inspect.getsource(ap.ig_enrichment))

In [ ]:
# ================================================================
# CELL 7 — Locate locked production artifacts
# ================================================================

from pathlib import Path

print("=" * 70)
print("LOCKED PRODUCTION ARTIFACT DISCOVERY")
print("=" * 70)

# ------------------------------------------------
# 1. XAI subset
# ------------------------------------------------
print("\n[1] XAI SUBSET CANDIDATES")

xai_hits = list(Path("/kaggle/input").rglob("LOCKED_XAI_SUBSET.csv"))

for p in xai_hits:
    print(" ", p)

print("Count:", len(xai_hits))


# ------------------------------------------------
# 2. 224x224 cache
# ------------------------------------------------
print("\n[2] CACHE CANDIDATES")

for pattern in ["clean_224.npy", "hair_224.npy", "cache_index.csv"]:
    hits = list(Path("/kaggle/input").rglob(pattern))
    print(f"\n{pattern}: {len(hits)}")
    for p in hits[:20]:
        print(" ", p)


# ------------------------------------------------
# 3. Production analysis hash
# ------------------------------------------------
print("\n[3] ANALYSIS CODE HASH")

hash_hits = list(
    Path("/kaggle/input").rglob("ANALYSIS_CODE_FINAL_HASH.json")
)

for p in hash_hits:
    print(" ", p)

print("Count:", len(hash_hits))


# ------------------------------------------------
# 4. GAP decision
# ------------------------------------------------
print("\n[4] GAP DECISION")

gap_hits = list(Path("/kaggle/input").rglob("GAP_DECISION.json"))

for p in gap_hits:
    print(" ", p)

print("Count:", len(gap_hits))


# ------------------------------------------------
# 5. Checkpoint files
# ------------------------------------------------
print("\n[5] CHECKPOINT FILES")

ckpt_hits = []

for ext in ["*.pt", "*.pth", "*.ckpt"]:
    ckpt_hits.extend(Path("/kaggle/input").rglob(ext))

print("Total checkpoint-like files:", len(ckpt_hits))

for p in sorted(ckpt_hits)[:100]:
    print(" ", p)


# ------------------------------------------------
# 6. Training directories
# ------------------------------------------------
print("\n[6] TRAINING-RUN DIRECTORIES")

for name in ["training_runs", "kaggle_records"]:
    hits = list(Path("/kaggle/input").rglob(name))
    print(f"\n{name}:")
    for p in hits:
        print(" ", p)

In [ ]:
# ================================================================
# CELL 8 — Locate production cache-building / XAI preparation code
# ================================================================

from pathlib import Path

PROJECT = Path("/kaggle/working/isic_final_project")

print("=" * 70)
print("SEARCHING LOCKED PROJECT FOR CACHE / XAI PIPELINE")
print("=" * 70)

# Show relevant files inside the frozen project
patterns = [
    "*analysis*.py",
    "*cache*.py",
    "*xai*.py",
    "*hair*.py",
    "*mask*.py",
]

seen = set()

for pattern in patterns:
    hits = sorted(PROJECT.rglob(pattern))
    for p in hits:
        if p not in seen:
            seen.add(p)
            print(p.relative_to(PROJECT))

print("\nTotal relevant files:", len(seen))

In [ ]:

# ================================================================
# CELL 9 — Inspect the locked analysis pipeline's public symbols
# ================================================================

import inspect

print("=" * 70)
print("ANALYSIS PIPELINE SYMBOLS")
print("=" * 70)

symbols = sorted(
    name for name in dir(ap)
    if not name.startswith("__")
)

for name in symbols:
    obj = getattr(ap, name)

    if callable(obj):
        try:
            sig = inspect.signature(obj)
        except Exception:
            sig = "(signature unavailable)"

        print(f"{name}{sig}")
    else:
        if isinstance(obj, (str, int, float, bool)):
            print(f"{name} = {obj}")

In [ ]:
# ================================================================
# CELL 10 — Find all functions in frozen analysis_pipeline
#          related to image/cache/mask/model preparation
# ================================================================

import inspect

keywords = [
    "cache",
    "mask",
    "hair",
    "image",
    "model",
    "checkpoint",
    "xai",
    "load",
    "generate",
]

for name in sorted(dir(ap)):
    if name.startswith("__"):
        continue

    obj = getattr(ap, name)

    if callable(obj):
        lname = name.lower()

        if any(k in lname for k in keywords):
            print("\n" + "=" * 70)
            print(name)
            print("=" * 70)

            try:
                print(inspect.signature(obj))
                print(inspect.getsource(obj))
            except Exception as e:
                print("Could not retrieve source:", repr(e))

In [ ]:
# ================================================================
# CELL 11B — EXACT PRODUCTION DATA ACCESS LAYER
#
# Corrected:
# - Does NOT require the previous notebook's 216-row working file.
# - Uses the persistent locked 500-image XAI subset.
# - Does NOT regenerate or alter any primary experiment artifact.
# ================================================================

import os
import json
import hashlib
import types
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision

print("=" * 70)
print("CELL 11B — EXACT PRODUCTION DATA ACCESS LAYER")
print("=" * 70)

INPUT = Path("/kaggle/input")

# ----------------------------------------------------------------
# 1. Locked project
# ----------------------------------------------------------------

PROJECT_CANDIDATES = [
    INPUT / "notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project",
    INPUT / "notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/isic_final_project",
    INPUT / "notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project",
    INPUT / "datasets/tejasseshadriiiii/isic-final-project",
]

PROJECT_SOURCE = next(
    (p for p in PROJECT_CANDIDATES if p.exists()),
    None
)

assert PROJECT_SOURCE is not None, "Locked project source not found."

print("Project source:")
print(PROJECT_SOURCE)


# ----------------------------------------------------------------
# 2. Training root
# ----------------------------------------------------------------

TRAINING_CANDIDATES = [
    INPUT / "notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs",
    INPUT / "notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs",
]

TRAINING_ROOT = next(
    (p for p in TRAINING_CANDIDATES if p.exists()),
    None
)

assert TRAINING_ROOT is not None, "Training root not found."

print("Training root:")
print(TRAINING_ROOT)


# ----------------------------------------------------------------
# 3. Frozen hair generator
# ----------------------------------------------------------------

GEN_PATH = (
    PROJECT_SOURCE
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

assert GEN_PATH.exists(), (
    f"Frozen hair generator not found:\n{GEN_PATH}"
)

print("Frozen generator:")
print(GEN_PATH)


# ----------------------------------------------------------------
# 4. Safely load frozen generator definitions
# ----------------------------------------------------------------

source = GEN_PATH.read_text(encoding="utf-8")
tree = ast.parse(source)

safe_nodes = []

for node in tree.body:

    if isinstance(node, (ast.Import, ast.ImportFrom)):
        safe_nodes.append(node)

    elif isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
        )
    ):
        safe_nodes.append(node)

    elif isinstance(node, ast.Assign):
        try:
            ast.literal_eval(node.value)
            safe_nodes.append(node)
        except Exception:
            pass

    elif isinstance(node, ast.AnnAssign):
        if node.value is not None:
            try:
                ast.literal_eval(node.value)
                safe_nodes.append(node)
            except Exception:
                pass


module_ast = ast.Module(
    body=safe_nodes,
    type_ignores=[]
)

ast.fix_missing_locations(module_ast)

generator = types.ModuleType(
    "frozen_hair_generator"
)

generator.__file__ = str(GEN_PATH)

exec(
    compile(
        module_ast,
        str(GEN_PATH),
        "exec"
    ),
    generator.__dict__
)

assert hasattr(generator, "generate_hair_mask")
assert hasattr(generator, "apply_hair")

assert generator.GLOBAL_SEED == 20260919
assert generator.PRIMARY_SEVERITY == "medium"

print("✓ generate_hair_mask() found")
print("✓ apply_hair() found")
print("✓ GLOBAL_SEED =", generator.GLOBAL_SEED)
print("✓ PRIMARY_SEVERITY =", generator.PRIMARY_SEVERITY)


# ----------------------------------------------------------------
# 5. Locate the AUTHORITATIVE locked XAI subset
# ----------------------------------------------------------------

XAI_SUBSET_CANDIDATES = [
    PROJECT_SOURCE
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv",

    INPUT
    / "notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/isic_final_project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv",

    INPUT
    / "notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv",

    INPUT
    / "notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv",

    INPUT
    / "datasets/tejasseshadriiiii/isic-final-project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv",
]

XAI_SUBSET = next(
    (p for p in XAI_SUBSET_CANDIDATES if p.exists()),
    None
)

assert XAI_SUBSET is not None, (
    "Authoritative LOCKED_XAI_SUBSET.csv not found."
)

print()
print("Locked XAI subset:")
print(XAI_SUBSET)


# ----------------------------------------------------------------
# 6. Inspect the locked subset
# ----------------------------------------------------------------

xai_df = pd.read_csv(XAI_SUBSET)

print()
print("XAI subset shape:", xai_df.shape)
print("XAI subset columns:")
print(xai_df.columns.tolist())

print()
print(xai_df.head())

assert len(xai_df) == 500, (
    f"Expected locked XAI subset size 500, got {len(xai_df)}"
)

assert "image" in xai_df.columns, (
    "LOCKED_XAI_SUBSET.csv does not contain an image column."
)

assert xai_df["image"].nunique() == 500, (
    "Locked XAI subset is not image-unique."
)

print()
print("✓ Exactly 500 locked XAI observations")
print("✓ 500 unique images")


# ----------------------------------------------------------------
# 7. Confirm no dependence on the obsolete 216-row artifact
# ----------------------------------------------------------------

print()
print("=" * 70)
print("216-ROW T1.4 WORKING ARTIFACT")
print("=" * 70)

print(
    "Not required for the new sanity checks."
)

print(
    "The previous 216-row resolved sample belonged to the "
    "previous notebook's /kaggle/working filesystem."
)

print(
    "✓ No regeneration of the old 216-row sample will be performed."
)


# ----------------------------------------------------------------
# 8. Locate an existing complete cache
# ----------------------------------------------------------------

CACHE_CANDIDATES = [
    Path("/kaggle/temp/isic_cache"),
    Path("/kaggle/working/isic_cache"),
]

CACHE = next(
    (
        p for p in CACHE_CANDIDATES
        if (
            (p / "clean_224.npy").exists()
            and
            (p / "hair_224.npy").exists()
            and
            (p / "mask_area.npy").exists()
            and
            (p / "cache_index.csv").exists()
        )
    ),
    None
)

print()
print("=" * 70)
print("CACHE STATUS")
print("=" * 70)

if CACHE is None:
    print("Complete cache not currently present.")
    print("Next step: deterministic production-faithful cache rebuild.")
else:
    print("Complete cache found:")
    print(CACHE)

In [ ]:
# ================================================================
# CELL 1 — CONFIG (ANALYSIS NOTEBOOK)
# ================================================================

AUTO_PIN_VERSIONS = globals().get("AUTO_PIN_VERSIONS", True)
ALLOW_VERSION_DEVIATION = globals().get("ALLOW_VERSION_DEVIATION", False)
N_PROC_OVERRIDE = globals().get("N_PROC_OVERRIDE", 4)
MAX_MODELS_THIS_RUN = globals().get("MAX_MODELS_THIS_RUN", None)
DEADLINE_HOURS = globals().get("DEADLINE_HOURS", 11.0)
EST_MODEL_HOURS = 0.15

print("✓ analysis-notebook config loaded")

In [ ]:
# ================================================================
# CELL 2 — HELPERS
# ================================================================

import os, sys, json, time, shutil, hashlib, subprocess, platform
from pathlib import Path
from collections import deque
from datetime import datetime, timezone

ROOT = Path(os.environ.get("KAGGLE_TEST_ROOT", "/kaggle"))
INPUT, WORK = ROOT / "input", ROOT / "working"
TEMP = ROOT / "temp"

TEMP.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

PROJECT = WORK / "isic_final_project"
CACHE = TEMP / "isic_cache"
RECORDS = WORK / "analysis_records"
RESULTS = WORK / "results"
TRAINED_MODELS = WORK / "trained_models_readonly"
LOCAL_TMP = TEMP / "tmp_work"

for d in (RECORDS, RESULTS, LOCAL_TMP):
    d.mkdir(parents=True, exist_ok=True)

T0 = time.time()

def utc():
    return datetime.now(timezone.utc).isoformat()

def elapsed_h():
    return (time.time() - T0) / 3600

def log(msg):
    print(f"[{elapsed_h():5.2f} h] {msg}", flush=True)

def sha_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def J(rel):
    return json.load(open(PROJECT / rel))

def save_record(name, obj):
    with open(RECORDS / name, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=str)

def find_by_markers(root, required_subdirs, max_depth=6):
    root = Path(root)
    if not root.exists():
        return None

    required = set(required_subdirs)
    q = deque([(root, 0)])

    while q:
        d, depth = q.popleft()

        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue

        names = {e.name for e in entries if e.is_dir()}

        if required <= names:
            return d

        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))

    return None

def _has_min_files(d, exts, min_count, scan_cap=20000):
    n = 0

    try:
        with os.scandir(d) as it:
            for i, e in enumerate(it):
                if i >= scan_cap:
                    break

                if e.is_file() and e.name.lower().endswith(exts):
                    n += 1

                    if n >= min_count:
                        return True

    except (PermissionError, FileNotFoundError, NotADirectoryError):
        return False

    return False

def find_image_dir(root, exts=(".jpg", ".jpeg"), min_images=100, max_depth=6):
    root = Path(root)

    if not root.exists():
        return None

    q = deque([(root, 0)])
    found = []

    while q:
        d, depth = q.popleft()

        try:
            entries = list(os.scandir(d))
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue

        if _has_min_files(d, exts, min_images):
            found.append(d)
            continue

        if depth < max_depth:
            for e in entries:
                if e.is_dir() and not e.name.startswith("."):
                    q.append((Path(e.path), depth + 1))

    if not found:
        return None

    found.sort(
        key=lambda p: (
            0 if "train" in str(p).lower() else 1,
            len(str(p))
        )
    )

    return found[0]

print("✓ helpers and paths loaded")
print("CACHE:", CACHE)

In [ ]:
# ================================================================
# STAGE 0 — BRING THE LOCKED PROJECT IN AND VERIFY THE WHOLE LOCK CHAIN
# ================================================================
PROJECT_MARKERS = {"step0I_locked_clean_population", "step1F_lesion_level_rho_assignments"}

if not PROJECT.exists():
    # Kaggle's mount layout is not fixed (seen both /kaggle/input/<slug>/... and /kaggle/input/datasets/<owner>/<slug>/...),
    # so this SEARCHES under /kaggle/input for the directory that actually has the two lock-chain marker folders,
    # rather than assuming a fixed depth. The dataset itself is never modified - only read and copied.
    src = find_by_markers(INPUT, PROJECT_MARKERS)
    assert src is not None, (
        "Project folder not found anywhere under /kaggle/input (searched up to 6 levels deep for a directory containing "
        f"{sorted(PROJECT_MARKERS)}). Check that your project dataset is attached under Add Data, and that the zip you "
        "uploaded actually has these folders at its top level (not nested inside an extra wrapper folder Kaggle can't see into).")
    log(f"found locked project at {src} - copying (read-only source, nothing there is modified) ...")
    shutil.copytree(src, PROJECT)
n_files = sum(1 for p in PROJECT.rglob("*") if p.is_file())
print(f"project working copy: {PROJECT}  ({n_files} files)")

# ---- verify every locked artifact against its recorded hash (environment and cache are re-recorded on this platform, so they are not checked here)
checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok))); print(("✓ " if ok else "✗ ") + name + (f"   {detail}" if detail else "")); assert ok, "LOCK CHAIN CHECK FAILED: " + name
sha = lambda rel: sha_file(PROJECT / rel)
F = J("step1U_final_protocol_lock/FINAL_PROTOCOL_HASH.json")
check("final protocol == recorded hash", sha("step1U_final_protocol_lock/FINAL_LOCKED_EXPERIMENT_PROTOCOL.json") == F["protocol_sha256"])
check("seed manifest == recorded hash", sha("step1P_seed_manifest/seed_manifest.json") == F["seed_manifest_sha256"])
_cand = {sha("step1I_protocol_lock/LOCKED_EXPERIMENT_PROTOCOL.json")} | {v for v in J("step1I_protocol_lock/protocol_hash.json").values() if isinstance(v, str)}
check("parent 1I protocol hash", F["parent_step1I_sha256"] in _cand)
check("A1 == recorded hash", sha("step2O_A1_final_lock/FINAL_LOCKED_AMENDMENT_A1.json") == J("step2O_A1_final_lock/A1_FINAL_HASH.json")["sha256"])
check("A1.1 == recorded hash", sha("step2P_A1_1_solver_addendum_lock/FINAL_LOCKED_A1_1_SOLVER_ADDENDUM.json") == J("step2P_A1_1_solver_addendum_lock/A1_1_FINAL_HASH.json")["sha256"])
GEN_SHA = J("step1J_generator_recovery_lock/generator_recovery_lock.json")["generator_sha256"]
check("generator == recorded hash", sha("step1J_generator_recovery_lock/recovered_original_hair_generator.py") == GEN_SHA)
check("solver == recorded hash", sha("step2Q_solver_implementation_v2/deterministic_evaluation_solver.py") == J("step2Q_solver_implementation_v2/solver_implementation_metadata.json")["solver_sha256"])
check("analysis code == frozen hash", sha("step2W_analysis_code_freeze/analysis_pipeline.py") == J("step2W_analysis_code_freeze/ANALYSIS_CODE_FINAL_HASH.json")["sha256"])
check("training code == frozen hash", sha("step2X_training_code_freeze/train_one_model.py") == J("step2X_training_code_freeze/TRAINING_CODE_FINAL_HASH.json")["sha256"])
E = J("step2S_evaluation_manifest_lock/EVALUATION_MANIFEST_FINAL_HASH.json")
check("3 evaluation manifests == locked hashes", all(sha(f"step2R_production_evaluation_manifests/evaluation_{c}.csv") == h for c, h in E["manifest_sha256"].items()))
check("XAI subset == locked hash", sha("step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv") == J("step2U_xai_subset_lock/XAI_SUBSET_FINAL_HASH.json")["sha256"])
RO = J("step2V_seed_table_and_run_order/RUN_ORDER_FINAL_HASH.json")
check("run-order manifest == locked hash", sha("step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json") == RO["sha256"])
check("seed table == locked hash", sha("step2V_seed_table_and_run_order/TRAINING_SEED_TABLE.csv") == RO["seed_table_sha256"])
asg = list((PROJECT / "step1F_lesion_level_rho_assignments").glob("rho_*_rep_*.csv"))
check("30 training assignment files present", len(asg) == 30, f"({len(asg)})")
RUN_ORDER = json.load(open(PROJECT / "step2V_seed_table_and_run_order/RUN_ORDER_MANIFEST.json"))["run_order"]
check("run order lists 30 models", len(RUN_ORDER) == 30)
TRAIN = PROJECT / "step2X_training_code_freeze" / "train_one_model.py"
ANALYSIS = PROJECT / "step2W_analysis_code_freeze" / "analysis_pipeline.py"
save_record("stage0_lock_chain_verified.json", {"verified": dict(checks), "project_files": n_files, "utc": utc()})
print("\nSTAGE 0 PASSED: the locked project arrived intact.")


In [ ]:
# ================================================================
# STAGE 0b — COMPATIBILITY SYMLINK
# ================================================================

CONTENT_PROJECT = Path("/content/isic_final_project")

if CONTENT_PROJECT.exists():
    print(f"✓ {CONTENT_PROJECT} already exists - leaving it as is")
else:
    CONTENT_PROJECT.parent.mkdir(parents=True, exist_ok=True)

    try:
        CONTENT_PROJECT.symlink_to(
            PROJECT,
            target_is_directory=True
        )

        print(
            f"✓ compatibility symlink created: "
            f"{CONTENT_PROJECT} -> {PROJECT}"
        )

    except OSError as e:
        print(f"Symlink failed ({e}) - falling back to a full copy")
        shutil.copytree(PROJECT, CONTENT_PROJECT)

gen_sha = sha_file(
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

assert gen_sha == json.load(
    open(
        PROJECT
        / "step1J_generator_recovery_lock"
        / "generator_recovery_lock.json"
    )
)["generator_sha256"]

print("✓ generator file hash unchanged")

In [ ]:
# ================================================================
# STAGE 2 — BUILD THE 224x224 CACHE ON KAGGLE
#
# CPU / multiprocessing only — CUDA is NOT used here.
#
# IMPORTANT:
# - Scientific hair generator is NOT modified.
# - Only imports, literal constants, and function definitions
#   are loaded from the recovered generator.
# - Module-level validation/test code is deliberately not executed.
# - Locked population + medium severity + deterministic generator
#   are preserved.
# ================================================================

import textwrap
import subprocess
import time
import json
import hashlib
from pathlib import Path

# ----------------------------------------------------------------
# Locate ISIC image directory
# ----------------------------------------------------------------

IMAGE_DIR = (
    globals().get("IMAGE_DIR_OVERRIDE")
    or find_image_dir(INPUT, min_images=100)
)

assert IMAGE_DIR is not None, (
    "ISIC image folder not found anywhere under /kaggle/input. "
    "Check that the isic-2019-dataset-full dataset is attached "
    "through Add Data."
)

print("image folder:", IMAGE_DIR)

# ----------------------------------------------------------------
# Required locked paths
# ----------------------------------------------------------------

GEN_PATH = (
    PROJECT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

POP_PATH = (
    PROJECT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

CACHE_LOCK = RECORDS / "cache_final_hash.json"

CACHE.mkdir(
    parents=True,
    exist_ok=True
)

assert GEN_PATH.exists(), (
    f"Recovered hair generator not found:\n{GEN_PATH}"
)

assert POP_PATH.exists(), (
    f"Locked development population not found:\n{POP_PATH}"
)

# ----------------------------------------------------------------
# Required cache files
# ----------------------------------------------------------------

REQUIRED_CACHE_FILES = [
    "clean_224.npy",
    "hair_224.npy",
    "mask_area.npy",
    "cache_index.csv",
]

cache_files_present = all(
    (CACHE / f).exists()
    for f in REQUIRED_CACHE_FILES
)

# ----------------------------------------------------------------
# Existing complete cache?
# ----------------------------------------------------------------

if (
    CACHE_LOCK.exists()
    and cache_files_present
    and not globals().get("ALLOW_RERUN_CACHE", False)
):

    print("=" * 80)
    print("STAGE 2 — EXISTING CACHE DETECTED")
    print("=" * 80)

    print("✓ cache_final_hash.json exists")

    for f in REQUIRED_CACHE_FILES:
        print("✓", CACHE / f)

    print("\n✓ Complete cache already exists.")
    print("Skipping rebuild.")

else:

    # ============================================================
    # Remove partial/stale cache
    # ============================================================

    print("=" * 80)
    print("STAGE 2 — PREPARING CLEAN CACHE BUILD")
    print("=" * 80)

    if CACHE_LOCK.exists():
        print("Removing stale/incomplete cache lock:", CACHE_LOCK)
        CACHE_LOCK.unlink()

    for name in [
        "clean_224.npy",
        "hair_224.npy",
        "mask_area.npy",
        "cache_index.csv",
        "_builder_record.json",
    ]:

        p = CACHE / name

        if p.exists():
            p.unlink()
            print("removed:", p)

    print("\n✓ Partial cache cleared.")

    # ============================================================
    # Standalone CPU cache-builder subprocess
    # ============================================================

    BUILDER = TEMP / "cache_builder.py"

    BUILDER.write_text(
        textwrap.dedent(
            r'''
            import argparse
            import ast
            import hashlib
            import json
            import random
            from pathlib import Path

            import numpy as np
            import pandas as pd
            from PIL import Image
            import multiprocessing as mp

            SEV = "medium"

            def sha_file(p, chunk=1 << 22):
                h = hashlib.sha256()

                with open(p, "rb") as f:
                    for b in iter(
                        lambda: f.read(chunk),
                        b""
                    ):
                        h.update(b)

                return h.hexdigest()

            def find_image(image_dir, image_id):

                for ext in (
                    ".jpg",
                    ".jpeg",
                    ".JPG",
                    ".JPEG"
                ):

                    p = (
                        image_dir
                        / f"{image_id}{ext}"
                    )

                    if p.exists():
                        return p

                raise FileNotFoundError(image_id)

            _G = {}

            # ====================================================
            # SAFE GENERATOR LOADER
            # ====================================================

            def load_generator(gen_path):

                source = Path(gen_path).read_text(
                    encoding="utf-8"
                )

                tree = ast.parse(source)

                safe_nodes = []

                for node in tree.body:

                    if isinstance(
                        node,
                        (
                            ast.Import,
                            ast.ImportFrom,
                        )
                    ):

                        safe_nodes.append(node)
                        continue

                    if isinstance(
                        node,
                        (
                            ast.FunctionDef,
                            ast.AsyncFunctionDef,
                        )
                    ):

                        safe_nodes.append(node)
                        continue

                    if isinstance(node, ast.Assign):

                        try:
                            ast.literal_eval(node.value)
                            safe_nodes.append(node)

                        except Exception:
                            pass

                    elif isinstance(node, ast.AnnAssign):

                        if node.value is not None:

                            try:
                                ast.literal_eval(node.value)
                                safe_nodes.append(node)

                            except Exception:
                                pass

                module = ast.Module(
                    body=safe_nodes,
                    type_ignores=[]
                )

                ast.fix_missing_locations(module)

                ns = {
                    "__file__": str(gen_path),
                    "__name__": "recovered_hair_generator",
                }

                exec(
                    compile(
                        module,
                        str(gen_path),
                        "exec"
                    ),
                    ns
                )

                assert "generate_hair_mask" in ns
                assert "apply_hair" in ns

                return ns

            # ====================================================
            # WORKER INITIALIZER
            # ====================================================

            def _init(
                gen_path,
                cp,
                hp,
                image_dir
            ):

                ns = load_generator(gen_path)

                _G.update(
                    gm=ns["generate_hair_mask"],
                    ah=ns["apply_hair"],

                    clean=np.load(
                        cp,
                        mmap_mode="r+"
                    ),

                    hair=np.load(
                        hp,
                        mmap_mode="r+"
                    ),

                    image_dir=Path(image_dir)
                )

            # ====================================================
            # WORKER
            # ====================================================

            def _work(job):

                lo, ids = job
                out = []

                for r, image_id in zip(
                    range(
                        lo,
                        lo + len(ids)
                    ),
                    ids
                ):

                    clean = np.asarray(
                        Image.open(
                            find_image(
                                _G["image_dir"],
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )

                    mask = _G["gm"](
                        image_id=image_id,
                        severity=SEV
                    )

                    hair = _G["ah"](
                        image=clean,
                        mask=mask,
                        severity=SEV,
                        image_id=image_id
                    )

                    m = mask > 0

                    outside = int(
                        (
                            np.any(
                                hair != clean,
                                axis=2
                            )
                            & ~m
                        ).sum()
                    )

                    _G["clean"][r] = clean
                    _G["hair"][r] = hair

                    out.append(
                        (
                            r,
                            int(m.sum()),
                            outside
                        )
                    )

                _G["clean"].flush()
                _G["hair"].flush()

                return out

            # ====================================================
            # MAIN
            # ====================================================

            def main():

                ap = argparse.ArgumentParser()

                ap.add_argument(
                    "--gen-path",
                    required=True
                )

                ap.add_argument(
                    "--pop-path",
                    required=True
                )

                ap.add_argument(
                    "--cache-dir",
                    required=True
                )

                ap.add_argument(
                    "--image-dir",
                    required=True
                )

                ap.add_argument(
                    "--n-proc",
                    type=int,
                    default=4
                )

                a = ap.parse_args()

                gen_path = Path(a.gen_path)
                pop_path = Path(a.pop_path)
                cache_dir = Path(a.cache_dir)
                image_dir = Path(a.image_dir)

                # ------------------------------------------------
                # Locked population
                # ------------------------------------------------

                pop = pd.read_csv(pop_path)

                assert list(pop.columns) == [
                    "image",
                    "lesion_id",
                    "class_name",
                    "malignant",
                    "split"
                ]

                pop = (
                    pop
                    .sort_values(
                        [
                            "split",
                            "image"
                        ]
                    )
                    .reset_index(drop=True)
                )

                N = len(pop)

                print(
                    f"locked population: {N} images",
                    flush=True
                )

                # ------------------------------------------------
                # Cache index
                # ------------------------------------------------

                idx = pop.copy()

                idx["row"] = np.arange(N)

                idx.to_csv(
                    cache_dir / "cache_index.csv",
                    index=False
                )

                # ------------------------------------------------
                # Allocate arrays
                # ------------------------------------------------

                clean_p = cache_dir / "clean_224.npy"
                hair_p = cache_dir / "hair_224.npy"
                area_p = cache_dir / "mask_area.npy"

                for p in (
                    clean_p,
                    hair_p
                ):

                    np.lib.format.open_memmap(
                        p,
                        mode="w+",
                        dtype=np.uint8,
                        shape=(
                            N,
                            224,
                            224,
                            3
                        )
                    ).flush()

                # ------------------------------------------------
                # Jobs
                # ------------------------------------------------

                ids = (
                    pop["image"]
                    .astype(str)
                    .tolist()
                )

                jobs = [
                    (
                        lo,
                        ids[
                            lo:
                            lo + 64
                        ]
                    )
                    for lo in range(
                        0,
                        N,
                        64
                    )
                ]

                area = np.zeros(
                    N,
                    dtype=np.int32
                )

                outside_total = 0
                done = 0

                print(
                    f"starting {a.n_proc} cache workers...",
                    flush=True
                )

                ctx = mp.get_context("fork")

                with ctx.Pool(
                    a.n_proc,
                    initializer=_init,
                    initargs=(
                        str(gen_path),
                        str(clean_p),
                        str(hair_p),
                        str(image_dir),
                    )
                ) as pool:

                    for res in pool.imap_unordered(
                        _work,
                        jobs
                    ):

                        for (
                            r,
                            ar,
                            o
                        ) in res:

                            area[r] = ar
                            outside_total += o

                        done += len(res)

                        print(
                            f"cache {done:6d}/{N}",
                            flush=True
                        )

                # ------------------------------------------------
                # Causal-integrity check
                # ------------------------------------------------

                if outside_total != 0:

                    print(
                        f"FATAL: {outside_total} pixels changed "
                        f"outside the hair mask",
                        flush=True
                    )

                    raise SystemExit(1)

                # ------------------------------------------------
                # Save mask areas
                # ------------------------------------------------

                np.save(
                    area_p,
                    area
                )

                # ------------------------------------------------
                # Deterministic spot check
                # ------------------------------------------------

                print(
                    "running deterministic 60-image spot check...",
                    flush=True
                )

                ns = load_generator(gen_path)

                cm = np.load(
                    clean_p,
                    mmap_mode="r"
                )

                hm = np.load(
                    hair_p,
                    mmap_mode="r"
                )

                sample_rows = random.Random(
                    20260919
                ).sample(
                    range(N),
                    min(60, N)
                )

                for r in sample_rows:

                    image_id = ids[r]

                    c = np.asarray(
                        Image.open(
                            find_image(
                                image_dir,
                                image_id
                            )
                        )
                        .convert("RGB")
                        .resize(
                            (
                                224,
                                224
                            ),
                            Image.Resampling.LANCZOS
                        ),
                        dtype=np.uint8
                    )

                    h = ns["apply_hair"](
                        image=c,
                        mask=ns["generate_hair_mask"](
                            image_id=image_id,
                            severity=SEV
                        ),
                        severity=SEV,
                        image_id=image_id
                    )

                    assert np.array_equal(
                        cm[r],
                        c
                    ), (
                        f"clean cache mismatch at row {r}"
                    )

                    assert np.array_equal(
                        hm[r],
                        h
                    ), (
                        f"hair cache mismatch at row {r}"
                    )

                print(
                    "SPOT_CHECK_OK",
                    flush=True
                )

                # ------------------------------------------------
                # SHA256
                # ------------------------------------------------

                file_sha = {
                    p.name: sha_file(p)
                    for p in (
                        clean_p,
                        hair_p,
                        area_p,
                        cache_dir / "cache_index.csv"
                    )
                }

                record = {
                    "n_images": N,

                    "mask_area": {
                        "min": int(area.min()),
                        "mean": float(area.mean()),
                        "max": int(area.max()),
                    },

                    "file_sha256": file_sha,

                    "generator_sha256": sha_file(
                        gen_path
                    ),

                    "image_dir": str(image_dir),

                    "severity": SEV,
                }

                with open(
                    cache_dir / "_builder_record.json",
                    "w"
                ) as f:

                    json.dump(
                        record,
                        f,
                        indent=2,
                        sort_keys=True
                    )

                print(
                    "CACHE_BUILD_COMPLETE",
                    flush=True
                )

                print(
                    json.dumps(
                        record,
                        indent=2,
                        sort_keys=True
                    ),
                    flush=True
                )

            if __name__ == "__main__":
                main()
            '''
        )
    )

    # ============================================================
    # Run CPU builder
    # ============================================================

    n_proc = int(
        globals().get(
            "N_PROC_OVERRIDE",
            4
        )
    )

    print("=" * 80)
    print("LAUNCHING CPU CACHE BUILDER")
    print("=" * 80)

    print("CPU workers:", n_proc)
    print("GPU: NOT USED")
    print("Image directory:", IMAGE_DIR)
    print("Cache directory:", CACHE)

    t_cache = time.time()

    cmd = [
        sys.executable,
        str(BUILDER),

        "--gen-path",
        str(GEN_PATH),

        "--pop-path",
        str(POP_PATH),

        "--cache-dir",
        str(CACHE),

        "--image-dir",
        str(IMAGE_DIR),

        "--n-proc",
        str(n_proc),
    ]

    proc = subprocess.run(
        cmd,
        text=True
    )

    if proc.returncode != 0:
        raise RuntimeError(
            f"Cache builder failed with return code "
            f"{proc.returncode}"
        )

    print()
    print(
        f"Cache build wall time: "
        f"{(time.time() - t_cache) / 60:.2f} minutes"
    )

# ----------------------------------------------------------------
# Final cache verification
# ----------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL CACHE VERIFICATION")
print("=" * 80)

for f in REQUIRED_CACHE_FILES:
    p = CACHE / f
    assert p.exists(), f"Missing cache file: {p}"
    print("✓", p)

assert (CACHE / "_builder_record.json").exists()

print("\n✓ CACHE BUILD / VERIFICATION COMPLETE")
print("✓ GPU was not used.")

In [ ]:
# ================================================================
# CELL 13 — LOCKED XAI SUBSET ↔ CACHE INTEGRITY VERIFICATION
#
# This cell performs verification only.
# It does NOT modify the cache or locked artifacts.
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("CELL 13 — XAI SUBSET ↔ CACHE INTEGRITY")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------------

CACHE = Path("/kaggle/temp/isic_cache")

clean_path = CACHE / "clean_224.npy"
hair_path = CACHE / "hair_224.npy"
area_path = CACHE / "mask_area.npy"
index_path = CACHE / "cache_index.csv"

assert clean_path.exists()
assert hair_path.exists()
assert area_path.exists()
assert index_path.exists()

# ---------------------------------------------------------------
# 2. Load cache metadata
# ---------------------------------------------------------------

cache_index = pd.read_csv(index_path)

clean = np.load(
    clean_path,
    mmap_mode="r"
)

hair = np.load(
    hair_path,
    mmap_mode="r"
)

mask_area = np.load(
    area_path,
    mmap_mode="r"
)

print("Cache shapes:")
print("  clean_224 :", clean.shape, clean.dtype)
print("  hair_224  :", hair.shape, hair.dtype)
print("  mask_area :", mask_area.shape, mask_area.dtype)
print("  index     :", cache_index.shape)

assert len(cache_index) == len(clean)
assert len(cache_index) == len(hair)
assert len(cache_index) == len(mask_area)

# ---------------------------------------------------------------
# 3. Load authoritative locked XAI subset
# ---------------------------------------------------------------

XAI_SUBSET_PATH = (
    PROJECT_SOURCE
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

assert XAI_SUBSET_PATH.exists()

xai = pd.read_csv(XAI_SUBSET_PATH)

print()
print("Locked XAI subset:")
print("  rows   :", len(xai))
print("  columns:", xai.columns.tolist())

assert len(xai) == 500
assert xai["image"].nunique() == 500

# ---------------------------------------------------------------
# 4. Verify cache image uniqueness
# ---------------------------------------------------------------

assert cache_index["image"].is_unique, (
    "Cache index contains duplicate image IDs."
)

print()
print("✓ Cache image IDs are unique")

# ---------------------------------------------------------------
# 5. Map every XAI image to exactly one cache row
# ---------------------------------------------------------------

cache_lookup = dict(
    zip(
        cache_index["image"].astype(str),
        cache_index["row"].astype(int)
    )
)

missing = []
rows = []

for image_id in xai["image"].astype(str):

    if image_id not in cache_lookup:
        missing.append(image_id)
    else:
        rows.append(cache_lookup[image_id])

assert not missing, (
    f"{len(missing)} XAI images missing from cache. "
    f"Examples: {missing[:10]}"
)

assert len(rows) == 500
assert len(set(rows)) == 500

print("✓ All 500 XAI images map to cache")
print("✓ One unique cache row per XAI image")

# ---------------------------------------------------------------
# 6. Verify stored mask areas against locked XAI metadata
# ---------------------------------------------------------------

cache_rows = np.asarray(rows, dtype=np.int64)

cached_areas = mask_area[cache_rows]

locked_areas = xai["mask_area"].to_numpy(
    dtype=np.int64
)

area_diff = cached_areas - locked_areas

print()
print("Mask-area comparison:")
print("  max absolute difference:",
      int(np.max(np.abs(area_diff))))
print("  mismatches:",
      int(np.sum(area_diff != 0)))

assert np.all(area_diff == 0), (
    "Mask-area mismatch between locked XAI subset "
    "and production cache."
)

print("✓ All 500 mask areas match exactly")

# ---------------------------------------------------------------
# 7. Regenerate exact frozen masks for all 500 images
# ---------------------------------------------------------------

print()
print("Regenerating exact frozen masks for 500 XAI images...")

mask_mismatches = []
regenerated_areas = []

for i, image_id in enumerate(
    xai["image"].astype(str)
):

    mask = generator.generate_hair_mask(
        image_id=image_id,
        severity=generator.PRIMARY_SEVERITY,
        global_seed=generator.GLOBAL_SEED,
        size=224,
    )

    mask = np.asarray(mask).astype(bool)

    cached_area = int(
        mask_area[cache_rows[i]]
    )

    generated_area = int(mask.sum())

    regenerated_areas.append(
        generated_area
    )

    if generated_area != cached_area:
        mask_mismatches.append(
            (
                image_id,
                generated_area,
                cached_area,
            )
        )

assert not mask_mismatches, (
    f"Mask regeneration mismatch for "
    f"{len(mask_mismatches)} images. "
    f"Examples: {mask_mismatches[:5]}"
)

print("✓ 500/500 regenerated mask areas match")

# ---------------------------------------------------------------
# 8. Verify XAI class labels / lesion uniqueness
# ---------------------------------------------------------------

assert xai["lesion_id"].nunique() == 500

print("✓ 500 unique lesion IDs")

print()
print("=" * 70)
print("CELL 13 COMPLETE — PASS")
print("=" * 70)

print(
    """
500/500 XAI images
500/500 unique cache rows
500/500 mask areas exact
500/500 frozen masks reproducible
500/500 unique lesions
"""
)

In [ ]:
# ================================================================
# CELL 14 — FROZEN PRODUCTION CHECKPOINT DISCOVERY + LOAD TEST
#
# Verification only.
# Does NOT modify checkpoints or locked artifacts.
# ================================================================

import os
import json
import hashlib
from pathlib import Path

import torch
import torch.nn as nn
import torchvision

print("=" * 70)
print("CELL 14 — FROZEN PRODUCTION CHECKPOINT VERIFICATION")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Locate the training root
# ---------------------------------------------------------------

TRAINING_ROOT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "training_runs"
)

assert TRAINING_ROOT.exists(), (
    f"Training root not found: {TRAINING_ROOT}"
)

print("Training root:")
print(TRAINING_ROOT)


# ---------------------------------------------------------------
# 2. Discover final_epoch15 checkpoints
# ---------------------------------------------------------------

checkpoints = sorted(
    TRAINING_ROOT.glob("*/final_epoch15.pt")
)

print()
print("final_epoch15 checkpoints found:", len(checkpoints))

assert len(checkpoints) == 30, (
    f"Expected exactly 30 final_epoch15 checkpoints, "
    f"found {len(checkpoints)}"
)

for p in checkpoints:
    print(" ", p.parent.name)


# ---------------------------------------------------------------
# 3. Extract model names
# ---------------------------------------------------------------

MODEL_NAMES = [
    p.parent.name
    for p in checkpoints
]

assert len(set(MODEL_NAMES)) == 30

print()
print("✓ 30 unique production models found")


# ---------------------------------------------------------------
# 4. Verify expected rho/rep structure
# ---------------------------------------------------------------

print()
print("Model naming structure:")

for name in MODEL_NAMES:
    print(" ", name)


# ---------------------------------------------------------------
# 5. Frozen production loader
# ---------------------------------------------------------------

def load_model_production(model_name, device):

    ckpt = TRAINING_ROOT / model_name / "final_epoch15.pt"

    assert ckpt.exists(), (
        f"Checkpoint missing: {ckpt}"
    )

    model = torchvision.models.resnet18(
        weights=None
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        2
    )

    checkpoint = torch.load(
        ckpt,
        map_location="cpu"
    )

    if isinstance(checkpoint, dict):

        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]

        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]

        else:
            state_dict = checkpoint

    else:
        state_dict = checkpoint

    state_dict = {
        k.replace("module.", "", 1)
        if k.startswith("module.")
        else k: v
        for k, v in state_dict.items()
    }

    model.load_state_dict(
        state_dict,
        strict=True
    )

    model.eval()
    model.to(device)

    return model


# ---------------------------------------------------------------
# 6. Determine device
# ---------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print()
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ---------------------------------------------------------------
# 7. Load every model once
# ---------------------------------------------------------------

print()
print("=" * 70)
print("LOADING ALL 30 PRODUCTION MODELS")
print("=" * 70)

models = {}

for i, model_name in enumerate(
    MODEL_NAMES,
    start=1
):

    print(
        f"[{i:02d}/30] Loading {model_name} ...",
        end=" "
    )

    model = load_model_production(
        model_name,
        DEVICE
    )

    # Basic structural verification
    assert isinstance(
        model,
        torchvision.models.ResNet
    )

    assert model.fc.out_features == 2

    models[model_name] = model

    print("PASS")


# ---------------------------------------------------------------
# 8. Basic forward-pass sanity test
# ---------------------------------------------------------------

print()
print("=" * 70)
print("FORWARD-PASS SANITY TEST")
print("=" * 70)

# Use one real cached image.
sample_cache_row = int(
    cache_index.iloc[0]["row"]
)

sample_clean = torch.from_numpy(
    clean[sample_cache_row]
).permute(2, 0, 1).float() / 255.0

# Production normalization
MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    device=DEVICE
).view(1, 3, 1, 1)

STD = torch.tensor(
    [0.229, 0.224, 0.225],
    device=DEVICE
).view(1, 3, 1, 1)

x = sample_clean.unsqueeze(0).to(DEVICE)

xn = (x - MEAN) / STD

with torch.no_grad():

    outputs = {}

    for model_name, model in models.items():

        z = model(xn)

        assert z.shape == (
            1,
            2
        )

        assert torch.isfinite(z).all()

        outputs[model_name] = (
            z.detach()
            .cpu()
            .numpy()
            .reshape(-1)
            .tolist()
        )

print("✓ All 30 models produced finite [1, 2] logits")


# ---------------------------------------------------------------
# 9. Summary
# ---------------------------------------------------------------

print()
print("=" * 70)
print("CELL 14 COMPLETE — PASS")
print("=" * 70)

print(
    f"""
Production checkpoints : {len(models)}
Architecture            : ResNet-18
Output classes          : 2
Checkpoint              : final_epoch15.pt
Loader strictness       : strict=True
Device                  : {DEVICE}
Forward pass            : PASS
"""
)

In [ ]:
# ================================================================
# CELL 15 — FROZEN INTEGRATED-GRADIENTS SMOKE TEST
#
# One production model × one locked XAI image.
# Uses the exact frozen ap.ig_enrichment().
#
# NO checkpoint modification.
# NO cache modification.
# NO primary-evidence modification.
# ================================================================

import numpy as np
import torch

print("=" * 70)
print("CELL 15 — FROZEN IG SMOKE TEST")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Select one locked XAI observation
# ---------------------------------------------------------------

TEST_MODEL_NAME = "rho_0.0_rep_1"

assert TEST_MODEL_NAME in models

test_row = int(cache_rows[0])
test_image = str(xai.iloc[0]["image"])

print("Model :", TEST_MODEL_NAME)
print("Image :", test_image)
print("Cache row:", test_row)


# ---------------------------------------------------------------
# 2. Retrieve cached hair image
# ---------------------------------------------------------------

hair_u8 = np.array(
    hair[test_row],
    dtype=np.uint8,
    copy=True,
)

assert hair_u8.shape == (
    224,
    224,
    3
)

assert hair_u8.dtype == np.uint8


# ---------------------------------------------------------------
# 3. Regenerate exact frozen mask
# ---------------------------------------------------------------

mask_bool = generator.generate_hair_mask(
    image_id=test_image,
    severity=generator.PRIMARY_SEVERITY,
    global_seed=generator.GLOBAL_SEED,
    size=224,
)

mask_bool = np.asarray(
    mask_bool
).astype(bool)

assert mask_bool.shape == (
    224,
    224
)

print("Mask area:", int(mask_bool.sum()))
print(
    "Cached area:",
    int(mask_area[test_row])
)

assert int(mask_bool.sum()) == int(
    mask_area[test_row]
)


# ---------------------------------------------------------------
# 4. Run EXACT frozen production IG
# ---------------------------------------------------------------

model = models[TEST_MODEL_NAME]

model.eval()

with torch.enable_grad():

    result = ap.ig_enrichment(
        model=model,
        hair_u8=hair_u8,
        mask_bool=mask_bool,
        device=DEVICE,
        steps=ap.IG_STEPS,
        retry_steps=ap.IG_RETRY_STEPS,
        sigma=ap.IG_SIGMA,
        chunk=25,
    )


# ---------------------------------------------------------------
# 5. Print result
# ---------------------------------------------------------------

print()
print("=" * 70)
print("FROZEN IG RESULT")
print("=" * 70)

for key, value in result.items():
    print(f"{key:30s}: {value}")


# ---------------------------------------------------------------
# 6. Validate result
# ---------------------------------------------------------------

assert "enrichment" in result
assert "completeness_error" in result
assert "delta" in result
assert "tolerance" in result
assert "steps_used" in result
assert "completeness_failed" in result
assert "signed_attribution_inside_mask" in result
assert "mask_area_fraction" in result

assert np.isfinite(
    result["completeness_error"]
)

assert np.isfinite(
    result["delta"]
)

assert np.isfinite(
    result["tolerance"]
)

assert np.isfinite(
    result["signed_attribution_inside_mask"]
)

assert np.isfinite(
    result["mask_area_fraction"]
)

assert result["mask_area_fraction"] > 0

assert result["steps_used"] in (
    ap.IG_STEPS,
    ap.IG_RETRY_STEPS,
)

assert result["completeness_failed"] is False

assert np.isfinite(
    result["enrichment"]
), (
    "IG enrichment is undefined/non-finite "
    "for the smoke-test observation."
)


# ---------------------------------------------------------------
# 7. Final
# ---------------------------------------------------------------

print()
print("=" * 70)
print("CELL 15 COMPLETE — PASS")
print("=" * 70)

print(
    f"""
Frozen IG implementation : ap.ig_enrichment
IG nominal steps         : {ap.IG_STEPS}
IG retry steps           : {ap.IG_RETRY_STEPS}
Baseline sigma           : {ap.IG_SIGMA}
Steps actually used     : {result["steps_used"]}
Completeness failed      : {result["completeness_failed"]}
Enrichment               : {result["enrichment"]:.8f}
"""
)

In [ ]:
# ================================================================
# CELL 16 — PARAMETER RANDOMIZATION SANITY CHECK: PILOT
# CORRECTED CUDA RANDOMIZATION
#
# Pilot:
#   1 production model × 20 locked XAI images
#
# Randomization proceeds from output -> input.
#
# The frozen production model is NEVER modified.
# ================================================================

import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

print("=" * 70)
print("CELL 16 — PARAMETER RANDOMIZATION PILOT")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Deterministic configuration
# ---------------------------------------------------------------

RAND_SEED = 20260925

PILOT_MODEL = "rho_0.0_rep_1"
PILOT_N = 20

assert PILOT_MODEL in models
assert PILOT_N <= len(xai)

print("Model :", PILOT_MODEL)
print("Images:", PILOT_N)
print("Seed  :", RAND_SEED)


# ---------------------------------------------------------------
# 2. Cascading randomization levels
# ---------------------------------------------------------------

RANDOMIZATION_LEVELS = [
    ("original", []),

    ("fc", [
        "fc"
    ]),

    ("fc+layer4", [
        "fc",
        "layer4"
    ]),

    ("fc+layer4+layer3", [
        "fc",
        "layer4",
        "layer3"
    ]),

    ("fc+layer4+layer3+layer2", [
        "fc",
        "layer4",
        "layer3",
        "layer2"
    ]),

    ("fc+layer4+layer3+layer2+layer1", [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1"
    ]),

    ("all", [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1"
    ]),
]

print()
print("Randomization levels:")

for name, groups in RANDOMIZATION_LEVELS:
    print(" ", name)


# ---------------------------------------------------------------
# 3. Deterministic parameter randomization
#
# IMPORTANT:
# Random numbers are generated on the SAME device as each
# parameter. This fixes the CPU-generator/CUDA-parameter error.
# ---------------------------------------------------------------

def randomize_module_parameters(
    model,
    module_names,
    seed,
):

    for module_offset, module_name in enumerate(
        module_names
    ):

        module = getattr(
            model,
            module_name
        )

        # Dedicated deterministic seed for each module.
        module_seed = (
            int(seed)
            + 1000003 * module_offset
        )

        for parameter_offset, p in enumerate(
            module.parameters()
        ):

            parameter_seed = (
                module_seed
                + 1009 * parameter_offset
            )

            # Create generator on SAME device as parameter.
            rng = torch.Generator(
                device=p.device
            )

            rng.manual_seed(
                parameter_seed
            )

            with torch.no_grad():

                if p.ndim >= 2:

                    fan_in = nn.init._calculate_correct_fan(
                        p,
                        "fan_in"
                    )

                    gain = nn.init.calculate_gain(
                        "relu"
                    )

                    std = gain / math.sqrt(
                        fan_in
                    )

                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * std
                    )

                else:

                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * 0.01
                    )

                p.copy_(values)

    return model


# ---------------------------------------------------------------
# 4. EXACT frozen IG map calculation
#
# This mirrors the already-verified ap.ig_enrichment internals:
#   - same normalization
#   - same Gaussian baseline
#   - same sigma
#   - same target: logit[1] - logit[0]
#   - same 50 -> 100 step completeness rule
#   - same absolute-channel attribution reduction
#
# We need the actual map because the frozen production function
# returns enrichment rather than the map itself.
# ---------------------------------------------------------------

def frozen_ig_map(
    model,
    hair_u8,
):

    mean = torch.tensor(
        ap._MEAN,
        device=DEVICE
    ).view(1, 3, 1, 1)

    std = torch.tensor(
        ap._STD,
        device=DEVICE
    ).view(1, 3, 1, 1)

    model.eval()

    x = torch.from_numpy(
        np.ascontiguousarray(
            hair_u8
        )
    ).to(
        DEVICE
    )

    x = (
        x.permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = (
        2
        * int(
            math.ceil(
                3 * ap.IG_SIGMA
            )
        )
        + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[
            ap.IG_SIGMA,
            ap.IG_SIGMA,
        ],
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    def zf(t):

        o = model(t)

        return (
            o[:, 1]
            - o[:, 0]
        )

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=DEVICE,
                dtype=torch.float32,
            )
            / n
        )

        for s in range(
            0,
            n,
            25
        ):

            a = (
                alphas[
                    s:s + 25
                ]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn
                + a * (
                    xn - bn
                )
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            (xn - bn)
            * acc
            / n
        )

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    - zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            )
            - delta
        )

        return (
            ig,
            delta,
            err
        )

    # Exact production 50-step attempt
    ig, delta, err = run(
        ap.IG_STEPS
    )

    tol = max(
        ap.IG_TOL_REL * abs(delta),
        ap.IG_TAU_ABS,
    )

    # Exact production retry condition
    if err > tol:

        ig, delta, err = run(
            ap.IG_RETRY_STEPS
        )

    # Exact production spatial attribution reduction
    a = (
        ig.abs()
        .sum(1)[0]
    )

    return (
        a.detach()
        .cpu()
        .numpy()
    )


# ---------------------------------------------------------------
# 5. Spearman correlation
# ---------------------------------------------------------------

def spearman_corr(a, b):

    a = np.asarray(
        a
    ).ravel()

    b = np.asarray(
        b
    ).ravel()

    ra = (
        pd.Series(a)
        .rank(
            method="average"
        )
        .to_numpy()
    )

    rb = (
        pd.Series(b)
        .rank(
            method="average"
        )
        .to_numpy()
    )

    if (
        np.std(ra) == 0
        or np.std(rb) == 0
    ):
        return np.nan

    return float(
        np.corrcoef(
            ra,
            rb
        )[0, 1]
    )


# ---------------------------------------------------------------
# 6. Prepare pilot observations
# ---------------------------------------------------------------

pilot_rows = np.asarray(
    cache_rows[:PILOT_N],
    dtype=np.int64
)

pilot_images = [
    str(
        xai.iloc[i]["image"]
    )
    for i in range(
        PILOT_N
    )
]

print()
print(
    "Preparing original attribution maps..."
)


# ---------------------------------------------------------------
# 7. Original attribution maps
# ---------------------------------------------------------------

base_model = models[
    PILOT_MODEL
]

base_model.eval()

original_maps = {}

for j, (
    cache_row,
    image_id
) in enumerate(
    zip(
        pilot_rows,
        pilot_images
    ),
    start=1
):

    img = np.array(
        hair[
            int(cache_row)
        ],
        dtype=np.uint8,
        copy=True,
    )

    with torch.enable_grad():

        amap = frozen_ig_map(
            base_model,
            img,
        )

    original_maps[
        image_id
    ] = amap

    print(
        f"\r  Original maps: "
        f"{j:02d}/{PILOT_N}",
        end=""
    )

print()


# ---------------------------------------------------------------
# 8. Cascading randomization
# ---------------------------------------------------------------

records = []

print()
print("=" * 70)
print("RUNNING CASCADING RANDOMIZATION")
print("=" * 70)

for level_idx, (
    level_name,
    groups,
) in enumerate(
    RANDOMIZATION_LEVELS
):

    print()
    print(
        f"LEVEL {level_idx}: "
        f"{level_name}"
    )

    # -----------------------------------------------------------
    # Original level:
    # use the frozen production model directly.
    # -----------------------------------------------------------

    if level_name == "original":

        rand_model = base_model

    # -----------------------------------------------------------
    # Randomized levels:
    # deep-copy first, then randomize the copy.
    # -----------------------------------------------------------

    else:

        rand_model = copy.deepcopy(
            base_model
        ).to(
            DEVICE
        )

        rand_model.eval()

        rand_model = (
            randomize_module_parameters(
                rand_model,
                groups,
                seed=(
                    RAND_SEED
                    + 1000 * level_idx
                ),
            )
        )

        rand_model.eval()

    similarities = []

    for j, (
        cache_row,
        image_id
    ) in enumerate(
        zip(
            pilot_rows,
            pilot_images
        ),
        start=1
    ):

        img = np.array(
            hair[
                int(cache_row)
            ],
            dtype=np.uint8,
            copy=True,
        )

        with torch.enable_grad():

            amap = frozen_ig_map(
                rand_model,
                img,
            )

        corr = spearman_corr(
            original_maps[
                image_id
            ],
            amap,
        )

        similarities.append(
            corr
        )

        records.append({
            "model": PILOT_MODEL,
            "image": image_id,
            "level": level_idx,
            "level_name": level_name,
            "randomized_groups": "+".join(
                groups
            ),
            "spearman_vs_original": corr,
        })

        print(
            f"\r  Images: "
            f"{j:02d}/{PILOT_N}",
            end=""
        )

    print()

    finite = np.asarray(
        similarities,
        dtype=float
    )

    finite = finite[
        np.isfinite(finite)
    ]

    print(
        "  Mean Spearman:",
        (
            float(
                finite.mean()
            )
            if len(finite)
            else np.nan
        )
    )

    print(
        "  Median Spearman:",
        (
            float(
                np.median(
                    finite
                )
            )
            if len(finite)
            else np.nan
        )
    )


# ---------------------------------------------------------------
# 9. Aggregate
# ---------------------------------------------------------------

pilot_results = pd.DataFrame(
    records
)

pilot_summary = (
    pilot_results
    .groupby(
        [
            "level",
            "level_name",
            "randomized_groups",
        ],
        as_index=False
    )
    ["spearman_vs_original"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
    )
)

print()
print("=" * 70)
print("PARAMETER RANDOMIZATION PILOT SUMMARY")
print("=" * 70)

display(
    pilot_summary
)


# ---------------------------------------------------------------
# 10. Structural checks
# ---------------------------------------------------------------

assert len(
    pilot_results
) == (
    len(RANDOMIZATION_LEVELS)
    * PILOT_N
)

assert (
    pilot_results[
        "spearman_vs_original"
    ]
    .notna()
    .all()
)

assert (
    pilot_results[
        "level"
    ].nunique()
    == len(
        RANDOMIZATION_LEVELS
    )
)

print()
print("=" * 70)
print("CELL 16 COMPLETE — PILOT PASS")
print("=" * 70)

print(
    """
No production checkpoint was modified.
No locked artifact was modified.
Original attribution maps were computed from
the frozen production model.
"""
)

In [ ]:
# ================================================================
# CELL 17 — FREEZE FULL PARAMETER-RANDOMIZATION DESIGN
#
# This cell defines the analysis BEFORE the full run.
# It does not run IG.
# It does not modify production artifacts.
# ================================================================

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("CELL 17 — FULL PARAMETER-RANDOMIZATION DESIGN")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Fixed design
# ---------------------------------------------------------------

PR_CONFIG = {
    "analysis_name": "T1.4_parameter_randomization_full",

    "purpose": (
        "Cascading parameter-randomization sanity check "
        "for Integrated Gradients attribution maps."
    ),

    "model_count": 30,
    "image_count": 500,

    "models": sorted(
        MODEL_NAMES
    ),

    "xai_subset": str(
        XAI_SUBSET_PATH
    ),

    "cache": "/kaggle/temp/isic_cache",

    "attribution_function":
        "ap.ig_enrichment / exact frozen IG construction",

    "ig_steps": int(
        ap.IG_STEPS
    ),

    "ig_retry_steps": int(
        ap.IG_RETRY_STEPS
    ),

    "ig_sigma": float(
        ap.IG_SIGMA
    ),

    "ig_tol_rel": float(
        ap.IG_TOL_REL
    ),

    "ig_tau_abs": float(
        ap.IG_TAU_ABS
    ),

    "target": (
        "class-1 malignant logit minus "
        "class-0 benign logit"
    ),

    "attribution_reduction":
        "absolute IG summed across RGB channels",

    "similarity_metric":
        "pixelwise Spearman correlation "
        "against original trained-model attribution map",

    "randomization_order":
        "output-to-input cascading",

    "levels": [
        {
            "level": 0,
            "name": "original",
            "groups": [],
        },
        {
            "level": 1,
            "name": "fc",
            "groups": ["fc"],
        },
        {
            "level": 2,
            "name": "fc+layer4",
            "groups": ["fc", "layer4"],
        },
        {
            "level": 3,
            "name": "fc+layer4+layer3",
            "groups": [
                "fc",
                "layer4",
                "layer3",
            ],
        },
        {
            "level": 4,
            "name": "fc+layer4+layer3+layer2",
            "groups": [
                "fc",
                "layer4",
                "layer3",
                "layer2",
            ],
        },
        {
            "level": 5,
            "name": "fc+layer4+layer3+layer2+layer1",
            "groups": [
                "fc",
                "layer4",
                "layer3",
                "layer2",
                "layer1",
            ],
        },
        {
            "level": 6,
            "name": "all",
            "groups": [
                "fc",
                "layer4",
                "layer3",
                "layer2",
                "layer1",
                "conv1",
                "bn1",
            ],
        },
    ],

    "randomization_seed":
        int(RAND_SEED),

    "randomization_method":
        "deterministic fresh parameter initialization "
        "using local per-parameter generators",

    "original_model_policy":
        "original production checkpoint is never modified",

    "checkpoint_policy":
        "final_epoch15.pt",

    "total_evaluations":
        30 * 500 * 7,

    "pilot":
        {
            "model": PILOT_MODEL,
            "images": PILOT_N,
            "completed": True,
        },
}

# ---------------------------------------------------------------
# 2. Structural assertions
# ---------------------------------------------------------------

assert len(
    PR_CONFIG["models"]
) == 30

assert len(
    PR_CONFIG["levels"]
) == 7

assert (
    PR_CONFIG["image_count"]
    == 500
)

assert (
    PR_CONFIG["total_evaluations"]
    == 105000
)

# ---------------------------------------------------------------
# 3. Confirm every model exists
# ---------------------------------------------------------------

missing_models = [
    m for m in PR_CONFIG["models"]
    if m not in models
]

assert not missing_models, (
    f"Missing loaded models: {missing_models}"
)

# ---------------------------------------------------------------
# 4. Confirm locked subset hasn't changed
# ---------------------------------------------------------------

current_xai = pd.read_csv(
    XAI_SUBSET_PATH
)

assert len(current_xai) == 500
assert current_xai["image"].nunique() == 500

# ---------------------------------------------------------------
# 5. Confirm cache population
# ---------------------------------------------------------------

assert len(cache_index) == 23227
assert clean.shape == (
    23227,
    224,
    224,
    3,
)

assert hair.shape == (
    23227,
    224,
    224,
    3,
)

# ---------------------------------------------------------------
# 6. Serialize manifest
# ---------------------------------------------------------------

manifest_dir = Path(
    "/kaggle/working/T1_4_parameter_randomization"
)

manifest_dir.mkdir(
    parents=True,
    exist_ok=True
)

manifest_path = (
    manifest_dir
    / "T1_4_PARAMETER_RANDOMIZATION_DESIGN.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        PR_CONFIG,
        f,
        indent=2,
        sort_keys=True,
    )

# ---------------------------------------------------------------
# 7. Hash manifest
# ---------------------------------------------------------------

manifest_bytes = manifest_path.read_bytes()

manifest_sha256 = hashlib.sha256(
    manifest_bytes
).hexdigest()

hash_path = (
    manifest_dir
    / "T1_4_PARAMETER_RANDOMIZATION_DESIGN_SHA256.txt"
)

hash_path.write_text(
    manifest_sha256 + "\n",
    encoding="utf-8"
)

# ---------------------------------------------------------------
# 8. Display design
# ---------------------------------------------------------------

print()
print("Models:", len(PR_CONFIG["models"]))
print("Images:", PR_CONFIG["image_count"])
print("Levels:", len(PR_CONFIG["levels"]))
print(
    "Total IG evaluations:",
    PR_CONFIG["total_evaluations"]
)

print()
print("Levels:")

for level in PR_CONFIG["levels"]:

    print(
        f'  {level["level"]}: '
        f'{level["name"]}'
    )

print()
print("Manifest:")
print(manifest_path)

print()
print("Manifest SHA256:")
print(manifest_sha256)

print()
print("=" * 70)
print("CELL 17 COMPLETE — DESIGN FROZEN")
print("=" * 70)

In [ ]:
# ================================================================
# CELL 18C — FULL PARAMETER RANDOMIZATION — DUAL T4 THREADS
#
# Same frozen design:
#   30 models × 500 images × 7 levels
#
# GPU 0 -> first 15 models
# GPU 1 -> last 15 models
#
# Uses threads instead of multiprocessing because this is a
# Jupyter/Kaggle notebook and CUDA workers cannot reliably spawn
# an interactive __main__ worker function.
#
# NO production checkpoint is modified.
# NO locked artifact is modified.
# ================================================================

import os
import gc
import copy
import math
import json
import time
import hashlib
import traceback
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.functional as TF

print("=" * 70)
print("CELL 18C — DUAL T4 THREADED EXECUTION")
print("=" * 70)

# ================================================================
# 1. GPU verification
# ================================================================

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2

print(
    "GPU 0:",
    torch.cuda.get_device_name(0)
)

print(
    "GPU 1:",
    torch.cuda.get_device_name(1)
)

# ================================================================
# 2. Frozen directories
# ================================================================

PR_DIR = Path(
    "/kaggle/working/T1_4_parameter_randomization"
)

RESULT_DIR = (
    PR_DIR / "model_level_results"
)

SUMMARY_DIR = (
    PR_DIR / "summaries"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ================================================================
# 3. Verify frozen design
# ================================================================

manifest_path = (
    PR_DIR
    / "T1_4_PARAMETER_RANDOMIZATION_DESIGN.json"
)

manifest_hash_path = (
    PR_DIR
    / "T1_4_PARAMETER_RANDOMIZATION_DESIGN_SHA256.txt"
)

assert manifest_path.exists()
assert manifest_hash_path.exists()

manifest_sha256 = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

assert manifest_sha256 == (
    manifest_hash_path
    .read_text()
    .strip()
)

with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as f:

    frozen_design = json.load(f)

assert frozen_design["model_count"] == 30
assert frozen_design["image_count"] == 500
assert len(frozen_design["levels"]) == 7

print()
print(
    "✓ Frozen design verified"
)

print(
    "Design SHA256:",
    manifest_sha256
)

# ================================================================
# 4. Exact XAI subset/cache mapping
# ================================================================

assert len(xai) == 500
assert len(cache_rows) == 500

FULL_IMAGE_IDS = [
    str(
        xai.iloc[i]["image"]
    )
    for i in range(500)
]

FULL_CACHE_ROWS = np.asarray(
    cache_rows[:500],
    dtype=np.int64
)

assert len(set(FULL_IMAGE_IDS)) == 500
assert len(set(FULL_CACHE_ROWS.tolist())) == 500

# Open cache once in parent.
# Threads can safely read the same memory-mapped file.

HAIR_MM = np.load(
    "/kaggle/temp/isic_cache/hair_224.npy",
    mmap_mode="r"
)

assert HAIR_MM.shape == (
    23227,
    224,
    224,
    3
)

print(
    "✓ Hair cache opened:",
    HAIR_MM.shape
)

# ================================================================
# 5. Frozen model list
# ================================================================

FULL_MODEL_NAMES = sorted(
    MODEL_NAMES
)

assert len(FULL_MODEL_NAMES) == 30

GPU0_MODELS = FULL_MODEL_NAMES[:15]
GPU1_MODELS = FULL_MODEL_NAMES[15:]

print()
print("GPU 0:", GPU0_MODELS)
print()
print("GPU 1:", GPU1_MODELS)

# ================================================================
# 6. Frozen randomization levels
# ================================================================

FULL_LEVELS = [
    ("original", []),

    ("fc", [
        "fc"
    ]),

    ("fc+layer4", [
        "fc",
        "layer4"
    ]),

    ("fc+layer4+layer3", [
        "fc",
        "layer4",
        "layer3"
    ]),

    ("fc+layer4+layer3+layer2", [
        "fc",
        "layer4",
        "layer3",
        "layer2"
    ]),

    ("fc+layer4+layer3+layer2+layer1", [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1"
    ]),

    ("all", [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1"
    ]),
]

assert len(FULL_LEVELS) == 7

# ================================================================
# 7. Checkpoint root
# ================================================================

TRAINING_ROOT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "training_runs"
)

assert TRAINING_ROOT.exists()

for model_name in FULL_MODEL_NAMES:

    assert (
        TRAINING_ROOT
        / model_name
        / "final_epoch15.pt"
    ).exists()

print(
    "✓ All 30 final_epoch15 checkpoints verified"
)

# ================================================================
# 8. Thread-safe production model loader
# ================================================================

def load_production_model(
    model_name,
    device,
):

    ckpt = (
        TRAINING_ROOT
        / model_name
        / "final_epoch15.pt"
    )

    model = torchvision.models.resnet18(
        weights=None
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        2
    )

    checkpoint = torch.load(
        ckpt,
        map_location="cpu"
    )

    if isinstance(
        checkpoint,
        dict
    ):

        if "state_dict" in checkpoint:

            state_dict = (
                checkpoint[
                    "state_dict"
                ]
            )

        elif "model_state_dict" in checkpoint:

            state_dict = (
                checkpoint[
                    "model_state_dict"
                ]
            )

        else:

            state_dict = checkpoint

    else:

        state_dict = checkpoint

    state_dict = {
        (
            k.replace(
                "module.",
                "",
                1
            )
            if k.startswith(
                "module."
            )
            else k
        ): v
        for k, v in state_dict.items()
    }

    model.load_state_dict(
        state_dict,
        strict=True
    )

    model.eval()
    model.to(device)

    return model

# ================================================================
# 9. Exact frozen IG map
# ================================================================

def frozen_ig_map(
    model,
    hair_u8,
    device,
):

    mean = torch.tensor(
        ap._MEAN,
        device=device
    ).view(
        1,
        3,
        1,
        1
    )

    std = torch.tensor(
        ap._STD,
        device=device
    ).view(
        1,
        3,
        1,
        1
    )

    model.eval()

    x = torch.from_numpy(
        np.ascontiguousarray(
            hair_u8
        )
    ).to(
        device
    )

    x = (
        x.permute(
            2,
            0,
            1
        )
        .float()
        .unsqueeze(0)
        / 255.0
    )

    sigma = float(
        ap.IG_SIGMA
    )

    k = (
        2
        * int(
            math.ceil(
                3 * sigma
            )
        )
        + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[
            k,
            k
        ],
        sigma=[
            sigma,
            sigma
        ],
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    def zf(t):

        o = model(t)

        return (
            o[:, 1]
            - o[:, 0]
        )

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            )
            / n
        )

        for s in range(
            0,
            n,
            25
        ):

            a = (
                alphas[
                    s:s + 25
                ]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn
                + a * (
                    xn - bn
                )
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            (xn - bn)
            * acc
            / n
        )

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    - zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            )
            - delta
        )

        return (
            ig,
            delta,
            err
        )

    # Exact frozen production settings.
    ig, delta, err = run(
        int(ap.IG_STEPS)
    )

    tol = max(
        float(ap.IG_TOL_REL)
        * abs(delta),
        float(ap.IG_TAU_ABS)
    )

    if err > tol:

        ig, delta, err = run(
            int(ap.IG_RETRY_STEPS)
        )

    # Exact production reduction.
    a = (
        ig.abs()
        .sum(1)[0]
    )

    return (
        a.detach()
        .cpu()
        .numpy()
    )

# ================================================================
# 10. Spearman
# ================================================================

def spearman_corr(
    a,
    b,
):

    a = np.asarray(
        a
    ).ravel()

    b = np.asarray(
        b
    ).ravel()

    ra = (
        pd.Series(a)
        .rank(
            method="average"
        )
        .to_numpy()
    )

    rb = (
        pd.Series(b)
        .rank(
            method="average"
        )
        .to_numpy()
    )

    if (
        np.std(ra) == 0
        or np.std(rb) == 0
    ):

        return np.nan

    return float(
        np.corrcoef(
            ra,
            rb
        )[0, 1]
    )

# ================================================================
# 11. Randomization function
# ================================================================

def randomize_module_parameters(
    model,
    module_names,
    seed,
):

    for module_offset, module_name in enumerate(
        module_names
    ):

        module = getattr(
            model,
            module_name
        )

        module_seed = (
            int(seed)
            + 1000003 * module_offset
        )

        for parameter_offset, p in enumerate(
            module.parameters()
        ):

            parameter_seed = (
                module_seed
                + 1009 * parameter_offset
            )

            rng = torch.Generator(
                device=p.device
            )

            rng.manual_seed(
                parameter_seed
            )

            with torch.no_grad():

                if p.ndim >= 2:

                    fan_in = (
                        nn.init
                        ._calculate_correct_fan(
                            p,
                            "fan_in"
                        )
                    )

                    gain = (
                        nn.init
                        .calculate_gain(
                            "relu"
                        )
                    )

                    std = (
                        gain
                        / math.sqrt(
                            fan_in
                        )
                    )

                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * std
                    )

                else:

                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * 0.01
                    )

                p.copy_(
                    values
                )

    return model

# ================================================================
# 12. Single-GPU worker
# ================================================================

def run_gpu_worker(
    gpu_id,
    assigned_models,
):

    device = torch.device(
        f"cuda:{gpu_id}"
    )

    torch.cuda.set_device(
        gpu_id
    )

    worker_start = time.time()

    print()
    print(
        f"[GPU {gpu_id}] START — "
        f"{len(assigned_models)} models",
        flush=True
    )

    completed_models = []

    for model_pos, model_name in enumerate(
        assigned_models,
        start=1
    ):

        model_start = time.time()

        print(
            f"\n[GPU {gpu_id}] "
            f"MODEL {model_pos}/"
            f"{len(assigned_models)} "
            f"{model_name}",
            flush=True
        )

        # -------------------------------------------------------
        # Load one frozen production model.
        # -------------------------------------------------------

        base_model = load_production_model(
            model_name,
            device
        )

        base_model.eval()

        # -------------------------------------------------------
        # Reference attribution maps.
        # -------------------------------------------------------

        print(
            f"[GPU {gpu_id}] "
            f"Computing 500 reference maps...",
            flush=True
        )

        original_maps = {}

        for i, (
            cache_row,
            image_id
        ) in enumerate(
            zip(
                FULL_CACHE_ROWS,
                FULL_IMAGE_IDS
            ),
            start=1
        ):

            img = np.array(
                HAIR_MM[
                    int(cache_row)
                ],
                dtype=np.uint8,
                copy=True
            )

            with torch.enable_grad():

                original_maps[
                    image_id
                ] = frozen_ig_map(
                    base_model,
                    img,
                    device
                )

            if (
                i == 1
                or i % 100 == 0
                or i == 500
            ):

                print(
                    f"[GPU {gpu_id}] "
                    f"  reference "
                    f"{i}/500",
                    flush=True
                )

        # -------------------------------------------------------
        # Level 0: mathematical identity.
        # -------------------------------------------------------

        level0_path = (
            RESULT_DIR
            / (
                f"{model_name}"
                f"__level_0.csv"
            )
        )

        if not level0_path.exists():

            level0_df = pd.DataFrame({
                "model":
                    [model_name] * 500,

                "image":
                    FULL_IMAGE_IDS,

                "level":
                    [0] * 500,

                "level_name":
                    ["original"] * 500,

                "randomized_groups":
                    [""] * 500,

                "spearman_vs_original":
                    [1.0] * 500,
            })

            level0_df.to_csv(
                level0_path,
                index=False
            )

        print(
            f"[GPU {gpu_id}] "
            f"  level 0 complete",
            flush=True
        )

        # -------------------------------------------------------
        # Randomized levels 1-6.
        # -------------------------------------------------------

        for level_idx in range(
            1,
            7
        ):

            level_name, groups = (
                FULL_LEVELS[
                    level_idx
                ]
            )

            out_path = (
                RESULT_DIR
                / (
                    f"{model_name}"
                    f"__level_{level_idx}"
                    f".csv"
                )
            )

            if out_path.exists():

                try:

                    existing = pd.read_csv(
                        out_path
                    )

                    if (
                        len(existing)
                        == 500
                        and
                        existing[
                            "spearman_vs_original"
                        ].notna().all()
                    ):

                        print(
                            f"[GPU {gpu_id}] "
                            f"  level {level_idx} "
                            f"already complete — SKIP",
                            flush=True
                        )

                        continue

                except Exception:
                    pass

            print(
                f"[GPU {gpu_id}] "
                f"  LEVEL {level_idx}: "
                f"{level_name}",
                flush=True
            )

            # ---------------------------------------------------
            # Fresh copy of production model.
            # ---------------------------------------------------

            rand_model = copy.deepcopy(
                base_model
            ).to(
                device
            )

            rand_model.eval()

            randomize_module_parameters(
                rand_model,
                groups,
                seed=(
                    RAND_SEED
                    + 100000 * model_pos
                    + 1000 * level_idx
                    + gpu_id
                ),
            )

            rand_model.eval()

            records = []

            level_start = time.time()

            for i, (
                cache_row,
                image_id
            ) in enumerate(
                zip(
                    FULL_CACHE_ROWS,
                    FULL_IMAGE_IDS
                ),
                start=1
            ):

                img = np.array(
                    HAIR_MM[
                        int(cache_row)
                    ],
                    dtype=np.uint8,
                    copy=True
                )

                with torch.enable_grad():

                    randomized_map = (
                        frozen_ig_map(
                            rand_model,
                            img,
                            device
                        )
                    )

                corr = spearman_corr(
                    original_maps[
                        image_id
                    ],
                    randomized_map
                )

                records.append({
                    "model":
                        model_name,

                    "image":
                        image_id,

                    "level":
                        level_idx,

                    "level_name":
                        level_name,

                    "randomized_groups":
                        "+".join(groups),

                    "spearman_vs_original":
                        corr,
                })

                if (
                    i == 1
                    or i % 100 == 0
                    or i == 500
                ):

                    elapsed = (
                        time.time()
                        - level_start
                    )

                    print(
                        f"[GPU {gpu_id}] "
                        f"    {i}/500 "
                        f"({elapsed/60:.1f} min)",
                        flush=True
                    )

            level_df = pd.DataFrame(
                records
            )

            assert len(
                level_df
            ) == 500

            assert (
                level_df[
                    "image"
                ].nunique()
                == 500
            )

            assert (
                level_df[
                    "spearman_vs_original"
                ].notna().all()
            )

            level_df.to_csv(
                out_path,
                index=False
            )

            print(
                f"[GPU {gpu_id}] "
                f"  SAVED level {level_idx} "
                f"| mean="
                f"{level_df['spearman_vs_original'].mean():.6f}",
                flush=True
            )

            del rand_model

            gc.collect()

            torch.cuda.empty_cache()

        # -------------------------------------------------------
        # Model cleanup.
        # -------------------------------------------------------

        del original_maps
        del base_model

        gc.collect()

        torch.cuda.empty_cache()

        elapsed = (
            time.time()
            - model_start
        )

        completed_models.append(
            model_name
        )

        print(
            f"[GPU {gpu_id}] "
            f"MODEL COMPLETE: "
            f"{model_name} "
            f"({elapsed/60:.2f} min)",
            flush=True
        )

    total = (
        time.time()
        - worker_start
    )

    print(
        f"\n[GPU {gpu_id}] "
        f"WORKER COMPLETE — "
        f"{total/3600:.2f} h",
        flush=True
    )

    return {
        "gpu": gpu_id,
        "models": completed_models,
        "runtime_seconds": total,
    }

# ================================================================
# 13. Launch two threads
# ================================================================

print()
print("=" * 70)
print("LAUNCHING GPU 0 + GPU 1")
print("=" * 70)

global_start = time.time()

with ThreadPoolExecutor(
    max_workers=2
) as executor:

    future0 = executor.submit(
        run_gpu_worker,
        0,
        GPU0_MODELS
    )

    future1 = executor.submit(
        run_gpu_worker,
        1,
        GPU1_MODELS
    )

    # Wait for both.
    result0 = future0.result()
    result1 = future1.result()

print()
print("=" * 70)
print("BOTH GPU WORKERS FINISHED")
print("=" * 70)

print(result0)
print(result1)

# ================================================================
# 14. Audit all 210 result files
# ================================================================

expected_files = (
    30 * 7
)

actual_files = 0

for model_name in FULL_MODEL_NAMES:

    for level_idx in range(7):

        p = (
            RESULT_DIR
            / (
                f"{model_name}"
                f"__level_{level_idx}"
                f".csv"
            )
        )

        if not p.exists():
            continue

        df = pd.read_csv(
            p
        )

        if (
            len(df) == 500
            and
            df["image"].nunique() == 500
            and
            df[
                "spearman_vs_original"
            ].notna().all()
        ):

            actual_files += 1

print()
print("=" * 70)
print("RESULT FILE AUDIT")
print("=" * 70)

print(
    "Expected:",
    expected_files
)

print(
    "Complete:",
    actual_files
)

assert actual_files == expected_files

# ================================================================
# 15. Combine all results
# ================================================================

all_results = []

for model_name in FULL_MODEL_NAMES:

    for level_idx in range(7):

        p = (
            RESULT_DIR
            / (
                f"{model_name}"
                f"__level_{level_idx}"
                f".csv"
            )
        )

        all_results.append(
            pd.read_csv(
                p
            )
        )

all_results = pd.concat(
    all_results,
    ignore_index=True
)

assert len(
    all_results
) == 105000

assert (
    all_results[
        "spearman_vs_original"
    ].notna().all()
)

combined_path = (
    PR_DIR
    / "T1_4_PARAMETER_RANDOMIZATION_ALL.csv"
)

all_results.to_csv(
    combined_path,
    index=False
)

# ================================================================
# 16. Full summary
# ================================================================

summary = (
    all_results
    .groupby(
        [
            "level",
            "level_name",
            "randomized_groups",
        ],
        as_index=False
    )
    [
        "spearman_vs_original"
    ]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
    )
)

summary_path = (
    SUMMARY_DIR
    / "T1_4_PARAMETER_RANDOMIZATION_SUMMARY.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

print()
print("=" * 70)
print("FULL PARAMETER-RANDOMIZATION SUMMARY")
print("=" * 70)

print(
    summary.to_string(
        index=False
    )
)

# ================================================================
# 17. Model-level summary
# ================================================================

model_summary = (
    all_results
    .groupby(
        [
            "model",
            "level",
            "level_name",
        ],
        as_index=False
    )
    [
        "spearman_vs_original"
    ]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
    )
)

model_summary_path = (
    SUMMARY_DIR
    / "T1_4_PARAMETER_RANDOMIZATION_MODEL_SUMMARY.csv"
)

model_summary.to_csv(
    model_summary_path,
    index=False
)

# ================================================================
# 18. Final audit
# ================================================================

total_runtime = (
    time.time()
    - global_start
)

audit = {
    "analysis":
        "T1.4_parameter_randomization_full",

    "design_manifest_sha256":
        manifest_sha256,

    "gpu_0":
        "Tesla T4",

    "gpu_1":
        "Tesla T4",

    "models":
        30,

    "images":
        500,

    "levels":
        7,

    "rows":
        int(len(all_results)),

    "expected_rows":
        105000,

    "result_files":
        actual_files,

    "expected_result_files":
        210,

    "all_finite":
        bool(
            np.isfinite(
                all_results[
                    "spearman_vs_original"
                ].to_numpy()
            ).all()
        ),

    "runtime_seconds":
        total_runtime,

    "runtime_hours":
        total_runtime / 3600,
}

audit_path = (
    PR_DIR
    / "T1_4_PARAMETER_RANDOMIZATION_AUDIT.json"
)

audit_path.write_text(
    json.dumps(
        audit,
        indent=2
    ),
    encoding="utf-8"
)

print()
print("=" * 70)
print("CELL 18C COMPLETE")
print("=" * 70)

print(
    f"""
Rows       : {len(all_results):,}
Files      : {actual_files}/210
All finite : {audit["all_finite"]}

Runtime:
{total_runtime/3600:.2f} hours

Combined:
{combined_path}

Summary:
{summary_path}

Audit:
{audit_path}
"""
)

In [ ]:
from pathlib import Path
import shutil
import json
from datetime import datetime

BASE = Path("/kaggle/working/T1_4_parameter_randomization")

# Everything produced by the stopped 30x500 execution
PARTIAL_SRC = BASE / "model_level_results"

# New immutable archive location
PARTIAL_ARCHIVE = BASE / "T1_4a_FULL_30x500_PARTIAL_NOT_USED"

if PARTIAL_SRC.exists():
    if PARTIAL_ARCHIVE.exists():
        raise RuntimeError(f"Archive already exists: {PARTIAL_ARCHIVE}")

    shutil.move(str(PARTIAL_SRC), str(PARTIAL_ARCHIVE))
    print("Archived partial 30x500 run:")
    print(PARTIAL_ARCHIVE)
else:
    print("No model_level_results directory found.")

In [ ]:
files = sorted(PARTIAL_ARCHIVE.rglob("*")) if PARTIAL_ARCHIVE.exists() else []

for p in files:
    if p.is_file():
        print(p.relative_to(PARTIAL_ARCHIVE))

print("\nPartial archive file count:",
      sum(p.is_file() for p in files))

In [ ]:
# ================================================================
# T1.4a — FREEZE 100-IMAGE SUBSET
# Correctly stratified using the existing `malignant` field
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
from datetime import datetime, timezone

# ------------------------------------------------
# Paths
# ------------------------------------------------
PR_DIR = Path("/kaggle/working/T1_4a_parameter_randomization_12x100")
PR_DIR.mkdir(parents=True, exist_ok=True)

DESIGN_PATH = PR_DIR / "T1_4A_12X100_DESIGN.json"
SUBSET_PATH = PR_DIR / "T1_4A_100_IMAGE_SUBSET.csv"

LOCKED_XAI_SUBSET = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project/"
    "step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv"
)

# ------------------------------------------------
# Frozen design constants
# ------------------------------------------------
MASTER_SEED = 20260925
SUBSET_NAMESPACE = "param-rand-subset-v1"

N_IMAGES = 100
N_PER_MALIGNANCY = 50

RHO_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
REPLICATES = [1, 2]

RANDOMIZATION_LEVELS = [
    {"level": 0, "name": "original", "randomized": []},
    {"level": 1, "name": "fc", "randomized": ["fc"]},
    {"level": 2, "name": "fc_layer4",
     "randomized": ["fc", "layer4"]},
    {"level": 3, "name": "fc_layer4_layer3",
     "randomized": ["fc", "layer4", "layer3"]},
    {"level": 4, "name": "fc_layer4_layer3_layer2",
     "randomized": ["fc", "layer4", "layer3", "layer2"]},
    {"level": 5, "name": "fc_layer4_layer3_layer2_layer1",
     "randomized": ["fc", "layer4", "layer3", "layer2", "layer1"]},
    {"level": 6, "name": "all",
     "randomized": [
         "fc", "layer4", "layer3", "layer2",
         "layer1", "conv1", "bn1"
     ]},
]

# ------------------------------------------------
# Load locked 500-image subset
# ------------------------------------------------
xai = pd.read_csv(LOCKED_XAI_SUBSET)

assert len(xai) == 500
assert xai["image"].nunique() == 500

print("Locked XAI subset:")
print("  rows:", len(xai))
print("  unique images:", xai["image"].nunique())
print("\nDiagnostic classes:")
print(xai["class_name"].value_counts().to_string())

# ------------------------------------------------
# Validate the actual binary malignancy field
# ------------------------------------------------
assert "malignant" in xai.columns, \
    "Expected `malignant` column is missing."

print("\nMalignant field:")
print(xai["malignant"].value_counts(dropna=False).sort_index())

assert xai["malignant"].notna().all(), \
    "`malignant` contains missing values."

# Accept either 0/1 numeric or boolean representation
malignant_values = set(xai["malignant"].unique())

assert malignant_values.issubset({0, 1, False, True}), (
    f"Unexpected values in malignant column: {malignant_values}"
)

# Normalize explicitly to integer 0/1
xai["malignant"] = xai["malignant"].astype(int)

mal_counts = xai["malignant"].value_counts().sort_index()

assert 0 in mal_counts and 1 in mal_counts
assert mal_counts[0] >= N_PER_MALIGNANCY
assert mal_counts[1] >= N_PER_MALIGNANCY

print("\nBinary malignancy counts:")
print("  benign (0):", int(mal_counts[0]))
print("  malignant (1):", int(mal_counts[1]))

# ------------------------------------------------
# Deterministic SHA256 ordering
# ------------------------------------------------
def selection_hash(image_id):
    payload = (
        f"{MASTER_SEED}|{SUBSET_NAMESPACE}|{image_id}"
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

xai["selection_hash"] = xai["image"].map(selection_hash)

# ------------------------------------------------
# Select exactly 50 benign + 50 malignant
# ------------------------------------------------
selected_parts = []

for malignant_value in [0, 1]:

    part = (
        xai[xai["malignant"] == malignant_value]
        .sort_values(["selection_hash", "image"])
        .head(N_PER_MALIGNANCY)
        .copy()
    )

    assert len(part) == N_PER_MALIGNANCY

    selected_parts.append(part)

subset = (
    pd.concat(selected_parts, ignore_index=True)
    .sort_values(["malignant", "selection_hash", "image"])
    .reset_index(drop=True)
)

# ------------------------------------------------
# Integrity checks
# ------------------------------------------------
assert len(subset) == 100
assert subset["image"].nunique() == 100
assert subset["lesion_id"].nunique() == 100

assert subset["malignant"].value_counts().to_dict() == {
    0: 50,
    1: 50
}

# ------------------------------------------------
# Save deterministic subset
# ------------------------------------------------
subset.to_csv(SUBSET_PATH, index=False)

subset_sha256 = hashlib.sha256(
    SUBSET_PATH.read_bytes()
).hexdigest()

# ------------------------------------------------
# Freeze the 12-model set
# ------------------------------------------------
models = [
    f"rho_{rho:.1f}_rep_{rep}"
    for rho in RHO_LEVELS
    for rep in REPLICATES
]

assert len(models) == 12

# ------------------------------------------------
# Freeze manifest
# ------------------------------------------------
manifest = {
    "experiment": "T1.4a_parameter_randomization",
    "design_version": "12x100_v1",

    "created_utc": datetime.now(timezone.utc).isoformat(),

    "model_count": 12,
    "models": models,

    "rho_levels": RHO_LEVELS,
    "replicates_per_rho": REPLICATES,

    "image_population": {
        "source": "LOCKED_XAI_SUBSET.csv",
        "source_size": 500,
        "selected_size": 100,
        "selection_unit": "image",
        "malignancy_stratified": True,
        "benign": 50,
        "malignant": 50,
    },

    "subset_selection": {
        "master_seed": MASTER_SEED,
        "namespace": SUBSET_NAMESPACE,
        "method": (
            "SHA256(master_seed | namespace | image_id); "
            "select smallest 50 hashes among benign and "
            "smallest 50 hashes among malignant"
        ),
        "subset_sha256": subset_sha256,
        "subset_file": str(SUBSET_PATH),
    },

    "cascade": RANDOMIZATION_LEVELS,

    "total_output_rows": 12 * 100 * 7,
    "total_ig_computations": 12 * 100 * 7,

    "checkpoint_policy": "final_epoch15",

    "randomization_protocol": {
        "method": "fresh deterministic parameter initialization",
        "original_checkpoint_modified": False,
        "parameter_generator_is_local_per_parameter": True,
    },

    "similarity_metric": (
        "pixelwise Spearman correlation between absolute-channel-"
        "summed IG attribution map of randomized model and "
        "original trained-model attribution map"
    ),

    "partial_previous_run": {
        "used_in_current_analysis": False,
        "directory": (
            "/kaggle/working/T1_4_parameter_randomization/"
            "T1_4a_FULL_30x500_PARTIAL_NOT_USED"
        ),
    },
}

DESIGN_PATH.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------
# Final report
# ------------------------------------------------
print("\n" + "=" * 70)
print("T1.4a 12×100 DESIGN FROZEN")
print("=" * 70)

print("\nModels:", len(models))
print("Images:", len(subset))
print("Cascade levels:", len(RANDOMIZATION_LEVELS))
print("Total rows:", 12 * 100 * 7)
print("Total IG computations:", 12 * 100 * 7)

print("\nModel set:")
for m in models:
    print(" ", m)

print("\nSelected malignancy balance:")
print(subset["malignant"].value_counts().sort_index())

print("\nSelected diagnostic classes:")
print(subset["class_name"].value_counts().to_string())

print("\nSubset SHA256:")
print(subset_sha256)

print("\nDesign manifest:")
print(DESIGN_PATH)

In [ ]:
# ================================================================
# T1.4a — FREEZE EXACT PARAMETER-RANDOMIZATION SEED PROTOCOL
# ================================================================

import json
from pathlib import Path

DESIGN_PATH = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/"
    "T1_4A_12X100_DESIGN.json"
)

manifest = json.loads(DESIGN_PATH.read_text())

manifest["randomization_protocol"]["seed_derivation"] = {
    "scheme": (
        "SHA256(master_seed | namespace | model_name | "
        "cascade_level | parameter_name)"
    ),
    "master_seed": 20260925,
    "namespace": "param-randomization-v1",
    "gpu_assignment_independent": True,
    "purpose": (
        "Ensure identical randomized parameters regardless of "
        "GPU/thread assignment."
    ),
}

# Freeze exact ordering as well
manifest["randomization_protocol"]["parameter_order"] = [
    "fc",
    "layer4",
    "layer3",
    "layer2",
    "layer1",
    "conv1",
    "bn1",
]

DESIGN_PATH.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("SEED PROTOCOL FROZEN")
print(json.dumps(
    manifest["randomization_protocol"],
    indent=2
))

In [ ]:
# ================================================================
# T1.4a — PARAMETER RANDOMIZATION
# FROZEN DESIGN: 12 MODELS × 100 IMAGES × 7 LEVELS
#
# IMPORTANT:
# - Do NOT modify frozen checkpoints.
# - Do NOT use the old 30×500 results.
# - Randomization seeds are GPU-independent.
# - Same 100 images are used for every model.
# ================================================================

import os
import gc
import json
import math
import time
import hashlib
import traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.functional as TF

# ------------------------------------------------
# Paths
# ------------------------------------------------
PR_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100"
)

DESIGN_PATH = PR_DIR / "T1_4A_12X100_DESIGN.json"
SUBSET_PATH = PR_DIR / "T1_4A_100_IMAGE_SUBSET.csv"

CACHE_DIR = Path("/kaggle/temp/isic_cache")
HAIR_PATH = CACHE_DIR / "hair_224.npy"

RESULT_DIR = PR_DIR / "model_level_results"
SUMMARY_DIR = PR_DIR / "summaries"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Load frozen design
# ------------------------------------------------
with open(DESIGN_PATH, "r", encoding="utf-8") as f:
    DESIGN = json.load(f)

subset = pd.read_csv(SUBSET_PATH)

assert len(subset) == 100
assert subset["image"].nunique() == 100
assert subset["lesion_id"].nunique() == 100

assert subset["malignant"].value_counts().to_dict() == {
    0: 50,
    1: 50
}

MODELS = DESIGN["models"]
CASCADE = DESIGN["cascade"]

assert len(MODELS) == 12
assert len(CASCADE) == 7

MASTER_SEED = DESIGN["randomization_protocol"]["seed_derivation"]["master_seed"]
SEED_NAMESPACE = DESIGN["randomization_protocol"]["seed_derivation"]["namespace"]

print("=" * 72)
print("T1.4a PARAMETER RANDOMIZATION — 12 × 100")
print("=" * 72)
print("Models:", len(MODELS))
print("Images:", len(subset))
print("Cascade levels:", len(CASCADE))
print("Expected rows:", 12 * 100 * 7)
print("Expected IG maps:", 12 * 100 * 7)
print("Seed namespace:", SEED_NAMESPACE)

# ------------------------------------------------
# Verify cache
# ------------------------------------------------
hair_mm = np.load(HAIR_PATH, mmap_mode="r")

print("\nHair cache:", hair_mm.shape, hair_mm.dtype)

assert hair_mm.shape[1:] == (224, 224, 3)
assert hair_mm.dtype == np.uint8

# ------------------------------------------------
# Production model root
# ------------------------------------------------
TRAINING_ROOT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "training_runs"
)

# ------------------------------------------------
# Production constants
# ------------------------------------------------
_MEAN = (0.485, 0.456, 0.406)
_STD = (0.229, 0.224, 0.225)

IG_STEPS = 50
IG_RETRY_STEPS = 100
IG_SIGMA = 10.0
IG_TOL_REL = 0.05
IG_TAU_ABS = 0.05
XAI_EPS = 1e-8
IG_CHUNK = 25

# ------------------------------------------------
# Production-faithful model loader
# ------------------------------------------------
def load_model(model_name, device):

    ckpt = TRAINING_ROOT / model_name / "final_epoch15.pt"

    assert ckpt.exists(), f"Missing checkpoint: {ckpt}"

    model = torchvision.models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)

    checkpoint = torch.load(
        ckpt,
        map_location="cpu"
    )

    if isinstance(checkpoint, dict):

        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]

        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]

        else:
            state_dict = checkpoint

    else:
        state_dict = checkpoint

    state_dict = {
        k.replace("module.", "", 1)
        if k.startswith("module.")
        else k: v
        for k, v in state_dict.items()
    }

    model.load_state_dict(
        state_dict,
        strict=True
    )

    model.eval()
    model.to(device)

    return model


# ------------------------------------------------
# Exact production-faithful IG map
# ------------------------------------------------
def production_ig_map(model, hair_u8, device):

    mean = torch.tensor(
        _MEAN,
        device=device
    ).view(1, 3, 1, 1)

    std = torch.tensor(
        _STD,
        device=device
    ).view(1, 3, 1, 1)

    model.eval()

    x = (
        torch.from_numpy(
            np.ascontiguousarray(hair_u8)
        )
        .to(device)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = 2 * int(math.ceil(3 * IG_SIGMA)) + 1

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[IG_SIGMA, IG_SIGMA]
    )

    xn = (x - mean) / std
    bn = (base - mean) / std

    def zf(t):
        o = model(t)
        return o[:, 1] - o[:, 0]

    def run(n):

        acc = torch.zeros_like(xn)

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            ) / n
        )

        for s in range(0, n, IG_CHUNK):

            a = alphas[s:s + IG_CHUNK].view(
                -1, 1, 1, 1
            )

            pts = (
                bn + a * (xn - bn)
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (xn - bn) * acc / n

        with torch.no_grad():

            delta = float(
                (
                    zf(xn) - zf(bn)
                ).item()
            )

        err = abs(
            float(ig.sum().item()) - delta
        )

        return ig, delta, err

    ig, delta, err = run(IG_STEPS)

    tol = max(
        IG_TOL_REL * abs(delta),
        IG_TAU_ABS
    )

    used = IG_STEPS

    if err > tol:

        ig, delta, err = run(
            IG_RETRY_STEPS
        )

        used = IG_RETRY_STEPS

    # Exact production attribution reduction:
    # absolute attribution summed over channels
    amap = (
        ig.abs()
        .sum(1)[0]
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    return amap, {
        "delta": delta,
        "completeness_error": err,
        "tolerance": tol,
        "steps_used": used,
        "completeness_failed": bool(err > tol),
    }


# ------------------------------------------------
# GPU-independent deterministic seed
# ------------------------------------------------
def parameter_seed(
    model_name,
    cascade_level,
    parameter_name
):

    material = (
        f"{MASTER_SEED}|"
        f"{SEED_NAMESPACE}|"
        f"{model_name}|"
        f"{cascade_level}|"
        f"{parameter_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).hexdigest()

    seed = int(
        digest[:16],
        16
    )

    seed = seed % (2**63 - 1)

    return seed


# ------------------------------------------------
# Deterministically randomize selected parameters
# ------------------------------------------------
def randomize_parameters(
    model,
    model_name,
    cascade_level,
    randomized_names
):

    randomized_names = set(randomized_names)

    # Map cascade names to actual modules.
    module_map = {
        "fc": model.fc,
        "layer4": model.layer4,
        "layer3": model.layer3,
        "layer2": model.layer2,
        "layer1": model.layer1,
        "conv1": model.conv1,
        "bn1": model.bn1,
    }

    # Randomize parameters in deterministic parameter-name order.
    for module_name in [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1",
    ]:

        if module_name not in randomized_names:
            continue

        module = module_map[module_name]

        for parameter_name, p in module.named_parameters():

            # Fully-qualified deterministic identifier
            full_name = (
                f"{module_name}.{parameter_name}"
            )

            seed = parameter_seed(
                model_name=model_name,
                cascade_level=cascade_level,
                parameter_name=full_name
            )

            # IMPORTANT:
            # generator must live on the same device as p
            generator = torch.Generator(
                device=p.device
            )

            generator.manual_seed(seed)

            with torch.no_grad():

                # Match the tensor's dtype/device.
                #
                # Kaiming-style initialization for weight tensors;
                # zero for bias tensors.
                if parameter_name.endswith("bias"):

                    p.zero_()

                else:

                    fan_in = nn.init._calculate_correct_fan(
                        p,
                        "fan_in"
                    )

                    gain = nn.init.calculate_gain(
                        "relu"
                    )

                    std = gain / math.sqrt(fan_in)

                    values = torch.randn(
                        p.shape,
                        generator=generator,
                        device=p.device,
                        dtype=p.dtype
                    ) * std

                    p.copy_(values)


# ------------------------------------------------
# Pixelwise Spearman correlation
# ------------------------------------------------
def pixelwise_spearman(a, b):

    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)

    # Rank without scipy dependency.
    # Stable sorting ensures deterministic tie handling.
    ra = np.empty_like(a, dtype=np.float64)
    rb = np.empty_like(b, dtype=np.float64)

    ia = np.argsort(a, kind="mergesort")
    ib = np.argsort(b, kind="mergesort")

    ra[ia] = np.arange(len(a), dtype=np.float64)
    rb[ib] = np.arange(len(b), dtype=np.float64)

    ra -= ra.mean()
    rb -= rb.mean()

    denom = np.sqrt(
        np.sum(ra * ra) *
        np.sum(rb * rb)
    )

    if denom == 0:
        return np.nan

    return float(
        np.sum(ra * rb) / denom
    )


# ------------------------------------------------
# One model worker
# ------------------------------------------------
def run_model(
    model_name,
    gpu_id,
    model_index
):

    device = torch.device(
        f"cuda:{gpu_id}"
    )

    torch.cuda.set_device(device)

    start_time = time.time()

    print(
        f"\n[{model_name}] START "
        f"GPU={gpu_id}"
    )

    # ------------------------------------------------
    # Load pristine production checkpoint
    # ------------------------------------------------
    model = load_model(
        model_name,
        device
    )

    # ------------------------------------------------
    # Original trained-model attribution maps
    # ------------------------------------------------
    reference_maps = {}

    reference_meta = {}

    for i, row in subset.iterrows():

        image_id = row["image"]
        cache_row = int(row["row"])

        # Writable copy from read-only memmap
        hair = np.array(
            hair_mm[cache_row],
            copy=True
        )

        amap, meta = production_ig_map(
            model,
            hair,
            device
        )

        reference_maps[image_id] = amap
        reference_meta[image_id] = meta

    print(
        f"[{model_name}] "
        f"reference maps complete"
    )

    model_results = []

    # ------------------------------------------------
    # Cascade
    # ------------------------------------------------
    for cascade in CASCADE:

        level = int(
            cascade["level"]
        )

        level_name = cascade["name"]
        randomized = cascade["randomized"]

        level_start = time.time()

        # ------------------------------------------------
        # Level 0 = original model.
        #
        # Similarity is analytically 1.0.
        # No redundant IG calculation.
        # ------------------------------------------------
        if level == 0:

            for _, row in subset.iterrows():

                image_id = row["image"]

                ref_meta = reference_meta[
                    image_id
                ]

                model_results.append({
                    "model_name": model_name,
                    "rho": float(
                        model_name.split("_")[1]
                    ),
                    "replicate": int(
                        model_name.split("_")[-1]
                    ),
                    "image": image_id,
                    "lesion_id": row["lesion_id"],
                    "class_name": row["class_name"],
                    "malignant": int(row["malignant"]),
                    "cascade_level": level,
                    "cascade_name": level_name,
                    "randomized_parameters": "",
                    "similarity_spearman": 1.0,
                    "reference_steps_used": ref_meta["steps_used"],
                    "reference_completeness_error":
                        ref_meta["completeness_error"],
                    "randomized_steps_used": 0,
                    "randomized_completeness_error": 0.0,
                    "randomized_completeness_failed": False,
                })

        # ================================================================
# T1.4a — FROZEN PARAMETER RANDOMIZATION
#
# Initialization distribution = EXACT CELL-16 PILOT
# Seed derivation = FROZEN 12×100 GPU-INDEPENDENT PROTOCOL
# ================================================================

def parameter_seed(
    model_name,
    cascade_level,
    parameter_name
):
    material = (
        f"{MASTER_SEED}|"
        f"{SEED_NAMESPACE}|"
        f"{model_name}|"
        f"{cascade_level}|"
        f"{parameter_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).hexdigest()

    seed = int(digest[:16], 16)

    return seed % (2**63 - 1)


def randomize_parameters(
    model,
    model_name,
    cascade_level,
    randomized_names
):
    """
    Exact Cell-16 initialization distribution, with the
    frozen GPU-independent T1.4a seed protocol.

    >=2-D parameters:
        N(0, gain/sqrt(fan_in)), gain=ReLU gain

    1-D parameters:
        N(0, 0.01)

    Every parameter gets its own deterministic generator.
    """

    randomized_names = set(
        randomized_names
    )

    module_map = {
        "fc": model.fc,
        "layer4": model.layer4,
        "layer3": model.layer3,
        "layer2": model.layer2,
        "layer1": model.layer1,
        "conv1": model.conv1,
        "bn1": model.bn1,
    }

    # Exact frozen module ordering
    module_order = [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1",
    ]

    for module_name in module_order:

        if module_name not in randomized_names:
            continue

        module = module_map[module_name]

        # Named parameter order is deterministic and lets the
        # seed identify the actual parameter rather than relying
        # on GPU/thread execution order.
        for parameter_name, p in module.named_parameters():

            full_parameter_name = (
                f"{module_name}.{parameter_name}"
            )

            seed = parameter_seed(
                model_name=model_name,
                cascade_level=cascade_level,
                parameter_name=full_parameter_name,
            )

            # Generator MUST be on same device as parameter.
            rng = torch.Generator(
                device=p.device
            )

            rng.manual_seed(seed)

            with torch.no_grad():

                # ------------------------------------------------
                # EXACT CELL-16 PILOT BEHAVIOR
                # ------------------------------------------------
                if p.ndim >= 2:

                    fan_in = nn.init._calculate_correct_fan(
                        p,
                        "fan_in"
                    )

                    gain = nn.init.calculate_gain(
                        "relu"
                    )

                    std = gain / math.sqrt(
                        fan_in
                    )

                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * std
                    )

                else:

                    # EXACT pilot behavior for 1-D parameters,
                    # including BatchNorm weights/biases.
                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * 0.01
                    )

                p.copy_(values)

    return model
    # ------------------------------------------------
    # Save model result
    # ------------------------------------------------
    result_df = pd.DataFrame(
        model_results
    )

    expected_rows = 100 * 7

    assert len(result_df) == expected_rows

    result_path = (
        RESULT_DIR /
        f"{model_name}.csv"
    )

    result_df.to_csv(
        result_path,
        index=False
    )

    elapsed_total = time.time() - start_time

    print(
        f"[{model_name}] DONE "
        f"{elapsed_total / 60:.2f} min "
        f"rows={len(result_df)}"
    )

    del model
    del reference_maps
    del reference_meta

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "model_name": model_name,
        "gpu_id": gpu_id,
        "rows": len(result_df),
        "runtime_seconds": elapsed_total,
        "result_path": str(result_path),
    }


# ------------------------------------------------
# Fixed GPU assignment
# ------------------------------------------------
GPU_ASSIGNMENT = {}

for i, model_name in enumerate(MODELS):

    # 6 models per GPU
    gpu_id = 0 if i < 6 else 1

    GPU_ASSIGNMENT[model_name] = gpu_id

print("\nGPU assignment:")
for model_name in MODELS:
    print(
        f"  {model_name:18s} -> GPU "
        f"{GPU_ASSIGNMENT[model_name]}"
    )

# ------------------------------------------------
# Verify GPUs
# ------------------------------------------------
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2

print("\nCUDA devices:")
for i in range(torch.cuda.device_count()):
    print(
        i,
        torch.cuda.get_device_name(i)
    )

# ------------------------------------------------
# Execute two model workers concurrently
# ------------------------------------------------
overall_start = time.time()

results = []
errors = []

with ThreadPoolExecutor(
    max_workers=2
) as executor:

    futures = {}

    for model_index, model_name in enumerate(MODELS):

        gpu_id = GPU_ASSIGNMENT[
            model_name
        ]

        future = executor.submit(
            run_model,
            model_name,
            gpu_id,
            model_index
        )

        futures[future] = model_name

    for future in as_completed(futures):

        model_name = futures[future]

        try:

            result = future.result()
            results.append(result)

        except Exception as exc:

            errors.append({
                "model_name": model_name,
                "error": repr(exc),
                "traceback": traceback.format_exc(),
            })

            print(
                f"\n!!! FAILED: {model_name}"
            )
            print(
                traceback.format_exc()
            )

# ------------------------------------------------
# Save execution audit
# ------------------------------------------------
overall_elapsed = (
    time.time() - overall_start
)

execution_audit = {
    "design": "T1.4a 12x100 v1",
    "models_expected": 12,
    "models_completed": len(results),
    "models_failed": len(errors),
    "expected_rows": 8400,
    "partial_previous_run_used": False,
    "gpu_assignment": GPU_ASSIGNMENT,
    "runtime_seconds": overall_elapsed,
    "runtime_minutes": overall_elapsed / 60,
    "results": results,
    "errors": errors,
}

AUDIT_PATH = (
    SUMMARY_DIR /
    "T1_4A_12X100_EXECUTION_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        execution_audit,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("EXECUTION COMPLETE")
print("=" * 72)

print(
    f"Completed models: "
    f"{len(results)}/12"
)

print(
    f"Failed models: "
    f"{len(errors)}"
)

print(
    f"Runtime: "
    f"{overall_elapsed / 60:.2f} min"
)

print(
    "Audit:",
    AUDIT_PATH
)

if errors:
    raise RuntimeError(
        f"{len(errors)} model(s) failed. "
        f"DO NOT aggregate results yet."
    )

In [ ]:
# ================================================================
# Preserve failed execution artifacts
# ================================================================

from pathlib import Path
import shutil

BASE = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100"
)

FAILED_ARCHIVE = BASE / (
    "T1_4a_12x100_FAILED_FANIN_BUG_NOT_USED"
)

RESULT_DIR = BASE / "model_level_results"

if FAILED_ARCHIVE.exists():
    raise RuntimeError(
        f"Archive already exists: {FAILED_ARCHIVE}"
    )

if RESULT_DIR.exists():

    # Check whether any completed result CSVs exist
    csvs = list(
        RESULT_DIR.glob("*.csv")
    )

    print(
        "Existing result CSVs:",
        len(csvs)
    )

    if csvs:
        print(
            "WARNING — these will be archived, "
            "NOT used in final analysis:"
        )

        for p in csvs:
            print(" ", p.name)

    shutil.move(
        str(RESULT_DIR),
        str(FAILED_ARCHIVE / "model_level_results")
    )

    print(
        "\nArchived failed-run result directory."
    )

# Recreate clean result directory
RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "\nClean result directory ready:"
)
print(RESULT_DIR)

In [ ]:
# ================================================================
# T1.4a — RANDOMIZATION UNIT TEST
# ================================================================

TEST_MODEL_NAME = "rho_0.0_rep_1"
TEST_LEVEL = 6

device = torch.device("cuda:0")

test_model = load_model(
    TEST_MODEL_NAME,
    device
)

# Save original parameter statistics
original_stats = {}

for name, p in test_model.named_parameters():
    original_stats[name] = (
        tuple(p.shape),
        p.detach().float().mean().item(),
        p.detach().float().std().item(),
    )

# Randomize ALL parameters
randomize_parameters(
    test_model,
    TEST_MODEL_NAME,
    TEST_LEVEL,
    [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1",
    ]
)

# ------------------------------------------------
# Verify no NaN / Inf
# ------------------------------------------------
bad = []

for name, p in test_model.named_parameters():

    if not torch.isfinite(p).all():
        bad.append(name)

assert not bad, (
    f"Non-finite randomized parameters: {bad}"
)

# ------------------------------------------------
# Verify 1-D parameters were successfully randomized
# ------------------------------------------------
one_d = []

for name, p in test_model.named_parameters():

    if p.ndim == 1:
        one_d.append({
            "name": name,
            "shape": tuple(p.shape),
            "mean": float(p.mean().item()),
            "std": float(p.std().item()),
        })

print("=" * 70)
print("RANDOMIZATION UNIT TEST")
print("=" * 70)

print(
    "1-D parameters successfully randomized:",
    len(one_d)
)

for x in one_d:
    print(
        f"{x['name']:35s} "
        f"shape={x['shape']} "
        f"mean={x['mean']:.6f} "
        f"std={x['std']:.6f}"
    )

print(
    "\nAll randomized parameters finite: PASS"
)

del test_model
gc.collect()
torch.cuda.empty_cache()

print(
    "\nRANDOMIZATION UNIT TEST: PASS"
)

In [ ]:
# ================================================================
# T1.4a — FROZEN PARAMETER RANDOMIZATION
#
# Initialization distribution = EXACT CELL-16 PILOT
# Seed derivation = FROZEN 12×100 GPU-INDEPENDENT PROTOCOL
# ================================================================

def parameter_seed(
    model_name,
    cascade_level,
    parameter_name
):
    material = (
        f"{MASTER_SEED}|"
        f"{SEED_NAMESPACE}|"
        f"{model_name}|"
        f"{cascade_level}|"
        f"{parameter_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).hexdigest()

    seed = int(digest[:16], 16)

    return seed % (2**63 - 1)


def randomize_parameters(
    model,
    model_name,
    cascade_level,
    randomized_names
):
    """
    Exact Cell-16 initialization distribution, with the
    frozen GPU-independent T1.4a seed protocol.

    >=2-D parameters:
        N(0, gain/sqrt(fan_in)), gain=ReLU gain

    1-D parameters:
        N(0, 0.01)

    Every parameter gets its own deterministic generator.
    """

    randomized_names = set(
        randomized_names
    )

    module_map = {
        "fc": model.fc,
        "layer4": model.layer4,
        "layer3": model.layer3,
        "layer2": model.layer2,
        "layer1": model.layer1,
        "conv1": model.conv1,
        "bn1": model.bn1,
    }

    # Exact frozen module ordering
    module_order = [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1",
    ]

    for module_name in module_order:

        if module_name not in randomized_names:
            continue

        module = module_map[module_name]

        # Named parameter order is deterministic and lets the
        # seed identify the actual parameter rather than relying
        # on GPU/thread execution order.
        for parameter_name, p in module.named_parameters():

            full_parameter_name = (
                f"{module_name}.{parameter_name}"
            )

            seed = parameter_seed(
                model_name=model_name,
                cascade_level=cascade_level,
                parameter_name=full_parameter_name,
            )

            # Generator MUST be on same device as parameter.
            rng = torch.Generator(
                device=p.device
            )

            rng.manual_seed(seed)

            with torch.no_grad():

                # ------------------------------------------------
                # EXACT CELL-16 PILOT BEHAVIOR
                # ------------------------------------------------
                if p.ndim >= 2:

                    fan_in = nn.init._calculate_correct_fan(
                        p,
                        "fan_in"
                    )

                    gain = nn.init.calculate_gain(
                        "relu"
                    )

                    std = gain / math.sqrt(
                        fan_in
                    )

                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * std
                    )

                else:

                    # EXACT pilot behavior for 1-D parameters,
                    # including BatchNorm weights/biases.
                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * 0.01
                    )

                p.copy_(values)

    return model

In [ ]:
import inspect

print(inspect.getsource(randomize_parameters))

In [ ]:
# ================================================================
# T1.4a — RANDOMIZATION UNIT TEST
# ================================================================

TEST_MODEL_NAME = "rho_0.0_rep_1"
TEST_LEVEL = 6

device = torch.device("cuda:0")

test_model = load_model(
    TEST_MODEL_NAME,
    device
)

# Verify the function currently loaded by the kernel
import inspect
src = inspect.getsource(randomize_parameters)

assert "if p.ndim >= 2" in src
assert "* 0.01" in src

print("Loaded randomize_parameters(): CORRECT")

# Randomize ALL parameters
randomize_parameters(
    test_model,
    TEST_MODEL_NAME,
    TEST_LEVEL,
    [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1",
    ]
)

# ------------------------------------------------
# Check every parameter is finite
# ------------------------------------------------
bad = []

for name, p in test_model.named_parameters():
    if not torch.isfinite(p).all():
        bad.append(name)

assert not bad, (
    f"Non-finite randomized parameters: {bad}"
)

# ------------------------------------------------
# Inspect 1-D parameters
# ------------------------------------------------
one_d = []

for name, p in test_model.named_parameters():

    if p.ndim == 1:

        one_d.append({
            "name": name,
            "shape": tuple(p.shape),
            "mean": float(p.detach().float().mean().item()),
            "std": float(
                p.detach().float().std().item()
            ),
        })

print("\n1-D parameters:")
for x in one_d:
    print(
        f"  {x['name']:40s} "
        f"shape={str(x['shape']):12s} "
        f"mean={x['mean']:.6f} "
        f"std={x['std']:.6f}"
    )

assert len(one_d) > 0

print("\nAll randomized parameters finite: PASS")
print("1-D parameter handling: PASS")

del test_model
gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("RANDOMIZATION UNIT TEST: PASS")
print("=" * 70)

In [ ]:
# ================================================================
# T1.4a — DETERMINISTIC RANDOMIZATION REPEATABILITY TEST
# ================================================================

TEST_MODEL_NAME = "rho_0.0_rep_1"
TEST_LEVEL = 6

TEST_GROUPS = [
    "fc",
    "layer4",
    "layer3",
    "layer2",
    "layer1",
    "conv1",
    "bn1",
]

device = torch.device("cuda:0")

print("=" * 70)
print("DETERMINISTIC RANDOMIZATION REPEATABILITY TEST")
print("=" * 70)

# ------------------------------------------------
# Create two independent copies from the same
# untouched production checkpoint
# ------------------------------------------------
model_a = load_model(
    TEST_MODEL_NAME,
    device
)

model_b = load_model(
    TEST_MODEL_NAME,
    device
)

# ------------------------------------------------
# Randomize both independently using EXACTLY the
# same frozen seed protocol
# ------------------------------------------------
randomize_parameters(
    model_a,
    TEST_MODEL_NAME,
    TEST_LEVEL,
    TEST_GROUPS
)

randomize_parameters(
    model_b,
    TEST_MODEL_NAME,
    TEST_LEVEL,
    TEST_GROUPS
)

# ------------------------------------------------
# Compare every parameter
# ------------------------------------------------
mismatches = []

for name_a, p_a in model_a.named_parameters():

    p_b = dict(
        model_b.named_parameters()
    )[name_a]

    if not torch.equal(p_a, p_b):

        max_diff = (
            p_a - p_b
        ).abs().max().item()

        mismatches.append({
            "parameter": name_a,
            "max_abs_difference": max_diff,
        })

# ------------------------------------------------
# Assertions
# ------------------------------------------------
assert not mismatches, (
    "Deterministic randomization FAILED:\n"
    + str(mismatches[:10])
)

print("\nParameter reproducibility: PASS")
print("All randomized parameters are bitwise identical.")

# ------------------------------------------------
# Check that the original production checkpoint
# itself was not modified by either operation.
#
# We prove this by loading a third pristine model
# and checking that it differs from the randomized
# copies for randomized parameters.
# ------------------------------------------------
model_original = load_model(
    TEST_MODEL_NAME,
    device
)

randomized_difference_count = 0

for name, p_orig in model_original.named_parameters():

    p_rand = dict(
        model_a.named_parameters()
    )[name]

    if not torch.equal(
        p_orig,
        p_rand
    ):
        randomized_difference_count += 1

assert randomized_difference_count > 0

print(
    "Production checkpoint remains pristine: PASS"
)

# ------------------------------------------------
# Clean up
# ------------------------------------------------
del model_a
del model_b
del model_original

gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("DETERMINISTIC RANDOMIZATION TEST: PASS")
print("=" * 70)

In [1]:
from pathlib import Path
import pandas as pd

RESULT_DIR = Path(
    "/kaggle/working/"
    "T1_4a_parameter_randomization_12x100/"
    "model_level_results"
)

print("Result directory:", RESULT_DIR)
print("Exists:", RESULT_DIR.exists())

if RESULT_DIR.exists():

    files = sorted(RESULT_DIR.glob("*.csv"))

    print("\nCSV files:", len(files))

    for f in files:

        try:
            df = pd.read_csv(f)

            print(
                f"{f.name:30s} "
                f"rows={len(df):4d} "
                f"levels={df['cascade_level'].nunique() if 'cascade_level' in df else 'NA'}"
            )

        except Exception as e:

            print(
                f"{f.name:30s} READ ERROR: {e}"
            )

Result directory: /kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results
Exists: True

CSV files: 2
rho_0.8_rep_1.csv              rows= 700 levels=7
rho_0.8_rep_2.csv              rows= 700 levels=7


In [2]:
# ================================================================
# T1.4a — AUDIT COMPLETED PARTIAL RESULTS
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np

RESULT_DIR = Path(
    "/kaggle/working/"
    "T1_4a_parameter_randomization_12x100/"
    "model_level_results"
)

EXPECTED_MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.2_rep_1",
    "rho_0.2_rep_2",
    "rho_0.4_rep_1",
    "rho_0.4_rep_2",
    "rho_0.6_rep_1",
    "rho_0.6_rep_2",
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

REQUIRED_COLUMNS = [
    "model_name",
    "rho",
    "replicate",
    "image",
    "lesion_id",
    "class_name",
    "malignant",
    "cascade_level",
    "cascade_name",
    "randomized_parameters",
    "similarity_spearman",
]

files = sorted(RESULT_DIR.glob("*.csv"))

print("=" * 72)
print("AUDITING COMPLETED T1.4a PARTIAL RESULTS")
print("=" * 72)

for f in files:

    df = pd.read_csv(f)

    print(f"\n{f.name}")
    print("  rows:", len(df))
    print(
        "  unique images:",
        df["image"].nunique()
    )
    print(
        "  cascade levels:",
        sorted(df["cascade_level"].unique())
    )

    # Exact row count
    assert len(df) == 700

    # Exact 7 levels
    assert sorted(
        df["cascade_level"].unique()
    ) == list(range(7))

    # 100 images at every level
    counts = (
        df.groupby("cascade_level")
        .size()
    )

    assert (counts == 100).all()

    # No duplicate model/image/level
    assert not df.duplicated(
        [
            "model_name",
            "image",
            "cascade_level"
        ]
    ).any()

    # Required columns
    missing = [
        c for c in REQUIRED_COLUMNS
        if c not in df.columns
    ]

    assert not missing, (
        f"Missing columns: {missing}"
    )

    # Similarity must be finite
    sim = df["similarity_spearman"].to_numpy()

    assert np.isfinite(sim).all()

    # Level 0 must be exactly 1
    level0 = df[
        df["cascade_level"] == 0
    ]["similarity_spearman"].to_numpy()

    assert np.allclose(
        level0,
        1.0,
        atol=0,
        rtol=0
    )

    print("  STRUCTURE: PASS")
    print("  DUPLICATES: PASS")
    print("  FINITE SIMILARITIES: PASS")
    print("  LEVEL-0 IDENTITY: PASS")

print("\n" + "=" * 72)
print("PARTIAL RESULT AUDIT: PASS")
print("=" * 72)

print("\nCompleted models:")
for f in files:
    print(" ", f.stem)

missing_models = [
    m for m in EXPECTED_MODELS
    if not (RESULT_DIR / f"{m}.csv").exists()
]

print("\nMissing models:", len(missing_models))
for m in missing_models:
    print(" ", m)

AUDITING COMPLETED T1.4a PARTIAL RESULTS

rho_0.8_rep_1.csv
  rows: 700
  unique images: 100
  cascade levels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
  STRUCTURE: PASS
  DUPLICATES: PASS
  FINITE SIMILARITIES: PASS
  LEVEL-0 IDENTITY: PASS

rho_0.8_rep_2.csv
  rows: 700
  unique images: 100
  cascade levels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
  STRUCTURE: PASS
  DUPLICATES: PASS
  FINITE SIMILARITIES: PASS
  LEVEL-0 IDENTITY: PASS

PARTIAL RESULT AUDIT: PASS

Completed models:
  rho_0.8_rep_1
  rho_0.8_rep_2

Missing models: 10
  rho_0.0_rep_1
  rho_0.0_rep_2
  rho_0.2_rep_1
  rho_0.2_rep_2
  rho_0.4_rep_1
  rho_0.4_rep_2
  rho_0.6_rep_1
  rho_0.6_rep_2
  rho_1.0_rep_1
  rho_1.0_rep_2


In [3]:
# ================================================================
# T1.4a — RECOVERY-SAFE EXECUTION
# 10 REMAINING MODELS ONLY
#
# Two models at a time:
#   GPU 0 -> one model
#   GPU 1 -> one model
#
# Completed models are automatically skipped.
# ================================================================

import gc
import json
import time
import traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import torch
import inspect

# ------------------------------------------------
# Paths
# ------------------------------------------------
PR_DIR = Path(
    "/kaggle/working/"
    "T1_4a_parameter_randomization_12x100"
)

RESULT_DIR = PR_DIR / "model_level_results"
SUMMARY_DIR = PR_DIR / "summaries"

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)
SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------
# Frozen model list
# ------------------------------------------------
ALL_MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.2_rep_1",
    "rho_0.2_rep_2",
    "rho_0.4_rep_1",
    "rho_0.4_rep_2",
    "rho_0.6_rep_1",
    "rho_0.6_rep_2",
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

# ------------------------------------------------
# Validate currently loaded randomization function
# ------------------------------------------------
src = inspect.getsource(randomize_parameters)

assert "if p.ndim >= 2" in src
assert "* 0.01" in src

print("Randomization implementation: VERIFIED")

# ------------------------------------------------
# Determine already-completed models
# ------------------------------------------------
def is_complete_model(model_name):

    path = RESULT_DIR / f"{model_name}.csv"

    if not path.exists():
        return False

    try:
        df = pd.read_csv(path)

        if len(df) != 700:
            return False

        if df["image"].nunique() != 100:
            return False

        if sorted(
            df["cascade_level"].unique().tolist()
        ) != list(range(7)):
            return False

        counts = (
            df.groupby("cascade_level")
            .size()
        )

        if not (counts == 100).all():
            return False

        if df.duplicated(
            [
                "model_name",
                "image",
                "cascade_level"
            ]
        ).any():
            return False

        if not np.isfinite(
            df["similarity_spearman"].to_numpy()
        ).all():
            return False

        level0 = df[
            df["cascade_level"] == 0
        ]["similarity_spearman"].to_numpy()

        if not np.all(
            level0 == 1.0
        ):
            return False

        return True

    except Exception:
        return False


completed = [
    m for m in ALL_MODELS
    if is_complete_model(m)
]

remaining = [
    m for m in ALL_MODELS
    if m not in completed
]

print("\n" + "=" * 72)
print("T1.4a RECOVERY STATUS")
print("=" * 72)

print("Completed:", len(completed))
for m in completed:
    print("  PASS:", m)

print("\nRemaining:", len(remaining))
for m in remaining:
    print("  TODO:", m)

assert len(completed) == 2
assert len(remaining) == 10

# ------------------------------------------------
# Fixed GPU assignment
# For the remaining models, alternate GPUs.
# ------------------------------------------------
gpu_assignment = {}

for i, model_name in enumerate(remaining):
    gpu_assignment[model_name] = i % 2

print("\nGPU assignment for remaining models:")
for m in remaining:
    print(
        f"  {m:18s} -> GPU "
        f"{gpu_assignment[m]}"
    )

# ------------------------------------------------
# Run only TWO models concurrently per batch.
# ------------------------------------------------
overall_start = time.time()

batch_results = []
batch_errors = []

for batch_start in range(
    0,
    len(remaining),
    2
):

    batch = remaining[
        batch_start:batch_start + 2
    ]

    print("\n" + "=" * 72)
    print(
        f"BATCH "
        f"{batch_start // 2 + 1}/5"
    )
    print("=" * 72)

    for m in batch:
        print(
            f"  {m} -> GPU "
            f"{gpu_assignment[m]}"
        )

    batch_start_time = time.time()

    with ThreadPoolExecutor(
        max_workers=len(batch)
    ) as executor:

        futures = {}

        for model_index, model_name in enumerate(batch):

            gpu_id = gpu_assignment[
                model_name
            ]

            future = executor.submit(
                run_model,
                model_name,
                gpu_id,
                batch_start + model_index
            )

            futures[future] = model_name

        for future in as_completed(futures):

            model_name = futures[future]

            try:

                result = future.result()

                batch_results.append(result)

                print(
                    f"\nCOMPLETED: {model_name}"
                )

            except Exception as exc:

                error_record = {
                    "model_name": model_name,
                    "error": repr(exc),
                    "traceback": traceback.format_exc(),
                }

                batch_errors.append(
                    error_record
                )

                print(
                    f"\nFAILED: {model_name}"
                )
                print(
                    traceback.format_exc()
                )

    batch_elapsed = (
        time.time() - batch_start_time
    )

    print(
        f"\nBatch runtime: "
        f"{batch_elapsed / 60:.2f} min"
    )

    # ------------------------------------------------
    # Immediate audit after EVERY batch
    # ------------------------------------------------
    print("\nBatch audit:")

    current_completed = [
        m for m in ALL_MODELS
        if is_complete_model(m)
    ]

    current_remaining = [
        m for m in ALL_MODELS
        if not is_complete_model(m)
    ]

    print(
        f"  Complete: "
        f"{len(current_completed)}/12"
    )

    print(
        f"  Remaining: "
        f"{len(current_remaining)}/12"
    )

    for m in current_completed:
        print("   PASS", m)

    # ------------------------------------------------
    # If a model failed, STOP rather than continuing
    # blindly. We want to inspect the failure.
    # ------------------------------------------------
    if batch_errors:

        print(
            "\nA model failed."
        )

        print(
            "STOPPING before next batch."
        )

        break

# ------------------------------------------------
# Final execution audit
# ------------------------------------------------
overall_elapsed = (
    time.time() - overall_start
)

final_completed = [
    m for m in ALL_MODELS
    if is_complete_model(m)
]

final_missing = [
    m for m in ALL_MODELS
    if not is_complete_model(m)
]

audit = {
    "experiment": "T1.4a_parameter_randomization",
    "design": "12x100_v1",
    "completed_models": final_completed,
    "missing_models": final_missing,
    "completed_count": len(final_completed),
    "missing_count": len(final_missing),
    "batch_errors": batch_errors,
    "runtime_minutes": overall_elapsed / 60,
}

AUDIT_PATH = (
    SUMMARY_DIR /
    "T1_4A_12X100_RECOVERY_EXECUTION_AUDIT.json"
)

AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("RECOVERY EXECUTION SUMMARY")
print("=" * 72)

print(
    f"Complete models: "
    f"{len(final_completed)}/12"
)

print(
    f"Remaining models: "
    f"{len(final_missing)}"
)

print(
    f"Total runtime: "
    f"{overall_elapsed / 60:.2f} min"
)

print("\nAudit:")
print(AUDIT_PATH)

if final_missing:
    print("\nStill missing:")
    for m in final_missing:
        print(" ", m)
else:
    print(
        "\nALL 12 MODELS COMPLETE."
    )

NameError: name 'randomize_parameters' is not defined

In [4]:
# ================================================================
# T1.4a — RESTORE RANDOMIZATION FUNCTIONS AFTER KERNEL RESTART
# ================================================================

import math
import hashlib
import inspect
import torch
import torch.nn as nn

# ------------------------------------------------
# Frozen T1.4a seed protocol
# ------------------------------------------------
MASTER_SEED = 20260925
SEED_NAMESPACE = "param-randomization-v1"


def parameter_seed(
    model_name,
    cascade_level,
    parameter_name
):
    material = (
        f"{MASTER_SEED}|"
        f"{SEED_NAMESPACE}|"
        f"{model_name}|"
        f"{cascade_level}|"
        f"{parameter_name}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        material
    ).hexdigest()

    seed = int(
        digest[:16],
        16
    )

    return seed % (2**63 - 1)


# ------------------------------------------------
# Exact Cell-16 initialization distribution
# + frozen GPU-independent seed protocol
# ------------------------------------------------
def randomize_parameters(
    model,
    model_name,
    cascade_level,
    randomized_names
):
    randomized_names = set(
        randomized_names
    )

    module_map = {
        "fc": model.fc,
        "layer4": model.layer4,
        "layer3": model.layer3,
        "layer2": model.layer2,
        "layer1": model.layer1,
        "conv1": model.conv1,
        "bn1": model.bn1,
    }

    module_order = [
        "fc",
        "layer4",
        "layer3",
        "layer2",
        "layer1",
        "conv1",
        "bn1",
    ]

    for module_name in module_order:

        if module_name not in randomized_names:
            continue

        module = module_map[module_name]

        for parameter_name, p in module.named_parameters():

            full_parameter_name = (
                f"{module_name}.{parameter_name}"
            )

            seed = parameter_seed(
                model_name=model_name,
                cascade_level=cascade_level,
                parameter_name=full_parameter_name,
            )

            rng = torch.Generator(
                device=p.device
            )

            rng.manual_seed(seed)

            with torch.no_grad():

                # Exact Cell-16 behavior
                if p.ndim >= 2:

                    fan_in = nn.init._calculate_correct_fan(
                        p,
                        "fan_in"
                    )

                    gain = nn.init.calculate_gain(
                        "relu"
                    )

                    std = gain / math.sqrt(
                        fan_in
                    )

                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * std
                    )

                else:

                    # Exact Cell-16 behavior for 1-D parameters
                    values = (
                        torch.randn(
                            p.shape,
                            generator=rng,
                            dtype=p.dtype,
                            device=p.device,
                        )
                        * 0.01
                    )

                p.copy_(values)

    return model


# ------------------------------------------------
# Verify function is now actually loaded
# ------------------------------------------------
src = inspect.getsource(
    randomize_parameters
)

assert "if p.ndim >= 2" in src
assert "* 0.01" in src

print("=" * 70)
print("RANDOMIZATION FUNCTIONS RESTORED")
print("=" * 70)
print("parameter_seed: available")
print("randomize_parameters: available")
print("1-D parameter handling: available")
print("GPU-independent SHA256 seeding: available")
print("\nRESTORE: PASS")

RANDOMIZATION FUNCTIONS RESTORED
parameter_seed: available
randomize_parameters: available
1-D parameter handling: available
GPU-independent SHA256 seeding: available

RESTORE: PASS


In [5]:
# ================================================================
# T1.4a — POST-RESTART RECOVERY ENVIRONMENT
#
# Restores everything needed to run ONLY the 10 missing models.
# Existing rho_0.8_rep_1 / rho_0.8_rep_2 results are preserved.
# ================================================================

import gc
import json
import math
import time
import traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.functional as TF

# ================================================================
# 1. FROZEN PATHS
# ================================================================

PR_DIR = Path(
    "/kaggle/working/"
    "T1_4a_parameter_randomization_12x100"
)

DESIGN_PATH = PR_DIR / "T1_4A_12X100_DESIGN.json"
SUBSET_PATH = PR_DIR / "T1_4A_100_IMAGE_SUBSET.csv"

RESULT_DIR = PR_DIR / "model_level_results"
SUMMARY_DIR = PR_DIR / "summaries"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path("/kaggle/temp/isic_cache")
HAIR_PATH = CACHE_DIR / "hair_224.npy"

TRAINING_ROOT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/"
    "training_runs"
)

# ================================================================
# 2. LOAD FROZEN DESIGN
# ================================================================

with open(DESIGN_PATH, "r", encoding="utf-8") as f:
    DESIGN = json.load(f)

subset = pd.read_csv(SUBSET_PATH)

assert len(subset) == 100
assert subset["image"].nunique() == 100
assert subset["lesion_id"].nunique() == 100

assert subset["malignant"].value_counts().to_dict() == {
    0: 50,
    1: 50,
}

MODELS = DESIGN["models"]
CASCADE = DESIGN["cascade"]

assert len(MODELS) == 12
assert len(CASCADE) == 7

print("=" * 72)
print("T1.4a POST-RESTART ENVIRONMENT RESTORE")
print("=" * 72)

print("Models:", len(MODELS))
print("Images:", len(subset))
print("Cascade levels:", len(CASCADE))

# ================================================================
# 3. VERIFY FROZEN SUBSET HASH
# ================================================================

import hashlib

actual_subset_sha = hashlib.sha256(
    SUBSET_PATH.read_bytes()
).hexdigest()

expected_subset_sha = DESIGN[
    "subset_selection"
]["subset_sha256"]

assert actual_subset_sha == expected_subset_sha

print(
    "Subset SHA256:",
    actual_subset_sha
)

# ================================================================
# 4. CACHE
# ================================================================

assert HAIR_PATH.exists()

hair_mm = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

assert hair_mm.shape == (
    23227,
    224,
    224,
    3
)

assert hair_mm.dtype == np.uint8

print(
    "Hair cache:",
    hair_mm.shape,
    hair_mm.dtype
)

# ================================================================
# 5. PRODUCTION CONSTANTS
# ================================================================

_MEAN = (
    0.485,
    0.456,
    0.406,
)

_STD = (
    0.229,
    0.224,
    0.225,
)

IG_STEPS = 50
IG_RETRY_STEPS = 100
IG_SIGMA = 10.0
IG_TOL_REL = 0.05
IG_TAU_ABS = 0.05
IG_CHUNK = 25

# ================================================================
# 6. EXACT PRODUCTION CHECKPOINT LOADER
# ================================================================

def load_model(
    model_name,
    device
):

    ckpt = (
        TRAINING_ROOT /
        model_name /
        "final_epoch15.pt"
    )

    assert ckpt.exists(), (
        f"Missing checkpoint: {ckpt}"
    )

    model = torchvision.models.resnet18(
        weights=None
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        2
    )

    checkpoint = torch.load(
        ckpt,
        map_location="cpu"
    )

    if isinstance(checkpoint, dict):

        if "state_dict" in checkpoint:
            state_dict = checkpoint[
                "state_dict"
            ]

        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint[
                "model_state_dict"
            ]

        else:
            state_dict = checkpoint

    else:
        state_dict = checkpoint

    state_dict = {
        k.replace("module.", "", 1)
        if k.startswith("module.")
        else k: v
        for k, v in state_dict.items()
    }

    model.load_state_dict(
        state_dict,
        strict=True
    )

    model.eval()
    model.to(device)

    return model

# ================================================================
# 7. EXACT PRODUCTION-FAITHFUL IG MAP
# ================================================================

def production_ig_map(
    model,
    hair_u8,
    device
):

    mean = torch.tensor(
        _MEAN,
        device=device
    ).view(
        1, 3, 1, 1
    )

    std = torch.tensor(
        _STD,
        device=device
    ).view(
        1, 3, 1, 1
    )

    model.eval()

    x = (
        torch.from_numpy(
            np.ascontiguousarray(
                hair_u8
            )
        )
        .to(device)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = (
        2 *
        int(
            math.ceil(
                3 * IG_SIGMA
            )
        )
        + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[
            IG_SIGMA,
            IG_SIGMA
        ]
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    def zf(t):

        o = model(t)

        return (
            o[:, 1]
            - o[:, 0]
        )

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            ) / n
        )

        for s in range(
            0,
            n,
            IG_CHUNK
        ):

            a = (
                alphas[
                    s:s + IG_CHUNK
                ]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn +
                a * (
                    xn - bn
                )
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            xn - bn
        ) * acc / n

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    -
                    zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            )
            -
            delta
        )

        return (
            ig,
            delta,
            err
        )

    ig, delta, err = run(
        IG_STEPS
    )

    tol = max(
        IG_TOL_REL * abs(delta),
        IG_TAU_ABS
    )

    used = IG_STEPS

    if err > tol:

        ig, delta, err = run(
            IG_RETRY_STEPS
        )

        used = IG_RETRY_STEPS

    amap = (
        ig.abs()
        .sum(1)[0]
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    return amap, {
        "delta": delta,
        "completeness_error": err,
        "tolerance": tol,
        "steps_used": used,
        "completeness_failed": bool(
            err > tol
        ),
    }

# ================================================================
# 8. PIXELWISE SPEARMAN
# ================================================================

def pixelwise_spearman(
    a,
    b
):

    a = np.asarray(
        a
    ).reshape(-1)

    b = np.asarray(
        b
    ).reshape(-1)

    ra = np.empty_like(
        a,
        dtype=np.float64
    )

    rb = np.empty_like(
        b,
        dtype=np.float64
    )

    ia = np.argsort(
        a,
        kind="mergesort"
    )

    ib = np.argsort(
        b,
        kind="mergesort"
    )

    ra[ia] = np.arange(
        len(a),
        dtype=np.float64
    )

    rb[ib] = np.arange(
        len(b),
        dtype=np.float64
    )

    ra -= ra.mean()
    rb -= rb.mean()

    denom = np.sqrt(
        np.sum(ra * ra)
        *
        np.sum(rb * rb)
    )

    if denom == 0:
        return np.nan

    return float(
        np.sum(ra * rb)
        /
        denom
    )

# ================================================================
# 9. FROZEN CASCADE VALIDATION
# ================================================================

EXPECTED_CASCADE = [
    "original",
    "fc",
    "fc_layer4",
    "fc_layer4_layer3",
    "fc_layer4_layer3_layer2",
    "fc_layer4_layer3_layer2_layer1",
    "all",
]

assert [
    x["name"]
    for x in CASCADE
] == EXPECTED_CASCADE

print(
    "Cascade definition: PASS"
)

# ================================================================
# 10. RESULT VALIDATION
# ================================================================

def is_complete_model(
    model_name
):

    path = (
        RESULT_DIR /
        f"{model_name}.csv"
    )

    if not path.exists():
        return False

    try:

        df = pd.read_csv(
            path
        )

        if len(df) != 700:
            return False

        if df["image"].nunique() != 100:
            return False

        if sorted(
            df["cascade_level"]
            .unique()
            .tolist()
        ) != list(range(7)):
            return False

        counts = (
            df.groupby(
                "cascade_level"
            )
            .size()
        )

        if not (
            counts == 100
        ).all():
            return False

        if df.duplicated(
            [
                "model_name",
                "image",
                "cascade_level"
            ]
        ).any():
            return False

        if not np.isfinite(
            df[
                "similarity_spearman"
            ].to_numpy()
        ).all():
            return False

        level0 = df[
            df["cascade_level"] == 0
        ][
            "similarity_spearman"
        ].to_numpy()

        if not np.all(
            level0 == 1.0
        ):
            return False

        return True

    except Exception:
        return False

# ================================================================
# 11. MODEL WORKER
# ================================================================

def run_model(
    model_name,
    gpu_id,
    model_index
):

    device = torch.device(
        f"cuda:{gpu_id}"
    )

    torch.cuda.set_device(
        device
    )

    start_time = time.time()

    print(
        f"\n[{model_name}] START "
        f"GPU={gpu_id}"
    )

    # ------------------------------------------------------------
    # Fresh pristine production checkpoint
    # ------------------------------------------------------------

    model = load_model(
        model_name,
        device
    )

    # ------------------------------------------------------------
    # Original reference maps
    # ------------------------------------------------------------

    reference_maps = {}
    reference_meta = {}

    for i, row in subset.iterrows():

        image_id = row["image"]
        cache_row = int(
            row["row"]
        )

        hair = np.array(
            hair_mm[cache_row],
            dtype=np.uint8,
            copy=True
        )

        amap, meta = production_ig_map(
            model,
            hair,
            device
        )

        reference_maps[
            image_id
        ] = amap

        reference_meta[
            image_id
        ] = meta

    print(
        f"[{model_name}] "
        f"reference maps complete"
    )

    model_results = []

    # ------------------------------------------------------------
    # Cascade
    # ------------------------------------------------------------

    for cascade in CASCADE:

        level = int(
            cascade["level"]
        )

        level_name = cascade["name"]
        randomized = cascade["randomized"]

        level_start = time.time()

        # --------------------------------------------------------
        # Level 0
        # --------------------------------------------------------

        if level == 0:

            for _, row in subset.iterrows():

                image_id = row["image"]

                ref_meta = (
                    reference_meta[
                        image_id
                    ]
                )

                model_results.append({
                    "model_name": model_name,
                    "rho": float(
                        model_name.split("_")[1]
                    ),
                    "replicate": int(
                        model_name.split("_")[-1]
                    ),
                    "image": image_id,
                    "lesion_id": row["lesion_id"],
                    "class_name": row["class_name"],
                    "malignant": int(
                        row["malignant"]
                    ),
                    "cascade_level": level,
                    "cascade_name": level_name,
                    "randomized_parameters": "",
                    "similarity_spearman": 1.0,
                    "reference_steps_used":
                        ref_meta[
                            "steps_used"
                        ],
                    "reference_completeness_error":
                        ref_meta[
                            "completeness_error"
                        ],
                    "randomized_steps_used": 0,
                    "randomized_completeness_error": 0.0,
                    "randomized_completeness_failed": False,
                })

        # --------------------------------------------------------
        # Randomized level
        # --------------------------------------------------------

        else:

            randomized_model = (
                load_model(
                    model_name,
                    device
                )
            )

            randomize_parameters(
                randomized_model,
                model_name,
                level,
                randomized
            )

            randomized_model.eval()

            for _, row in subset.iterrows():

                image_id = row["image"]
                cache_row = int(
                    row["row"]
                )

                hair = np.array(
                    hair_mm[cache_row],
                    dtype=np.uint8,
                    copy=True
                )

                amap, meta = (
                    production_ig_map(
                        randomized_model,
                        hair,
                        device
                    )
                )

                ref_map = (
                    reference_maps[
                        image_id
                    ]
                )

                similarity = (
                    pixelwise_spearman(
                        amap,
                        ref_map
                    )
                )

                model_results.append({
                    "model_name": model_name,
                    "rho": float(
                        model_name.split("_")[1]
                    ),
                    "replicate": int(
                        model_name.split("_")[-1]
                    ),
                    "image": image_id,
                    "lesion_id": row["lesion_id"],
                    "class_name": row["class_name"],
                    "malignant": int(
                        row["malignant"]
                    ),
                    "cascade_level": level,
                    "cascade_name": level_name,
                    "randomized_parameters": "+".join(
                        randomized
                    ),
                    "similarity_spearman": similarity,
                    "reference_steps_used":
                        reference_meta[
                            image_id
                        ]["steps_used"],
                    "reference_completeness_error":
                        reference_meta[
                            image_id
                        ]["completeness_error"],
                    "randomized_steps_used":
                        meta["steps_used"],
                    "randomized_completeness_error":
                        meta[
                            "completeness_error"
                        ],
                    "randomized_completeness_failed":
                        meta[
                            "completeness_failed"
                        ],
                })

            del randomized_model
            gc.collect()
            torch.cuda.empty_cache()

        elapsed = (
            time.time()
            -
            level_start
        )

        print(
            f"[{model_name}] "
            f"level {level} "
            f"({level_name}) "
            f"complete: "
            f"{elapsed / 60:.2f} min"
        )

    # ------------------------------------------------------------
    # Save only after COMPLETE model
    # ------------------------------------------------------------

    result_df = pd.DataFrame(
        model_results
    )

    assert len(
        result_df
    ) == 700

    result_path = (
        RESULT_DIR /
        f"{model_name}.csv"
    )

    # Do not overwrite an already-valid model.
    if result_path.exists():

        if is_complete_model(
            model_name
        ):

            print(
                f"[{model_name}] "
                f"already complete — "
                f"NOT overwritten."
            )

            return {
                "model_name": model_name,
                "gpu_id": gpu_id,
                "rows": 700,
                "skipped_existing": True,
            }

    result_df.to_csv(
        result_path,
        index=False
    )

    elapsed_total = (
        time.time()
        -
        start_time
    )

    print(
        f"[{model_name}] DONE "
        f"{elapsed_total / 60:.2f} min "
        f"rows={len(result_df)}"
    )

    del model
    del reference_maps
    del reference_meta

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "model_name": model_name,
        "gpu_id": gpu_id,
        "rows": len(result_df),
        "runtime_seconds": elapsed_total,
        "result_path": str(
            result_path
        ),
    }

# ================================================================
# 12. RECOVERY STATUS
# ================================================================

completed = [
    m for m in MODELS
    if is_complete_model(m)
]

remaining = [
    m for m in MODELS
    if not is_complete_model(m)
]

print("\n" + "=" * 72)
print("RECOVERY STATUS")
print("=" * 72)

print(
    f"Completed: "
    f"{len(completed)}/12"
)

for m in completed:
    print(
        "  PASS:",
        m
    )

print(
    f"\nRemaining: "
    f"{len(remaining)}"
)

for m in remaining:
    print(
        "  TODO:",
        m
    )

assert len(completed) == 2
assert len(remaining) == 10

# ================================================================
# 13. GPU CHECK
# ================================================================

assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2

print("\nCUDA:")
for i in range(
    torch.cuda.device_count()
):
    print(
        i,
        torch.cuda.get_device_name(i)
    )

print("\n" + "=" * 72)
print("RECOVERY ENVIRONMENT READY")
print("=" * 72)

print(
    "No models have been launched yet."
)
print(
    "Completed files will be preserved."
)
print(
    "Remaining models:",
    len(remaining)
)

T1.4a POST-RESTART ENVIRONMENT RESTORE
Models: 12
Images: 100
Cascade levels: 7
Subset SHA256: 16b457ddf44ba552d88612c13f1637b04fa79e5df8fa775ca01215f631e9cb5b
Hair cache: (23227, 224, 224, 3) uint8
Cascade definition: PASS

RECOVERY STATUS
Completed: 2/12
  PASS: rho_0.8_rep_1
  PASS: rho_0.8_rep_2

Remaining: 10
  TODO: rho_0.0_rep_1
  TODO: rho_0.0_rep_2
  TODO: rho_0.2_rep_1
  TODO: rho_0.2_rep_2
  TODO: rho_0.4_rep_1
  TODO: rho_0.4_rep_2
  TODO: rho_0.6_rep_1
  TODO: rho_0.6_rep_2
  TODO: rho_1.0_rep_1
  TODO: rho_1.0_rep_2

CUDA:
0 Tesla T4
1 Tesla T4

RECOVERY ENVIRONMENT READY
No models have been launched yet.
Completed files will be preserved.
Remaining models: 10


In [6]:
# ================================================================
# T1.4a RECOVERY EXECUTION — 10 REMAINING MODELS
# Two models at a time: one model per T4
# NEVER rerun already-completed models
# ================================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import time
import pandas as pd

remaining_models = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.2_rep_1",
    "rho_0.2_rep_2",
    "rho_0.4_rep_1",
    "rho_0.4_rep_2",
    "rho_0.6_rep_1",
    "rho_0.6_rep_2",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

# ------------------------------------------------
# Safety: verify completed files before proceeding
# ------------------------------------------------

completed_models = [
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
]

for model_name in completed_models:
    path = RESULTS_DIR / f"{model_name}.csv"
    assert path.exists(), f"Missing completed result: {path}"

    df = pd.read_csv(path)

    assert len(df) == 700, f"{model_name}: expected 700 rows, got {len(df)}"
    assert df["image"].nunique() == 100
    assert sorted(df["level"].unique().tolist()) == list(range(7))

    counts = df.groupby("level").size().to_dict()
    assert all(counts.get(i, 0) == 100 for i in range(7))

    assert df.duplicated(
        subset=["image", "level"]
    ).sum() == 0

    assert df["spearman_similarity"].notna().all()
    assert (df["level"] == 0).sum() == 100
    assert (df.loc[df["level"] == 0, "spearman_similarity"] == 1.0).all()

print("Completed-file safety audit: PASS")

# ------------------------------------------------
# Never include completed models
# ------------------------------------------------

todo = []

for model_name in remaining_models:
    path = RESULTS_DIR / f"{model_name}.csv"

    if path.exists():
        df = pd.read_csv(path)

        if (
            len(df) == 700
            and df["image"].nunique() == 100
            and sorted(df["level"].unique().tolist()) == list(range(7))
            and df.duplicated(subset=["image", "level"]).sum() == 0
            and df["spearman_similarity"].notna().all()
        ):
            print(f"SKIP — already complete: {model_name}")
            continue

    todo.append(model_name)

print()
print("=" * 72)
print(f"MODELS TO RUN: {len(todo)}")
for m in todo:
    print("  TODO:", m)
print("=" * 72)

assert len(todo) == 10, (
    f"Expected exactly 10 remaining models, found {len(todo)}. "
    "Stop and inspect before launching."
)

# ------------------------------------------------
# Run in batches of 2
# ------------------------------------------------

for batch_start in range(0, len(todo), 2):

    batch = todo[batch_start:batch_start + 2]

    print()
    print("=" * 72)
    print(
        f"BATCH {batch_start // 2 + 1}/5 — "
        f"{batch[0]}" +
        (f" + {batch[1]}" if len(batch) > 1 else "")
    )
    print("=" * 72)

    t0 = time.time()

    # Explicitly assign one GPU per model.
    jobs = []

    for gpu_id, model_name in enumerate(batch):
        jobs.append((model_name, gpu_id))

    results = {}

    with ThreadPoolExecutor(max_workers=len(jobs)) as executor:

        future_map = {
            executor.submit(
                run_model,
                model_name,
                gpu_id,
            ): (model_name, gpu_id)
            for model_name, gpu_id in jobs
        }

        for future in as_completed(future_map):

            model_name, gpu_id = future_map[future]

            try:
                result = future.result()
                results[model_name] = result

                print(
                    f"\nFINISHED: {model_name} "
                    f"(GPU {gpu_id})"
                )

                if result is not None:
                    print(result)

            except Exception as e:
                print(
                    f"\nFAILED: {model_name} "
                    f"(GPU {gpu_id})"
                )
                print(type(e).__name__, str(e))
                raise

    elapsed = time.time() - t0

    print()
    print(
        f"BATCH COMPLETE: {len(batch)} models "
        f"in {elapsed / 60:.2f} minutes"
    )

    # ------------------------------------------------
    # Immediate audit after each batch
    # ------------------------------------------------

    print("\nPOST-BATCH AUDIT")

    for model_name in batch:

        path = RESULTS_DIR / f"{model_name}.csv"

        assert path.exists(), (
            f"{model_name}: result file missing"
        )

        df = pd.read_csv(path)

        assert len(df) == 700
        assert df["image"].nunique() == 100
        assert sorted(df["level"].unique().tolist()) == list(range(7))

        counts = df.groupby("level").size().to_dict()

        assert all(
            counts.get(i, 0) == 100
            for i in range(7)
        )

        assert df.duplicated(
            subset=["image", "level"]
        ).sum() == 0

        assert df["spearman_similarity"].notna().all()

        level0 = df.loc[
            df["level"] == 0,
            "spearman_similarity"
        ]

        assert len(level0) == 100
        assert (level0 == 1.0).all()

        print(
            f"  PASS {model_name}: "
            f"700 rows | 100 images | 7 levels"
        )

    print("\nBATCH AUDIT: PASS")

print()
print("=" * 72)
print("T1.4a RECOVERY COMPLETE")
print("=" * 72)

NameError: name 'RESULTS_DIR' is not defined

In [7]:
# ================================================================
# RESTORE T1.4a RESULTS DIRECTORY
# ================================================================

from pathlib import Path
import pandas as pd

RESULTS_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR =", RESULTS_DIR)
print("Exists:", RESULTS_DIR.exists())

print("\nCurrent CSV files:")
for p in sorted(RESULTS_DIR.glob("*.csv")):
    print(" ", p.name)

# Verify the two known completed models
for model_name in [
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
]:
    path = RESULTS_DIR / f"{model_name}.csv"

    assert path.exists(), f"Missing expected completed file: {path}"

    df = pd.read_csv(path)

    assert len(df) == 700
    assert df["image"].nunique() == 100
    assert sorted(df["level"].unique().tolist()) == list(range(7))
    assert df.duplicated(
        subset=["image", "level"]
    ).sum() == 0
    assert df["spearman_similarity"].notna().all()

    level_counts = df.groupby("level").size()

    assert all(
        level_counts.get(i, 0) == 100
        for i in range(7)
    )

    level0 = df.loc[
        df["level"] == 0,
        "spearman_similarity"
    ]

    assert len(level0) == 100
    assert (level0 == 1.0).all()

    print(
        f"PASS: {model_name} | "
        f"{len(df)} rows | "
        f"{df['image'].nunique()} images | "
        f"7 levels"
    )

print("\nRESULTS DIRECTORY RESTORE: PASS")

RESULTS_DIR = /kaggle/working/T1_4a_parameter_randomization_12x100
Exists: True

Current CSV files:
  T1_4A_100_IMAGE_SUBSET.csv


AssertionError: Missing expected completed file: /kaggle/working/T1_4a_parameter_randomization_12x100/rho_0.8_rep_1.csv

In [8]:
from pathlib import Path

print("=" * 72)
print("SEARCHING /kaggle/working FOR T1.4a RESULT FILES")
print("=" * 72)

matches = []

for p in Path("/kaggle/working").rglob("rho_*.csv"):
    matches.append(p)

if matches:
    for p in sorted(matches):
        print(p)
else:
    print("NO rho_*.csv FILES FOUND")

print()
print("All T1.4-related CSVs:")
for p in sorted(Path("/kaggle/working").rglob("*.csv")):
    if "T1_4" in str(p) or "rho_" in p.name:
        print(p)

SEARCHING /kaggle/working FOR T1.4a RESULT FILES
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_0.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_1.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_2.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_3.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_4.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_5.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_6.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_2__level_0.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_2__

In [9]:
from pathlib import Path

for root in [
    Path("/kaggle/working/T1_4a_parameter_randomization_12x100"),
    Path("/kaggle/working/T1_4_parameter_randomization"),
]:
    print("\n" + "=" * 72)
    print(root)
    print("=" * 72)

    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                print(p)
    else:
        print("DIRECTORY DOES NOT EXIST")


/kaggle/working/T1_4a_parameter_randomization_12x100
/kaggle/working/T1_4a_parameter_randomization_12x100/T1_4A_100_IMAGE_SUBSET.csv
/kaggle/working/T1_4a_parameter_randomization_12x100/T1_4A_12X100_DESIGN.json
/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results/rho_0.8_rep_1.csv
/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results/rho_0.8_rep_2.csv

/kaggle/working/T1_4_parameter_randomization
/kaggle/working/T1_4_parameter_randomization/T1_4_PARAMETER_RANDOMIZATION_DESIGN.json
/kaggle/working/T1_4_parameter_randomization/T1_4_PARAMETER_RANDOMIZATION_DESIGN_SHA256.txt
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_0.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_1.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x500_PARTIAL_NOT_USED/rho_0.0_rep_1__level_2.csv
/kaggle/working/T1_4_parameter_randomization/T1_4a_FULL_30x50

In [10]:
RESULTS_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

In [11]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

print("RESULTS_DIR =", RESULTS_DIR)

for model_name in [
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
]:
    path = RESULTS_DIR / f"{model_name}.csv"

    print(f"\nChecking {model_name}")
    print("Path:", path)
    print("Exists:", path.exists())

    assert path.exists()

    df = pd.read_csv(path)

    assert len(df) == 700
    assert df["image"].nunique() == 100
    assert sorted(df["level"].unique()) == list(range(7))
    assert df.duplicated(
        subset=["image", "level"]
    ).sum() == 0
    assert df["spearman_similarity"].notna().all()

    counts = df.groupby("level").size()

    assert all(counts.get(i, 0) == 100 for i in range(7))

    level0 = df.loc[
        df["level"] == 0,
        "spearman_similarity"
    ]

    assert len(level0) == 100
    assert (level0 == 1.0).all()

    print(
        f"PASS — {model_name}: "
        f"{len(df)} rows, "
        f"{df['image'].nunique()} images, "
        f"7 levels"
    )

print("\nT1.4a EXISTING RESULTS: PASS")

RESULTS_DIR = /kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results

Checking rho_0.8_rep_1
Path: /kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results/rho_0.8_rep_1.csv
Exists: True


KeyError: 'level'

In [12]:
# ================================================================
# INSPECT EXISTING T1.4a RESULT SCHEMA
# READ-ONLY — DOES NOT MODIFY ANYTHING
# ================================================================

from pathlib import Path
import pandas as pd

RESULTS_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

for model_name in [
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
]:
    path = RESULTS_DIR / f"{model_name}.csv"

    print("\n" + "=" * 72)
    print(model_name)
    print("=" * 72)

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("\nColumns:")
    for i, col in enumerate(df.columns):
        print(f"  {i}: {col!r}")

    print("\nFirst 5 rows:")
    display(df.head())

    print("\nDtypes:")
    print(df.dtypes)


rho_0.8_rep_1
Shape: (700, 16)

Columns:
  0: 'model_name'
  1: 'rho'
  2: 'replicate'
  3: 'image'
  4: 'lesion_id'
  5: 'class_name'
  6: 'malignant'
  7: 'cascade_level'
  8: 'cascade_name'
  9: 'randomized_parameters'
  10: 'similarity_spearman'
  11: 'reference_steps_used'
  12: 'reference_completeness_error'
  13: 'randomized_steps_used'
  14: 'randomized_completeness_error'
  15: 'randomized_completeness_failed'

First 5 rows:


,model_name,rho,replicate,image,lesion_id,class_name,malignant,cascade_level,cascade_name,randomized_parameters,similarity_spearman,reference_steps_used,reference_completeness_error,randomized_steps_used,randomized_completeness_error,randomized_completeness_failed
0,rho_0.8_rep_1,0.8,1,ISIC_0028453,HAM_0002856,NV,0,0,original,NaN,1.0,50,0.120297,0,0.0,False
1,rho_0.8_rep_1,0.8,1,ISIC_0027947,HAM_0004416,NV,0,0,original,NaN,1.0,100,0.230187,0,0.0,False
2,rho_0.8_rep_1,0.8,1,ISIC_0028437,HAM_0002990,NV,0,0,original,NaN,1.0,100,0.381017,0,0.0,False
3,rho_0.8_rep_1,0.8,1,ISIC_0064540,BCN_0002135,NV,0,0,original,NaN,1.0,100,0.254351,0,0.0,False
4,rho_0.8_rep_1,0.8,1,ISIC_0026404,HAM_0003727,NV,0,0,original,NaN,1.0,100,0.141110,0,0.0,False



Dtypes:
model_name                         object
rho                               float64
replicate                           int64
image                              object
lesion_id                          object
class_name                         object
malignant                           int64
cascade_level                       int64
cascade_name                       object
randomized_parameters              object
similarity_spearman               float64
reference_steps_used                int64
reference_completeness_error      float64
randomized_steps_used               int64
randomized_completeness_error     float64
randomized_completeness_failed       bool
dtype: object

rho_0.8_rep_2
Shape: (700, 16)

Columns:
  0: 'model_name'
  1: 'rho'
  2: 'replicate'
  3: 'image'
  4: 'lesion_id'
  5: 'class_name'
  6: 'malignant'
  7: 'cascade_level'
  8: 'cascade_name'
  9: 'randomized_parameters'
  10: 'similarity_spearman'
  11: 'reference_steps_used'
  12: 'reference_complete

,model_name,rho,replicate,image,lesion_id,class_name,malignant,cascade_level,cascade_name,randomized_parameters,similarity_spearman,reference_steps_used,reference_completeness_error,randomized_steps_used,randomized_completeness_error,randomized_completeness_failed
0,rho_0.8_rep_2,0.8,2,ISIC_0028453,HAM_0002856,NV,0,0,original,NaN,1.0,100,0.297869,0,0.0,False
1,rho_0.8_rep_2,0.8,2,ISIC_0027947,HAM_0004416,NV,0,0,original,NaN,1.0,100,0.351763,0,0.0,False
2,rho_0.8_rep_2,0.8,2,ISIC_0028437,HAM_0002990,NV,0,0,original,NaN,1.0,50,0.026450,0,0.0,False
3,rho_0.8_rep_2,0.8,2,ISIC_0064540,BCN_0002135,NV,0,0,original,NaN,1.0,100,0.009397,0,0.0,False
4,rho_0.8_rep_2,0.8,2,ISIC_0026404,HAM_0003727,NV,0,0,original,NaN,1.0,50,0.071426,0,0.0,False



Dtypes:
model_name                         object
rho                               float64
replicate                           int64
image                              object
lesion_id                          object
class_name                         object
malignant                           int64
cascade_level                       int64
cascade_name                       object
randomized_parameters              object
similarity_spearman               float64
reference_steps_used                int64
reference_completeness_error      float64
randomized_steps_used               int64
randomized_completeness_error     float64
randomized_completeness_failed       bool
dtype: object


In [13]:
# ================================================================
# T1.4a — VALIDATE EXISTING COMPLETED MODELS
# READ-ONLY
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

completed_models = [
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
]

required_columns = {
    "model_name",
    "rho",
    "replicate",
    "image",
    "lesion_id",
    "class_name",
    "malignant",
    "cascade_level",
    "cascade_name",
    "randomized_parameters",
    "similarity_spearman",
    "reference_steps_used",
    "reference_completeness_error",
    "randomized_steps_used",
    "randomized_completeness_error",
    "randomized_completeness_failed",
}

for model_name in completed_models:

    path = RESULTS_DIR / f"{model_name}.csv"

    assert path.exists(), f"Missing: {path}"

    df = pd.read_csv(path)

    # -----------------------------
    # Schema
    # -----------------------------
    assert required_columns.issubset(df.columns)

    # -----------------------------
    # Dimensions
    # -----------------------------
    assert len(df) == 700
    assert df["image"].nunique() == 100

    # -----------------------------
    # Cascade
    # -----------------------------
    assert sorted(
        df["cascade_level"].unique().tolist()
    ) == list(range(7))

    counts = df.groupby("cascade_level").size()

    assert all(
        counts.get(i, 0) == 100
        for i in range(7)
    )

    # -----------------------------
    # No duplicate image × level
    # -----------------------------
    assert df.duplicated(
        subset=["image", "cascade_level"]
    ).sum() == 0

    # -----------------------------
    # Similarity finite
    # -----------------------------
    assert np.isfinite(
        df["similarity_spearman"]
    ).all()

    # -----------------------------
    # Level 0 must be exactly 1
    # -----------------------------
    level0 = df.loc[
        df["cascade_level"] == 0,
        "similarity_spearman"
    ]

    assert len(level0) == 100
    assert (level0 == 1.0).all()

    # -----------------------------
    # Model identity
    # -----------------------------
    assert df["model_name"].nunique() == 1
    assert df["model_name"].iloc[0] == model_name

    # -----------------------------
    # Replicate identity
    # -----------------------------
    expected_rep = int(model_name.split("_rep_")[1])

    assert df["replicate"].nunique() == 1
    assert int(df["replicate"].iloc[0]) == expected_rep

    # -----------------------------
    # rho identity
    # -----------------------------
    expected_rho = float(
        model_name.split("_rep_")[0].replace("rho_", "")
    )

    assert df["rho"].nunique() == 1
    assert float(df["rho"].iloc[0]) == expected_rho

    # -----------------------------
    # Completeness flags
    # -----------------------------
    assert df["randomized_completeness_failed"].dtype == bool

    print(
        f"PASS | {model_name} | "
        f"700 rows | 100 images | "
        f"7 levels | "
        f"all similarities finite"
    )

print()
print("=" * 72)
print("EXISTING T1.4a RESULTS: VALIDATED")
print("=" * 72)

PASS | rho_0.8_rep_1 | 700 rows | 100 images | 7 levels | all similarities finite
PASS | rho_0.8_rep_2 | 700 rows | 100 images | 7 levels | all similarities finite

EXISTING T1.4a RESULTS: VALIDATED


In [14]:
# ================================================================
# T1.4a — PRE-LAUNCH SAFETY CHECK
# ================================================================

assert "run_model" in globals(), (
    "run_model is missing. Restore the T1.4a environment first."
)

assert "randomize_parameters" in globals(), (
    "randomize_parameters is missing."
)

print("run_model: PASS")
print("randomize_parameters: PASS")
print("RESULTS_DIR:", RESULTS_DIR)

print("\nExisting completed models:")
for m in completed_models:
    print("  PASS:", m)

print("\nReady for recovery.")

run_model: PASS
randomize_parameters: PASS
RESULTS_DIR: /kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results

Existing completed models:
  PASS: rho_0.8_rep_1
  PASS: rho_0.8_rep_2

Ready for recovery.


In [15]:
# ================================================================
# T1.4a RECOVERY EXECUTION
# 10 REMAINING MODELS — 2 GPUs, 2 MODELS/BATCH
# ================================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc
import torch

# ------------------------------------------------
# Paths / model list
# ------------------------------------------------

RESULTS_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

ALL_MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.2_rep_1",
    "rho_0.2_rep_2",
    "rho_0.4_rep_1",
    "rho_0.4_rep_2",
    "rho_0.6_rep_1",
    "rho_0.6_rep_2",
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

# ------------------------------------------------
# Safety checks
# ------------------------------------------------

assert "run_model" in globals()
assert "randomize_parameters" in globals()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("T1.4a RECOVERY")
print("=" * 72)

# ------------------------------------------------
# Determine which models are genuinely complete
# ------------------------------------------------

def validate_result_file(model_name):

    path = RESULTS_DIR / f"{model_name}.csv"

    if not path.exists():
        return False

    try:
        df = pd.read_csv(path)

        required = {
            "model_name",
            "rho",
            "replicate",
            "image",
            "lesion_id",
            "class_name",
            "malignant",
            "cascade_level",
            "cascade_name",
            "randomized_parameters",
            "similarity_spearman",
            "reference_steps_used",
            "reference_completeness_error",
            "randomized_steps_used",
            "randomized_completeness_error",
            "randomized_completeness_failed",
        }

        if not required.issubset(df.columns):
            return False

        if len(df) != 700:
            return False

        if df["image"].nunique() != 100:
            return False

        if sorted(
            df["cascade_level"].unique().tolist()
        ) != list(range(7)):
            return False

        counts = df.groupby("cascade_level").size()

        if not all(
            counts.get(i, 0) == 100
            for i in range(7)
        ):
            return False

        if df.duplicated(
            subset=["image", "cascade_level"]
        ).sum() != 0:
            return False

        if not np.isfinite(
            df["similarity_spearman"]
        ).all():
            return False

        level0 = df.loc[
            df["cascade_level"] == 0,
            "similarity_spearman"
        ]

        if len(level0) != 100:
            return False

        if not (level0 == 1.0).all():
            return False

        if df["model_name"].nunique() != 1:
            return False

        if df["model_name"].iloc[0] != model_name:
            return False

        return True

    except Exception:
        return False


completed = [
    m for m in ALL_MODELS
    if validate_result_file(m)
]

remaining = [
    m for m in ALL_MODELS
    if m not in completed
]

print("\nVALID COMPLETED RESULTS:")
for m in completed:
    print("  PASS:", m)

print("\nREMAINING:")
for m in remaining:
    print("  TODO:", m)

print(
    f"\nProgress: {len(completed)}/12 complete"
)

# We expect exactly these two at this point.
assert set(completed) == {
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
}, (
    "Unexpected result state. "
    "Stop rather than launching models."
)

assert len(remaining) == 10

print("\nPRE-LAUNCH SAFETY CHECK: PASS")

# ================================================================
# Run exactly two models per batch
# ================================================================

for batch_index in range(0, len(remaining), 2):

    batch = remaining[
        batch_index:batch_index + 2
    ]

    print()
    print("=" * 72)
    print(
        f"BATCH {batch_index // 2 + 1}/5"
    )
    print("=" * 72)

    for gpu_id, model_name in enumerate(batch):
        print(
            f"GPU {gpu_id} -> {model_name}"
        )

    t0 = time.time()

    futures = {}

    with ThreadPoolExecutor(
        max_workers=len(batch)
    ) as executor:

        for gpu_id, model_name in enumerate(batch):

            future = executor.submit(
                run_model,
                model_name,
                gpu_id
            )

            futures[future] = (
                model_name,
                gpu_id
            )

        # Wait for every model in this batch.
        for future in as_completed(futures):

            model_name, gpu_id = futures[future]

            try:
                result = future.result()

                print()
                print(
                    f"FINISHED: {model_name} "
                    f"on GPU {gpu_id}"
                )

                if result is not None:
                    print(result)

            except Exception as exc:

                print()
                print("=" * 72)
                print("MODEL FAILED")
                print("=" * 72)
                print("Model:", model_name)
                print("GPU:", gpu_id)
                print(
                    type(exc).__name__,
                    str(exc)
                )

                # Do not continue to later batches.
                raise

    elapsed = time.time() - t0

    print()
    print(
        f"BATCH {batch_index // 2 + 1} COMPLETE "
        f"in {elapsed / 60:.2f} minutes"
    )

    # ============================================================
    # Immediate post-batch audit
    # ============================================================

    print("\nPOST-BATCH AUDIT")

    for model_name in batch:

        assert validate_result_file(
            model_name
        ), (
            f"POST-BATCH AUDIT FAILED: "
            f"{model_name}"
        )

        df = pd.read_csv(
            RESULTS_DIR / f"{model_name}.csv"
        )

        print(
            f"  PASS: {model_name} | "
            f"700 rows | "
            f"100 images | "
            f"7 cascade levels"
        )

    # Release Python references / cached CUDA allocations
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("\nBATCH AUDIT: PASS")

# ================================================================
# FINAL RECOVERY AUDIT
# ================================================================

print()
print("=" * 72)
print("FINAL T1.4a MODEL COMPLETENESS AUDIT")
print("=" * 72)

for model_name in ALL_MODELS:

    assert validate_result_file(
        model_name
    ), f"Final audit failed: {model_name}"

    print(
        f"PASS | {model_name}"
    )

print()
print("=" * 72)
print("T1.4a RECOVERY COMPLETE")
print("=" * 72)
print("Models: 12/12")
print("Images/model: 100")
print("Cascade levels/model: 7")
print("Total observations: 8,400")
print("=" * 72)

T1.4a RECOVERY

VALID COMPLETED RESULTS:
  PASS: rho_0.8_rep_1
  PASS: rho_0.8_rep_2

REMAINING:
  TODO: rho_0.0_rep_1
  TODO: rho_0.0_rep_2
  TODO: rho_0.2_rep_1
  TODO: rho_0.2_rep_2
  TODO: rho_0.4_rep_1
  TODO: rho_0.4_rep_2
  TODO: rho_0.6_rep_1
  TODO: rho_0.6_rep_2
  TODO: rho_1.0_rep_1
  TODO: rho_1.0_rep_2

Progress: 2/12 complete

PRE-LAUNCH SAFETY CHECK: PASS

BATCH 1/5
GPU 0 -> rho_0.0_rep_1
GPU 1 -> rho_0.0_rep_2

MODEL FAILED
Model: rho_0.0_rep_1
GPU: 0
TypeError run_model() missing 1 required positional argument: 'model_index'


TypeError: run_model() missing 1 required positional argument: 'model_index'

In [16]:
# ================================================================
# T1.4a — INSPECT RESTORED run_model SIGNATURE
# READ-ONLY
# ================================================================

import inspect

print("=" * 72)
print("run_model SIGNATURE")
print("=" * 72)

print(inspect.signature(run_model))

print("\n" + "=" * 72)
print("run_model SOURCE")
print("=" * 72)

print(inspect.getsource(run_model))

run_model SIGNATURE
(model_name, gpu_id, model_index)

run_model SOURCE
def run_model(
    model_name,
    gpu_id,
    model_index
):

    device = torch.device(
        f"cuda:{gpu_id}"
    )

    torch.cuda.set_device(
        device
    )

    start_time = time.time()

    print(
        f"\n[{model_name}] START "
        f"GPU={gpu_id}"
    )

    # ------------------------------------------------------------
    # Fresh pristine production checkpoint
    # ------------------------------------------------------------

    model = load_model(
        model_name,
        device
    )

    # ------------------------------------------------------------
    # Original reference maps
    # ------------------------------------------------------------

    reference_maps = {}
    reference_meta = {}

    for i, row in subset.iterrows():

        image_id = row["image"]
        cache_row = int(
            row["row"]
        )

        hair = np.array(
            hair_mm[cache_row],
      

In [17]:
print("=" * 72)
print("randomize_parameters SIGNATURE")
print("=" * 72)

print(inspect.signature(randomize_parameters))

randomize_parameters SIGNATURE
(model, model_name, cascade_level, randomized_names)


In [18]:
import inspect

print("=" * 72)
print("run_model SIGNATURE")
print("=" * 72)
print(inspect.signature(run_model))

run_model SIGNATURE
(model_name, gpu_id, model_index)


In [19]:
print("\nrun_model SOURCE:")
print(inspect.getsource(run_model))


run_model SOURCE:
def run_model(
    model_name,
    gpu_id,
    model_index
):

    device = torch.device(
        f"cuda:{gpu_id}"
    )

    torch.cuda.set_device(
        device
    )

    start_time = time.time()

    print(
        f"\n[{model_name}] START "
        f"GPU={gpu_id}"
    )

    # ------------------------------------------------------------
    # Fresh pristine production checkpoint
    # ------------------------------------------------------------

    model = load_model(
        model_name,
        device
    )

    # ------------------------------------------------------------
    # Original reference maps
    # ------------------------------------------------------------

    reference_maps = {}
    reference_meta = {}

    for i, row in subset.iterrows():

        image_id = row["image"]
        cache_row = int(
            row["row"]
        )

        hair = np.array(
            hair_mm[cache_row],
            dtype=np.uint8,
            copy=True
        )

In [20]:
# ================================================================
# T1.4a — CORRECTED RECOVERY EXECUTION
# ================================================================
#
# Uses the EXACT restored run_model(model_name, gpu_id, model_index)
#
# Existing valid:
#   rho_0.8_rep_1
#   rho_0.8_rep_2
#
# Remaining:
#   10 models
#
# 2 models per batch, one per T4.
# ================================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc
import torch

# ------------------------------------------------
# EXACT result directory used by run_model()
# ------------------------------------------------

RESULT_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Keep the alias too, so our audit code is unambiguous.
RESULTS_DIR = RESULT_DIR

# ------------------------------------------------
# Frozen 12-model order
# ------------------------------------------------

ALL_MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",

    "rho_0.2_rep_1",
    "rho_0.2_rep_2",

    "rho_0.4_rep_1",
    "rho_0.4_rep_2",

    "rho_0.6_rep_1",
    "rho_0.6_rep_2",

    "rho_0.8_rep_1",
    "rho_0.8_rep_2",

    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

# ------------------------------------------------
# Safety: exact restored function must exist
# ------------------------------------------------

assert "run_model" in globals()
assert "randomize_parameters" in globals()

print("=" * 72)
print("T1.4a CORRECTED RECOVERY")
print("=" * 72)

print("\nrun_model signature:")
import inspect
print(inspect.signature(run_model))

assert list(
    inspect.signature(run_model).parameters.keys()
) == [
    "model_name",
    "gpu_id",
    "model_index",
]

print("\nrun_model signature: PASS")

# ------------------------------------------------
# Result validator
# ------------------------------------------------

REQUIRED_COLUMNS = {
    "model_name",
    "rho",
    "replicate",
    "image",
    "lesion_id",
    "class_name",
    "malignant",
    "cascade_level",
    "cascade_name",
    "randomized_parameters",
    "similarity_spearman",
    "reference_steps_used",
    "reference_completeness_error",
    "randomized_steps_used",
    "randomized_completeness_error",
    "randomized_completeness_failed",
}


def validate_result_file(model_name):

    path = RESULT_DIR / f"{model_name}.csv"

    if not path.exists():
        return False

    try:
        df = pd.read_csv(path)

        # Schema
        if not REQUIRED_COLUMNS.issubset(df.columns):
            return False

        # 12 x 100 x 7 design
        if len(df) != 700:
            return False

        if df["image"].nunique() != 100:
            return False

        if sorted(
            df["cascade_level"].unique().tolist()
        ) != list(range(7)):
            return False

        counts = df.groupby("cascade_level").size()

        if not all(
            counts.get(i, 0) == 100
            for i in range(7)
        ):
            return False

        # No image/cascade duplicates
        if df.duplicated(
            subset=["image", "cascade_level"]
        ).sum() != 0:
            return False

        # Similarity must be finite
        if not np.isfinite(
            df["similarity_spearman"]
        ).all():
            return False

        # Original/reference level must be exactly 1
        level0 = df.loc[
            df["cascade_level"] == 0,
            "similarity_spearman"
        ]

        if len(level0) != 100:
            return False

        if not (level0 == 1.0).all():
            return False

        # Model identity
        if df["model_name"].nunique() != 1:
            return False

        if df["model_name"].iloc[0] != model_name:
            return False

        return True

    except Exception:
        return False


# ------------------------------------------------
# Detect existing valid results
# ------------------------------------------------

completed = [
    m
    for m in ALL_MODELS
    if validate_result_file(m)
]

remaining = [
    m
    for m in ALL_MODELS
    if m not in completed
]

print("\nExisting VALID results:")

for m in completed:
    print("  PASS:", m)

print("\nModels still required:")

for m in remaining:
    print("  TODO:", m)

print(
    f"\nProgress: {len(completed)}/12"
)

# ------------------------------------------------
# Critical safety assertion
# ------------------------------------------------
#
# We expect exactly the two already completed models.
# ------------------------------------------------

assert set(completed) == {
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
}, (
    "Unexpected result state detected. "
    "STOP rather than launching."
)

assert len(remaining) == 10

print("\nPRE-LAUNCH STATE: PASS")

# ================================================================
# FIVE BATCHES × TWO MODELS
# ================================================================

for batch_number, start in enumerate(
    range(0, len(remaining), 2),
    start=1
):

    batch = remaining[start:start + 2]

    print()
    print("=" * 72)
    print(
        f"BATCH {batch_number}/5"
    )
    print("=" * 72)

    # Stable model indices from the frozen ALL_MODELS order.
    jobs = []

    for gpu_id, model_name in enumerate(batch):

        model_index = ALL_MODELS.index(
            model_name
        )

        jobs.append(
            (
                model_name,
                gpu_id,
                model_index
            )
        )

        print(
            f"GPU {gpu_id} -> "
            f"{model_name} "
            f"(model_index={model_index})"
        )

    print()

    batch_start = time.time()

    # ------------------------------------------------------------
    # Two simultaneous models
    # ------------------------------------------------------------

    futures = {}

    with ThreadPoolExecutor(
        max_workers=len(jobs)
    ) as executor:

        for (
            model_name,
            gpu_id,
            model_index
        ) in jobs:

            future = executor.submit(
                run_model,
                model_name,
                gpu_id,
                model_index
            )

            futures[future] = (
                model_name,
                gpu_id,
                model_index
            )

        # --------------------------------------------------------
        # Wait for both models
        # --------------------------------------------------------

        for future in as_completed(futures):

            (
                model_name,
                gpu_id,
                model_index
            ) = futures[future]

            try:

                result = future.result()

                print()
                print(
                    f"FINISHED: {model_name}"
                )
                print(
                    f"GPU: {gpu_id}"
                )
                print(
                    f"model_index: {model_index}"
                )

                if result is not None:
                    print(
                        "Result:",
                        result
                    )

            except Exception as exc:

                print()
                print("=" * 72)
                print("MODEL FAILED — STOPPING")
                print("=" * 72)
                print("Model:", model_name)
                print("GPU:", gpu_id)
                print("model_index:", model_index)
                print(
                    "Error:",
                    type(exc).__name__,
                    str(exc)
                )

                raise

    batch_elapsed = (
        time.time() - batch_start
    )

    print()
    print(
        f"BATCH {batch_number}/5 COMPLETE"
    )
    print(
        f"Runtime: "
        f"{batch_elapsed / 60:.2f} minutes"
    )

    # ------------------------------------------------------------
    # Immediate validation
    # ------------------------------------------------------------

    print("\nPOST-BATCH AUDIT")

    for model_name in batch:

        path = (
            RESULT_DIR /
            f"{model_name}.csv"
        )

        assert path.exists(), (
            f"Missing result: {path}"
        )

        assert validate_result_file(
            model_name
        ), (
            f"Validation failed: "
            f"{model_name}"
        )

        df = pd.read_csv(path)

        print(
            f"  PASS | {model_name} | "
            f"{len(df)} rows | "
            f"{df['image'].nunique()} images | "
            f"7 levels"
        )

    print("\nPOST-BATCH AUDIT: PASS")

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ================================================================
# FINAL AUDIT
# ================================================================

print()
print("=" * 72)
print("FINAL T1.4a COMPLETENESS AUDIT")
print("=" * 72)

for model_name in ALL_MODELS:

    assert validate_result_file(
        model_name
    ), (
        f"Final validation failed: "
        f"{model_name}"
    )

    print(
        f"PASS | {model_name}"
    )

print()
print("=" * 72)
print("T1.4a RECOVERY COMPLETE")
print("=" * 72)

print("Models:             12 / 12")
print("Images/model:       100")
print("Cascade levels:     7")
print("Rows/model:         700")
print("Total observations: 8,400")
print("=" * 72)

T1.4a CORRECTED RECOVERY

run_model signature:
(model_name, gpu_id, model_index)

run_model signature: PASS

Existing VALID results:
  PASS: rho_0.8_rep_1
  PASS: rho_0.8_rep_2

Models still required:
  TODO: rho_0.0_rep_1
  TODO: rho_0.0_rep_2
  TODO: rho_0.2_rep_1
  TODO: rho_0.2_rep_2
  TODO: rho_0.4_rep_1
  TODO: rho_0.4_rep_2
  TODO: rho_0.6_rep_1
  TODO: rho_0.6_rep_2
  TODO: rho_1.0_rep_1
  TODO: rho_1.0_rep_2

Progress: 2/12

PRE-LAUNCH STATE: PASS

BATCH 1/5
GPU 0 -> rho_0.0_rep_1 (model_index=0)
GPU 1 -> rho_0.0_rep_2 (model_index=1)


[rho_0.0_rep_2] START GPU=1

[rho_0.0_rep_1] START GPU=0
[rho_0.0_rep_1] reference maps complete
[rho_0.0_rep_1] level 0 (original) complete: 0.00 min
[rho_0.0_rep_2] reference maps complete
[rho_0.0_rep_2] level 0 (original) complete: 0.00 min
[rho_0.0_rep_1] level 1 (fc) complete: 0.49 min
[rho_0.0_rep_2] level 1 (fc) complete: 0.48 min
[rho_0.0_rep_1] level 2 (fc_layer4) complete: 0.22 min
[rho_0.0_rep_2] level 2 (fc_layer4) complete: 0.23 m

In [21]:
# ================================================================
# T1.4a — FULL STATISTICAL / SCIENTIFIC AUDIT
# ================================================================
#
# Uses ONLY the completed 12 x 100 x 7 result files.
# NO GPU / NO model loading / NO modification of primary evidence.
#
# Outputs:
#   1. integrity audit
#   2. cascade-level descriptive statistics
#   3. rho-wise statistics
#   4. replicate consistency
#   5. monotonicity analysis
#   6. paired image-level effect sizes
#   7. completeness diagnostics
#   8. publication-ready CSV summaries
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import scipy.stats as stats

# ------------------------------------------------
# Paths
# ------------------------------------------------

RESULT_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

AUDIT_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/T1_4A_STATISTICAL_AUDIT"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# Frozen model list
# ------------------------------------------------

MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.2_rep_1",
    "rho_0.2_rep_2",
    "rho_0.4_rep_1",
    "rho_0.4_rep_2",
    "rho_0.6_rep_1",
    "rho_0.6_rep_2",
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

CASCADE_LEVELS = list(range(7))

# ------------------------------------------------
# Load
# ------------------------------------------------

frames = []

for model_name in MODELS:

    path = RESULT_DIR / f"{model_name}.csv"

    assert path.exists(), (
        f"Missing result file: {path}"
    )

    df = pd.read_csv(path)

    df["_source_file"] = path.name

    frames.append(df)

data = pd.concat(
    frames,
    ignore_index=True
)

print("=" * 72)
print("T1.4a FULL DATASET LOADED")
print("=" * 72)

print("Rows:", len(data))
print("Models:", data["model_name"].nunique())
print("Images/model:", data.groupby("model_name")["image"].nunique().unique())
print("Cascade levels:", sorted(data["cascade_level"].unique()))

assert len(data) == 8400
assert data["model_name"].nunique() == 12
assert sorted(data["cascade_level"].unique()) == CASCADE_LEVELS

print("\nLOAD CHECK: PASS")


# ================================================================
# 1. GLOBAL INTEGRITY AUDIT
# ================================================================

print()
print("=" * 72)
print("1. GLOBAL INTEGRITY")
print("=" * 72)

integrity = []

for model_name in MODELS:

    d = data[
        data["model_name"] == model_name
    ].copy()

    checks = {
        "rows_700": len(d) == 700,
        "images_100": d["image"].nunique() == 100,
        "levels_7": sorted(
            d["cascade_level"].unique()
        ) == CASCADE_LEVELS,
        "no_duplicate_image_level":
            d.duplicated(
                ["image", "cascade_level"]
            ).sum() == 0,
        "finite_similarity":
            np.isfinite(
                d["similarity_spearman"]
            ).all(),
        "level0_exact_1":
            (
                d.loc[
                    d["cascade_level"] == 0,
                    "similarity_spearman"
                ] == 1.0
            ).all(),
        "100_per_level":
            all(
                d.groupby("cascade_level").size()
                == 100
            ),
    }

    integrity.append({
        "model_name": model_name,
        **checks,
        "all_pass": all(checks.values())
    })

integrity_df = pd.DataFrame(integrity)

display(integrity_df)

assert integrity_df["all_pass"].all()

print("\nGLOBAL INTEGRITY: PASS")


# ================================================================
# 2. COMPLETENESS DIAGNOSTICS
# ================================================================

print()
print("=" * 72)
print("2. COMPLETENESS DIAGNOSTICS")
print("=" * 72)

completeness_summary = (
    data
    .groupby(["model_name", "cascade_level"])
    .agg(
        n=("image", "size"),
        completeness_failures=(
            "randomized_completeness_failed",
            "sum"
        ),
        mean_completeness_error=(
            "randomized_completeness_error",
            "mean"
        ),
        median_completeness_error=(
            "randomized_completeness_error",
            "median"
        ),
    )
    .reset_index()
)

display(
    completeness_summary.head(20)
)

print(
    "\nTotal randomized completeness failures:",
    int(
        data["randomized_completeness_failed"].sum()
    )
)

print(
    "Total randomized observations:",
    len(
        data[data["cascade_level"] > 0]
    )
)

# Level 0 has no randomized completeness calculation.
randomized_data = data[
    data["cascade_level"] > 0
]

assert np.isfinite(
    randomized_data[
        "randomized_completeness_error"
    ]
).all()

completeness_fail_count = int(
    randomized_data[
        "randomized_completeness_failed"
    ].sum()
)

print(
    "Failure rate:",
    completeness_fail_count /
    len(randomized_data)
)


# ================================================================
# 3. CASCADE-LEVEL DESCRIPTIVE STATISTICS
# ================================================================

print()
print("=" * 72)
print("3. CASCADE-LEVEL DESCRIPTIVES")
print("=" * 72)

cascade_summary = (
    data
    .groupby("cascade_level")[
        "similarity_spearman"
    ]
    .agg(
        n="count",
        mean="mean",
        std="std",
        median="median",
        min="min",
        max="max",
    )
    .reset_index()
)

# Standard error + 95% CI
cascade_summary["sem"] = (
    cascade_summary["std"] /
    np.sqrt(cascade_summary["n"])
)

cascade_summary["ci95_low"] = (
    cascade_summary["mean"]
    - 1.96 * cascade_summary["sem"]
)

cascade_summary["ci95_high"] = (
    cascade_summary["mean"]
    + 1.96 * cascade_summary["sem"]
)

display(cascade_summary)

cascade_summary.to_csv(
    AUDIT_DIR /
    "T1_4A_CASCADE_LEVEL_SUMMARY.csv",
    index=False
)


# ================================================================
# 4. RHO × CASCADE SUMMARY
# ================================================================

print()
print("=" * 72)
print("4. RHO × CASCADE")
print("=" * 72)

rho_cascade = (
    data
    .groupby(
        ["rho", "cascade_level"]
    )["similarity_spearman"]
    .agg(
        n="count",
        mean="mean",
        std="std",
        median="median",
        min="min",
        max="max",
    )
    .reset_index()
)

rho_cascade["sem"] = (
    rho_cascade["std"] /
    np.sqrt(rho_cascade["n"])
)

rho_cascade["ci95_low"] = (
    rho_cascade["mean"]
    - 1.96 * rho_cascade["sem"]
)

rho_cascade["ci95_high"] = (
    rho_cascade["mean"]
    + 1.96 * rho_cascade["sem"]
)

display(rho_cascade)

rho_cascade.to_csv(
    AUDIT_DIR /
    "T1_4A_RHO_CASCADE_SUMMARY.csv",
    index=False
)


# ================================================================
# 5. MODEL × CASCADE SUMMARY
# ================================================================

print()
print("=" * 72)
print("5. MODEL × CASCADE")
print("=" * 72)

model_cascade = (
    data
    .groupby(
        ["model_name", "rho", "replicate",
         "cascade_level"]
    )["similarity_spearman"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
    .reset_index()
)

display(model_cascade)

model_cascade.to_csv(
    AUDIT_DIR /
    "T1_4A_MODEL_CASCADE_SUMMARY.csv",
    index=False
)


# ================================================================
# 6. IMAGE-PAIRED EFFECTS
# ================================================================
#
# Since the SAME 100 images are evaluated at every cascade level,
# the most informative effect is paired change from level 0.
#
# Level 0 = exact self-reference (similarity = 1)
#
# Therefore:
#   degradation = 1 - similarity
#
# We calculate this for each image/model/level.
# ================================================================

paired = data.copy()

paired["degradation_from_original"] = (
    1.0 -
    paired["similarity_spearman"]
)

degradation_summary = (
    paired
    .groupby("cascade_level")
    ["degradation_from_original"]
    .agg(
        n="count",
        mean="mean",
        std="std",
        median="median",
        min="min",
        max="max",
    )
    .reset_index()
)

display(degradation_summary)

degradation_summary.to_csv(
    AUDIT_DIR /
    "T1_4A_DEGRADATION_SUMMARY.csv",
    index=False
)


# ================================================================
# 7. PER-MODEL MONOTONICITY
# ================================================================
#
# For each model, calculate:
#
#   Spearman(cascade level, mean similarity)
#   Kendall(cascade level, mean similarity)
#
# Negative values indicate decreasing similarity as more
# parameters are randomized.
# ================================================================

monotonicity_rows = []

for model_name in MODELS:

    d = (
        model_cascade[
            model_cascade["model_name"]
            == model_name
        ]
        .sort_values("cascade_level")
    )

    x = d["cascade_level"].to_numpy()
    y = d["mean"].to_numpy()

    sp = stats.spearmanr(x, y)
    kt = stats.kendalltau(x, y)

    monotonicity_rows.append({
        "model_name": model_name,
        "rho": d["rho"].iloc[0],
        "replicate": d["replicate"].iloc[0],
        "spearman_rho": sp.statistic,
        "spearman_p": sp.pvalue,
        "kendall_tau": kt.statistic,
        "kendall_p": kt.pvalue,
        "mean_level0": y[0],
        "mean_level6": y[-1],
        "absolute_drop":
            y[0] - y[-1],
    })

monotonicity_df = pd.DataFrame(
    monotonicity_rows
)

display(monotonicity_df)

monotonicity_df.to_csv(
    AUDIT_DIR /
    "T1_4A_MODEL_MONOTONICITY.csv",
    index=False
)


# ================================================================
# 8. REPLICATE CONSISTENCY
# ================================================================

print()
print("=" * 72)
print("8. REPLICATE CONSISTENCY")
print("=" * 72)

replicate_rows = []

for rho in sorted(data["rho"].unique()):

    r1 = (
        model_cascade[
            (model_cascade["rho"] == rho) &
            (model_cascade["replicate"] == 1)
        ]
        .sort_values("cascade_level")
    )

    r2 = (
        model_cascade[
            (model_cascade["rho"] == rho) &
            (model_cascade["replicate"] == 2)
        ]
        .sort_values("cascade_level")
    )

    y1 = r1["mean"].to_numpy()
    y2 = r2["mean"].to_numpy()

    sp = stats.spearmanr(y1, y2)
    kt = stats.kendalltau(y1, y2)

    replicate_rows.append({
        "rho": rho,
        "replicate1_mean_level6": y1[-1],
        "replicate2_mean_level6": y2[-1],
        "replicate_mean_absolute_difference":
            np.mean(np.abs(y1 - y2)),
        "replicate_spearman":
            sp.statistic,
        "replicate_spearman_p":
            sp.pvalue,
        "replicate_kendall_tau":
            kt.statistic,
        "replicate_kendall_p":
            kt.pvalue,
    })

replicate_df = pd.DataFrame(
    replicate_rows
)

display(replicate_df)

replicate_df.to_csv(
    AUDIT_DIR /
    "T1_4A_REPLICATE_CONSISTENCY.csv",
    index=False
)


# ================================================================
# 9. IMAGE-LEVEL LEVEL-6 DEGRADATION
# ================================================================
#
# Level 6 is the strongest randomization:
# all specified parameter groups randomized.
#
# Calculate distribution of:
#
#   1 - Spearman similarity
#
# across the 100 images for every model.
# ================================================================

level6 = data[
    data["cascade_level"] == 6
].copy()

level6["degradation"] = (
    1 -
    level6["similarity_spearman"]
)

level6_summary = (
    level6
    .groupby("model_name")["degradation"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        q25=lambda x: x.quantile(.25),
        q75=lambda x: x.quantile(.75),
        q10=lambda x: x.quantile(.10),
        q90=lambda x: x.quantile(.90),
        min="min",
        max="max",
    )
    .reset_index()
)

display(level6_summary)

level6_summary.to_csv(
    AUDIT_DIR /
    "T1_4A_LEVEL6_IMAGE_DEGRADATION.csv",
    index=False
)


# ================================================================
# 10. PERCENTAGE OF IMAGES WITH SUBSTANTIAL DEGRADATION
# ================================================================
#
# These are descriptive thresholds, NOT significance tests.
# ================================================================

thresholds = [
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
]

threshold_rows = []

for model_name in MODELS:

    d = level6[
        level6["model_name"] == model_name
    ]

    row = {
        "model_name": model_name,
        "rho": d["rho"].iloc[0],
    }

    for threshold in thresholds:

        row[
            f"fraction_degradation_ge_{threshold:.2f}"
        ] = (
            d["degradation"] >= threshold
        ).mean()

    threshold_rows.append(row)

threshold_df = pd.DataFrame(
    threshold_rows
)

display(threshold_df)

threshold_df.to_csv(
    AUDIT_DIR /
    "T1_4A_LEVEL6_DEGRADATION_THRESHOLDS.csv",
    index=False
)


# ================================================================
# 11. BENIGN vs MALIGNANT
# ================================================================

print()
print("=" * 72)
print("11. BENIGN vs MALIGNANT")
print("=" * 72)

class_summary = (
    data
    .groupby(
        ["malignant", "cascade_level"]
    )["similarity_spearman"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
    )
    .reset_index()
)

display(class_summary)

class_summary.to_csv(
    AUDIT_DIR /
    "T1_4A_CLASS_CASCADE_SUMMARY.csv",
    index=False
)


# ================================================================
# 12. GLOBAL CASCADE TREND
# ================================================================
#
# Treat each MODEL as the unit for the trend analysis rather than
# pretending 8,400 observations are independent.
#
# First average over images within each model/cascade.
# Then test the 12 model-level trajectories.
# ================================================================

model_level = (
    data
    .groupby(
        ["model_name", "rho", "replicate",
         "cascade_level"]
    )["similarity_spearman"]
    .mean()
    .reset_index()
)

global_trend_rows = []

for model_name in MODELS:

    d = (
        model_level[
            model_level["model_name"] == model_name
        ]
        .sort_values("cascade_level")
    )

    sp = stats.spearmanr(
        d["cascade_level"],
        d["similarity_spearman"]
    )

    global_trend_rows.append({
        "model_name": model_name,
        "rho": d["rho"].iloc[0],
        "replicate": d["replicate"].iloc[0],
        "trend_spearman":
            sp.statistic,
        "trend_p":
            sp.pvalue,
    })

global_trend = pd.DataFrame(
    global_trend_rows
)

print(
    "Per-model monotonic trend:"
)

display(global_trend)

global_trend.to_csv(
    AUDIT_DIR /
    "T1_4A_GLOBAL_MODEL_TRENDS.csv",
    index=False
)


# ================================================================
# 13. MASTER AUDIT MANIFEST
# ================================================================

manifest = {
    "experiment": "T1.4a_parameter_randomization",
    "models": 12,
    "images_per_model": 100,
    "cascade_levels": 7,
    "total_rows": int(len(data)),
    "rho_levels": sorted(
        data["rho"].unique().tolist()
    ),
    "replicates_per_rho": 2,
    "level0_similarity_exact": True,
    "all_similarity_finite": bool(
        np.isfinite(
            data["similarity_spearman"]
        ).all()
    ),
    "duplicate_image_level_rows": int(
        data.duplicated(
            ["model_name",
             "image",
             "cascade_level"]
        ).sum()
    ),
    "randomized_completeness_failures": int(
        data[
            "randomized_completeness_failed"
        ].sum()
    ),
    "audit_files": sorted(
        p.name
        for p in AUDIT_DIR.glob("*")
        if p.is_file()
    ),
}

with open(
    AUDIT_DIR /
    "T1_4A_STATISTICAL_AUDIT_MANIFEST.json",
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

print()
print("=" * 72)
print("T1.4a STATISTICAL AUDIT COMPLETE")
print("=" * 72)

print(
    "Audit directory:",
    AUDIT_DIR
)

print(
    "Rows analyzed:",
    len(data)
)

print(
    "Models:",
    data["model_name"].nunique()
)

print(
    "Randomized completeness failures:",
    int(
        data[
            "randomized_completeness_failed"
        ].sum()
    )
)

print("=" * 72)

T1.4a FULL DATASET LOADED
Rows: 8400
Models: 12
Images/model: [100]
Cascade levels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

LOAD CHECK: PASS

1. GLOBAL INTEGRITY


,model_name,rows_700,images_100,levels_7,no_duplicate_image_level,finite_similarity,level0_exact_1,100_per_level,all_pass
0,rho_0.0_rep_1,True,True,True,True,True,True,True,True
1,rho_0.0_rep_2,True,True,True,True,True,True,True,True
2,rho_0.2_rep_1,True,True,True,True,True,True,True,True
3,rho_0.2_rep_2,True,True,True,True,True,True,True,True
4,rho_0.4_rep_1,True,True,True,True,True,True,True,True
5,rho_0.4_rep_2,True,True,True,True,True,True,True,True
6,rho_0.6_rep_1,True,True,True,True,True,True,True,True
7,rho_0.6_rep_2,True,True,True,True,True,True,True,True
8,rho_0.8_rep_1,True,True,True,True,True,True,True,True
9,rho_0.8_rep_2,True,True,True,True,True,True,True,True



GLOBAL INTEGRITY: PASS

2. COMPLETENESS DIAGNOSTICS


,model_name,cascade_level,n,completeness_failures,mean_completeness_error,median_completeness_error
0,rho_0.0_rep_1,0,100,0,0.000000e+00,0.000000e+00
1,rho_0.0_rep_1,1,100,42,9.278181e-02,7.252246e-02
2,rho_0.0_rep_1,2,100,0,2.463498e-03,2.022481e-03
3,rho_0.0_rep_1,3,100,0,1.695284e-04,1.476186e-04
4,rho_0.0_rep_1,4,100,0,2.455661e-06,1.608472e-06
5,rho_0.0_rep_1,5,100,0,3.589856e-07,3.235327e-07
6,rho_0.0_rep_1,6,100,0,2.138203e-06,1.709115e-06
7,rho_0.0_rep_2,0,100,0,0.000000e+00,0.000000e+00
8,rho_0.0_rep_2,1,100,40,9.404691e-02,7.200122e-02
9,rho_0.0_rep_2,2,100,0,9.855136e-04,8.436701e-04



Total randomized completeness failures: 508
Total randomized observations: 7200
Failure rate: 0.07055555555555555

3. CASCADE-LEVEL DESCRIPTIVES


...


11. BENIGN vs MALIGNANT


,malignant,cascade_level,n,mean,median,std
0,0,0,600,1.000000,1.000000,0.000000
1,0,1,600,0.718141,0.718537,0.050850
2,0,2,600,0.618066,0.614155,0.035655
3,0,3,600,0.568412,0.564881,0.038931
4,0,4,600,0.415126,0.413370,0.046292
5,0,5,600,0.476342,0.474620,0.043152
6,0,6,600,0.442640,0.442928,0.051317
7,1,0,600,1.000000,1.000000,0.000000
8,1,1,600,0.756337,0.741426,0.074390
9,1,2,600,0.645854,0.626914,0.095362


Per-model monotonic trend:


,model_name,rho,replicate,trend_spearman,trend_p
0,rho_0.0_rep_1,0.0,1,-0.892857,0.006807
1,rho_0.0_rep_2,0.0,2,-0.964286,0.000454
2,rho_0.2_rep_1,0.2,1,-0.892857,0.006807
3,rho_0.2_rep_2,0.2,2,-0.892857,0.006807
4,rho_0.4_rep_1,0.4,1,-0.892857,0.006807
5,rho_0.4_rep_2,0.4,2,-0.964286,0.000454
6,rho_0.6_rep_1,0.6,1,-0.892857,0.006807
7,rho_0.6_rep_2,0.6,2,-0.857143,0.013697
8,rho_0.8_rep_1,0.8,1,-0.892857,0.006807
9,rho_0.8_rep_2,0.8,2,-0.892857,0.006807



T1.4a STATISTICAL AUDIT COMPLETE
Audit directory: /kaggle/working/T1_4a_parameter_randomization_12x100/T1_4A_STATISTICAL_AUDIT
Rows analyzed: 8400
Models: 12
Randomized completeness failures: 508


In [22]:
# ================================================================
# T1.4a — COMPLETENESS-FAILURE SENSITIVITY AUDIT
# ================================================================
#
# Question:
# Does the parameter-randomization result remain after excluding
# observations where the randomized IG completeness check failed?
#
# NO MODEL RECOMPUTATION.
# CPU ONLY.
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

RESULT_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.2_rep_1",
    "rho_0.2_rep_2",
    "rho_0.4_rep_1",
    "rho_0.4_rep_2",
    "rho_0.6_rep_1",
    "rho_0.6_rep_2",
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

frames = []

for model_name in MODELS:

    df = pd.read_csv(
        RESULT_DIR / f"{model_name}.csv"
    )

    frames.append(df)

data = pd.concat(
    frames,
    ignore_index=True
)

randomized = data[
    data["cascade_level"] > 0
].copy()

# ------------------------------------------------
# Failure distribution
# ------------------------------------------------

failure_table = (
    randomized
    .groupby("cascade_level")
    .agg(
        n=("similarity_spearman", "size"),
        failures=(
            "randomized_completeness_failed",
            "sum"
        ),
    )
    .reset_index()
)

failure_table["failure_rate"] = (
    failure_table["failures"] /
    failure_table["n"]
)

print("=" * 72)
print("COMPLETENESS FAILURE DISTRIBUTION")
print("=" * 72)

display(failure_table)

# ------------------------------------------------
# Compare similarity:
#
# ALL observations
# PASS-only observations
# FAIL-only observations
# ------------------------------------------------

rows = []

for level in range(1, 7):

    d = randomized[
        randomized["cascade_level"] == level
    ]

    passed = d[
        ~d["randomized_completeness_failed"]
    ]

    failed = d[
        d["randomized_completeness_failed"]
    ]

    for label, subset in [
        ("all", d),
        ("completeness_pass", passed),
        ("completeness_fail", failed),
    ]:

        if len(subset) == 0:
            continue

        rows.append({
            "cascade_level": level,
            "group": label,
            "n": len(subset),
            "mean_similarity":
                subset[
                    "similarity_spearman"
                ].mean(),
            "median_similarity":
                subset[
                    "similarity_spearman"
                ].median(),
            "std_similarity":
                subset[
                    "similarity_spearman"
                ].std(),
            "mean_degradation":
                (
                    1 -
                    subset[
                        "similarity_spearman"
                    ]
                ).mean(),
        })

comparison = pd.DataFrame(rows)

print()
print("=" * 72)
print("SIMILARITY BY COMPLETENESS STATUS")
print("=" * 72)

display(comparison)

# ------------------------------------------------
# Difference between failed and passed observations
# ------------------------------------------------

print()
print("=" * 72)
print("FAILED vs PASSED — DISTRIBUTION DIFFERENCES")
print("=" * 72)

difference_rows = []

for level in range(1, 7):

    d = randomized[
        randomized["cascade_level"] == level
    ]

    passed = d[
        ~d["randomized_completeness_failed"]
    ]["similarity_spearman"]

    failed = d[
        d["randomized_completeness_failed"]
    ]["similarity_spearman"]

    if len(failed) == 0:
        continue

    mw = stats.mannwhitneyu(
        passed,
        failed,
        alternative="two-sided"
    )

    difference_rows.append({
        "cascade_level": level,
        "n_pass": len(passed),
        "n_fail": len(failed),
        "mean_pass": passed.mean(),
        "mean_fail": failed.mean(),
        "median_pass": passed.median(),
        "median_fail": failed.median(),
        "mean_difference_fail_minus_pass":
            failed.mean() - passed.mean(),
        "mann_whitney_U":
            mw.statistic,
        "mann_whitney_p":
            mw.pvalue,
    })

difference_df = pd.DataFrame(
    difference_rows
)

display(difference_df)

# ------------------------------------------------
# Main sensitivity:
#
# model-level cascade trajectories using:
#   A) all observations
#   B) completeness-pass only
#
# ------------------------------------------------

print()
print("=" * 72)
print("MODEL-LEVEL TREND SENSITIVITY")
print("=" * 72)

trend_rows = []

for model_name in MODELS:

    d = data[
        data["model_name"] == model_name
    ].copy()

    # All observations
    all_means = (
        d.groupby("cascade_level")
        ["similarity_spearman"]
        .mean()
        .sort_index()
    )

    # Exclude failed randomized observations.
    # Level 0 is always retained.
    d_pass = d[
        (d["cascade_level"] == 0) |
        (~d["randomized_completeness_failed"])
    ]

    pass_means = (
        d_pass.groupby("cascade_level")
        ["similarity_spearman"]
        .mean()
        .sort_index()
    )

    # Require all seven levels to remain.
    if len(pass_means) != 7:
        print(
            "WARNING:",
            model_name,
            "missing cascade level after filtering"
        )
        continue

    all_x = all_means.index.to_numpy()
    all_y = all_means.to_numpy()

    pass_x = pass_means.index.to_numpy()
    pass_y = pass_means.to_numpy()

    all_sp = stats.spearmanr(
        all_x,
        all_y
    )

    pass_sp = stats.spearmanr(
        pass_x,
        pass_y
    )

    trend_rows.append({
        "model_name": model_name,
        "rho": d["rho"].iloc[0],
        "all_level6_mean":
            all_y[-1],
        "pass_only_level6_mean":
            pass_y[-1],
        "level6_difference_pass_minus_all":
            pass_y[-1] - all_y[-1],
        "all_trend_spearman":
            all_sp.statistic,
        "pass_only_trend_spearman":
            pass_sp.statistic,
        "all_trend_p":
            all_sp.pvalue,
        "pass_only_trend_p":
            pass_sp.pvalue,
    })

trend_sensitivity = pd.DataFrame(
    trend_rows
)

display(trend_sensitivity)

# ------------------------------------------------
# Global level summaries after exclusion
# ------------------------------------------------

pass_only = data[
    (data["cascade_level"] == 0) |
    (~data["randomized_completeness_failed"])
].copy()

global_pass = (
    pass_only
    .groupby("cascade_level")
    ["similarity_spearman"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
    )
    .reset_index()
)

print()
print("=" * 72)
print("GLOBAL PASS-ONLY CASCADE SUMMARY")
print("=" * 72)

display(global_pass)

# ------------------------------------------------
# Save
# ------------------------------------------------

AUDIT_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/"
    "T1_4A_STATISTICAL_AUDIT"
)

failure_table.to_csv(
    AUDIT_DIR /
    "T1_4A_COMPLETENESS_FAILURE_DISTRIBUTION.csv",
    index=False
)

comparison.to_csv(
    AUDIT_DIR /
    "T1_4A_COMPLETENESS_SIMILARITY_COMPARISON.csv",
    index=False
)

difference_df.to_csv(
    AUDIT_DIR /
    "T1_4A_COMPLETENESS_FAILED_VS_PASS_STATISTICS.csv",
    index=False
)

trend_sensitivity.to_csv(
    AUDIT_DIR /
    "T1_4A_COMPLETENESS_TREND_SENSITIVITY.csv",
    index=False
)

global_pass.to_csv(
    AUDIT_DIR /
    "T1_4A_COMPLETENESS_PASS_ONLY_GLOBAL_SUMMARY.csv",
    index=False
)

print()
print("=" * 72)
print("COMPLETENESS SENSITIVITY AUDIT COMPLETE")
print("=" * 72)

COMPLETENESS FAILURE DISTRIBUTION


,cascade_level,n,failures,failure_rate
0,1,1200,508,0.423333
1,2,1200,0,0.000000
2,3,1200,0,0.000000
3,4,1200,0,0.000000
4,5,1200,0,0.000000
5,6,1200,0,0.000000



SIMILARITY BY COMPLETENESS STATUS


,cascade_level,group,n,mean_similarity,median_similarity,std_similarity,mean_degradation
0,1,all,1200,0.737239,0.732130,0.066494,0.262761
1,1,completeness_pass,692,0.739902,0.733249,0.066299,0.260098
2,1,completeness_fail,508,0.733612,0.729273,0.066652,0.266388
3,2,all,1200,0.631960,0.618116,0.073290,0.368040
4,2,completeness_pass,1200,0.631960,0.618116,0.073290,0.368040
5,3,all,1200,0.582623,0.566779,0.077373,0.417377
6,3,completeness_pass,1200,0.582623,0.566779,0.077373,0.417377
7,4,all,1200,0.433492,0.416209,0.090488,0.566508
8,4,completeness_pass,1200,0.433492,0.416209,0.090488,0.566508
9,5,all,1200,0.490359,0.477448,0.081236,0.509641



FAILED vs PASSED — DISTRIBUTION DIFFERENCES


...


MODEL-LEVEL TREND SENSITIVITY


,model_name,rho,all_level6_mean,pass_only_level6_mean,level6_difference_pass_minus_all,all_trend_spearman,pass_only_trend_spearman,all_trend_p,pass_only_trend_p
0,rho_0.0_rep_1,0.0,0.446944,0.446944,0.0,-0.892857,-0.892857,0.006807,0.006807
1,rho_0.0_rep_2,0.0,0.431458,0.431458,0.0,-0.964286,-0.964286,0.000454,0.000454
2,rho_0.2_rep_1,0.2,0.459856,0.459856,0.0,-0.892857,-0.892857,0.006807,0.006807
3,rho_0.2_rep_2,0.2,0.455320,0.455320,0.0,-0.892857,-0.892857,0.006807,0.006807
4,rho_0.4_rep_1,0.4,0.454608,0.454608,0.0,-0.892857,-0.892857,0.006807,0.006807
5,rho_0.4_rep_2,0.4,0.425766,0.425766,0.0,-0.964286,-0.964286,0.000454,0.000454
6,rho_0.6_rep_1,0.6,0.471845,0.471845,0.0,-0.892857,-0.892857,0.006807,0.006807
7,rho_0.6_rep_2,0.6,0.480610,0.480610,0.0,-0.857143,-0.857143,0.013697,0.013697
8,rho_0.8_rep_1,0.8,0.485440,0.485440,0.0,-0.892857,-0.892857,0.006807,0.006807
9,rho_0.8_rep_2,0.8,0.471009,0.471009,0.0,-0.892857,-0.892857,0.006807,0.006807



GLOBAL PASS-ONLY CASCADE SUMMARY


,cascade_level,n,mean,median,std
0,0,1200,1.000000,1.000000,0.000000
1,1,692,0.739902,0.733249,0.066299
2,2,1200,0.631960,0.618116,0.073290
3,3,1200,0.582623,0.566779,0.077373
4,4,1200,0.433492,0.416209,0.090488
5,5,1200,0.490359,0.477448,0.081236
6,6,1200,0.458819,0.445075,0.090640



COMPLETENESS SENSITIVITY AUDIT COMPLETE


In [23]:
# ================================================================
# T1.4a — MODEL-LEVEL INFERENTIAL STATISTICS
# ================================================================
#
# Statistical unit = MODEL (n = 12)
# NOT individual image observations.
#
# Tests:
#   1. Friedman test across all 7 cascade levels
#   2. Paired Wilcoxon: level 0 vs levels 1..6
#   3. Holm correction
#   4. Model-level effect sizes
#
# No GPU / no model recomputation.
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

RESULT_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/model_level_results"
)

MODELS = [
    "rho_0.0_rep_1",
    "rho_0.0_rep_2",
    "rho_0.2_rep_1",
    "rho_0.2_rep_2",
    "rho_0.4_rep_1",
    "rho_0.4_rep_2",
    "rho_0.6_rep_1",
    "rho_0.6_rep_2",
    "rho_0.8_rep_1",
    "rho_0.8_rep_2",
    "rho_1.0_rep_1",
    "rho_1.0_rep_2",
]

frames = []

for model_name in MODELS:
    df = pd.read_csv(
        RESULT_DIR / f"{model_name}.csv"
    )
    frames.append(df)

data = pd.concat(
    frames,
    ignore_index=True
)

# ------------------------------------------------
# Exclude completeness-failed observations.
#
# Level 0 is retained automatically.
# Levels 2–6 have no failures anyway.
# ------------------------------------------------

analysis_data = data[
    (data["cascade_level"] == 0) |
    (~data["randomized_completeness_failed"])
].copy()

# ------------------------------------------------
# Model × cascade means
# ------------------------------------------------

model_level = (
    analysis_data
    .groupby(
        ["model_name", "rho", "replicate",
         "cascade_level"]
    )["similarity_spearman"]
    .mean()
    .reset_index()
)

pivot = (
    model_level
    .pivot(
        index="model_name",
        columns="cascade_level",
        values="similarity_spearman"
    )
    .loc[MODELS]
)

assert pivot.shape == (12, 7)

print("=" * 72)
print("MODEL-LEVEL CASCADE MATRIX")
print("=" * 72)

display(pivot)


# ================================================================
# 1. FRIEDMAN TEST
# ================================================================

friedman = stats.friedmanchisquare(
    *[
        pivot[level].to_numpy()
        for level in range(7)
    ]
)

print()
print("=" * 72)
print("FRIEDMAN TEST")
print("=" * 72)

print(
    "Statistic:",
    friedman.statistic
)

print(
    "p-value:",
    friedman.pvalue
)


# ================================================================
# 2. PAIRED WILCOXON VS LEVEL 0
# ================================================================

wilcox_rows = []

baseline = pivot[0].to_numpy()

for level in range(1, 7):

    randomized = pivot[level].to_numpy()

    differences = randomized - baseline

    # Exact=False because ties / zero differences can occur.
    test = stats.wilcoxon(
        randomized,
        baseline,
        alternative="less",
        zero_method="wilcox",
        correction=False,
        method="auto"
    )

    # Rank-biserial effect size:
    # positive magnitude here means randomized similarity is lower.
    nonzero = differences[differences != 0]

    if len(nonzero) > 0:

        ranks = stats.rankdata(
            np.abs(nonzero)
        )

        W_plus = ranks[
            nonzero > 0
        ].sum()

        W_minus = ranks[
            nonzero < 0
        ].sum()

        rbc = (
            W_minus - W_plus
        ) / (
            W_minus + W_plus
        )

    else:
        rbc = 0.0

    wilcox_rows.append({
        "cascade_level": level,
        "n_models": len(randomized),
        "mean_similarity": randomized.mean(),
        "median_similarity": np.median(randomized),
        "mean_degradation":
            np.mean(
                baseline - randomized
            ),
        "median_degradation":
            np.median(
                baseline - randomized
            ),
        "wilcoxon_W":
            test.statistic,
        "raw_p":
            test.pvalue,
        "rank_biserial_effect":
            rbc,
        "n_models_degraded":
            int(
                np.sum(
                    randomized < baseline
                )
            ),
    })

wilcox_df = pd.DataFrame(
    wilcox_rows
)


# ================================================================
# 3. HOLM MULTIPLE-TEST CORRECTION
# ================================================================

def holm_adjust(p_values):

    p = np.asarray(
        p_values,
        dtype=float
    )

    m = len(p)

    order = np.argsort(p)

    adjusted = np.empty(m)

    for rank, idx in enumerate(order):
        adjusted[idx] = (
            p[idx] *
            (m - rank)
        )

    # Enforce monotonicity in ordered sequence.
    ordered_adj = adjusted[order]
    ordered_adj = np.maximum.accumulate(
        ordered_adj
    )

    ordered_adj = np.minimum(
        ordered_adj,
        1.0
    )

    result = np.empty(m)
    result[order] = ordered_adj

    return result


wilcox_df["holm_p"] = holm_adjust(
    wilcox_df["raw_p"].to_numpy()
)

wilcox_df["significant_holm_0.05"] = (
    wilcox_df["holm_p"] < 0.05
)

print()
print("=" * 72)
print("PAIRED WILCOXON TESTS VS LEVEL 0")
print("=" * 72)

display(wilcox_df)


# ================================================================
# 4. DIRECTIONAL CONSISTENCY
# ================================================================

direction_rows = []

for level in range(1, 7):

    y = pivot[level].to_numpy()

    degradation = (
        pivot[0].to_numpy() - y
    )

    direction_rows.append({
        "cascade_level": level,
        "models_with_lower_similarity":
            int(np.sum(degradation > 0)),
        "models_with_equal_similarity":
            int(np.sum(degradation == 0)),
        "models_with_higher_similarity":
            int(np.sum(degradation < 0)),
        "fraction_lower":
            np.mean(degradation > 0),
    })

direction_df = pd.DataFrame(
    direction_rows
)

print()
print("=" * 72)
print("DIRECTIONAL CONSISTENCY")
print("=" * 72)

display(direction_df)


# ================================================================
# 5. SAVE
# ================================================================

AUDIT_DIR = Path(
    "/kaggle/working/T1_4a_parameter_randomization_12x100/"
    "T1_4A_STATISTICAL_AUDIT"
)

wilcox_df.to_csv(
    AUDIT_DIR /
    "T1_4A_MODEL_LEVEL_WILCOXON_HOLM.csv",
    index=False
)

direction_df.to_csv(
    AUDIT_DIR /
    "T1_4A_MODEL_LEVEL_DIRECTIONAL_CONSISTENCY.csv",
    index=False
)

pivot.to_csv(
    AUDIT_DIR /
    "T1_4A_MODEL_LEVEL_CASCADE_MATRIX.csv"
)

# ------------------------------------------------
# Final concise output
# ------------------------------------------------

print()
print("=" * 72)
print("T1.4a INFERENTIAL AUDIT COMPLETE")
print("=" * 72)

print(
    f"Friedman p = {friedman.pvalue:.6g}"
)

for _, row in wilcox_df.iterrows():

    print(
        f"Level {int(row['cascade_level'])}: "
        f"mean similarity = "
        f"{row['mean_similarity']:.6f}, "
        f"Holm p = "
        f"{row['holm_p']:.6g}, "
        f"models degraded = "
        f"{int(row['n_models_degraded'])}/12"
    )

print("=" * 72)

MODEL-LEVEL CASCADE MATRIX


cascade_level,0,1,2,3,4,5,6
model_name,,,,,,,
rho_0.0_rep_1,1.0,0.719942,0.622226,0.571915,0.432615,0.485477,0.446944
rho_0.0_rep_2,1.0,0.745602,0.623291,0.572901,0.439878,0.483325,0.431458
rho_0.2_rep_1,1.0,0.712365,0.626010,0.576875,0.445580,0.493036,0.459856
rho_0.2_rep_2,1.0,0.745782,0.641191,0.583004,0.442512,0.490025,0.455320
rho_0.4_rep_1,1.0,0.819652,0.634862,0.582111,0.423804,0.482802,0.454608
rho_0.4_rep_2,1.0,0.726568,0.639847,0.582162,0.438849,0.500625,0.425766
rho_0.6_rep_1,1.0,0.735911,0.629954,0.592874,0.434424,0.492232,0.471845
rho_0.6_rep_2,1.0,0.722420,0.636660,0.584759,0.425119,0.461579,0.480610
rho_0.8_rep_1,1.0,0.704532,0.625604,0.582782,0.429569,0.504500,0.485440



FRIEDMAN TEST
Statistic: 70.82142857142856
p-value: 2.773818563434795e-13

PAIRED WILCOXON TESTS VS LEVEL 0


,cascade_level,n_models,mean_similarity,median_similarity,mean_degradation,median_degradation,wilcoxon_W,raw_p,rank_biserial_effect,n_models_degraded,holm_p,significant_holm_0.05
0,1,12,0.738948,0.738875,0.261052,0.261125,0.0,0.000244,1.0,12,0.001465,True
1,2,12,0.631960,0.632408,0.368040,0.367592,0.0,0.000244,1.0,12,0.001465,True
2,3,12,0.582623,0.582525,0.417377,0.417475,0.0,0.000244,1.0,12,0.001465,True
3,4,12,0.433492,0.432628,0.566508,0.567372,0.0,0.000244,1.0,12,0.001465,True
4,5,12,0.490359,0.491128,0.509641,0.508872,0.0,0.000244,1.0,12,0.001465,True
5,6,12,0.458819,0.458518,0.541181,0.541482,0.0,0.000244,1.0,12,0.001465,True



DIRECTIONAL CONSISTENCY


,cascade_level,models_with_lower_similarity,models_with_equal_similarity,models_with_higher_similarity,fraction_lower
0,1,12,0,0,1.0
1,2,12,0,0,1.0
2,3,12,0,0,1.0
3,4,12,0,0,1.0
4,5,12,0,0,1.0
5,6,12,0,0,1.0



T1.4a INFERENTIAL AUDIT COMPLETE
Friedman p = 2.77382e-13
Level 1: mean similarity = 0.738948, Holm p = 0.00146484, models degraded = 12/12
Level 2: mean similarity = 0.631960, Holm p = 0.00146484, models degraded = 12/12
Level 3: mean similarity = 0.582623, Holm p = 0.00146484, models degraded = 12/12
Level 4: mean similarity = 0.433492, Holm p = 0.00146484, models degraded = 12/12
Level 5: mean similarity = 0.490359, Holm p = 0.00146484, models degraded = 12/12
Level 6: mean similarity = 0.458819, Holm p = 0.00146484, models degraded = 12/12


In [24]:
# ================================================================
# T1.4a — CORRECTED WILCOXON TABLE LABELING
# ================================================================

wilcox_rows = []

baseline = pivot[0].to_numpy()

for level in range(1, 7):

    randomized = pivot[level].to_numpy()

    differences = randomized - baseline

    test = stats.wilcoxon(
        randomized,
        baseline,
        alternative="less",
        zero_method="wilcox",
        correction=False,
        method="auto"
    )

    # Rank-biserial effect size
    nonzero = differences[differences != 0]

    if len(nonzero) > 0:

        ranks = stats.rankdata(
            np.abs(nonzero)
        )

        W_plus = ranks[
            nonzero > 0
        ].sum()

        W_minus = ranks[
            nonzero < 0
        ].sum()

        rbc = (
            W_minus - W_plus
        ) / (
            W_minus + W_plus
        )

    else:
        rbc = 0.0

    wilcox_rows.append({
        # --------------------------------------------------------
        # IMPORTANT:
        # This is the RANDOMIZED cascade level being compared
        # against the original level 0.
        # --------------------------------------------------------
        "cascade_level": level,

        "comparison": f"level_0_vs_level_{level}",

        "n_models": len(randomized),

        "mean_similarity": randomized.mean(),

        "median_similarity": np.median(randomized),

        "mean_degradation":
            np.mean(
                baseline - randomized
            ),

        "median_degradation":
            np.median(
                baseline - randomized
            ),

        "wilcoxon_W":
            test.statistic,

        "raw_p":
            test.pvalue,

        "rank_biserial_effect":
            rbc,

        "n_models_degraded":
            int(
                np.sum(
                    randomized < baseline
                )
            ),

        "n_models_equal":
            int(
                np.sum(
                    randomized == baseline
                )
            ),

        "n_models_higher":
            int(
                np.sum(
                    randomized > baseline
                )
            ),
    })

wilcox_df = pd.DataFrame(wilcox_rows)


# ================================================================
# HOLM CORRECTION
# ================================================================

def holm_adjust(p_values):

    p = np.asarray(
        p_values,
        dtype=float
    )

    m = len(p)

    order = np.argsort(p)

    adjusted = np.empty(m)

    for rank, idx in enumerate(order):
        adjusted[idx] = p[idx] * (m - rank)

    # Enforce monotonicity of adjusted p-values
    ordered_adj = adjusted[order]

    ordered_adj = np.maximum.accumulate(
        ordered_adj
    )

    ordered_adj = np.minimum(
        ordered_adj,
        1.0
    )

    result = np.empty(m)

    result[order] = ordered_adj

    return result


wilcox_df["holm_p"] = holm_adjust(
    wilcox_df["raw_p"].to_numpy()
)

wilcox_df["significant_holm_0.05"] = (
    wilcox_df["holm_p"] < 0.05
)


# ================================================================
# PRINT CORRECTED TABLE
# ================================================================

print()
print("=" * 80)
print("PAIRED WILCOXON TESTS — EACH RANDOMIZED LEVEL vs ORIGINAL")
print("=" * 80)

display(
    wilcox_df[
        [
            "cascade_level",
            "comparison",
            "n_models",
            "mean_similarity",
            "median_similarity",
            "mean_degradation",
            "median_degradation",
            "wilcoxon_W",
            "raw_p",
            "holm_p",
            "rank_biserial_effect",
            "n_models_degraded",
            "n_models_equal",
            "n_models_higher",
            "significant_holm_0.05",
        ]
    ]
)


# ================================================================
# SAVE CORRECTED VERSION
# ================================================================

AUDIT_DIR = Path(
    "/kaggle/working/"
    "T1_4a_parameter_randomization_12x100/"
    "T1_4A_STATISTICAL_AUDIT"
)

output_path = (
    AUDIT_DIR /
    "T1_4A_MODEL_LEVEL_WILCOXON_HOLM.csv"
)

wilcox_df.to_csv(
    output_path,
    index=False
)


# ================================================================
# MANUSCRIPT-FRIENDLY SUMMARY
# ================================================================

print()
print("=" * 80)
print("CORRECTED WILCOXON SUMMARY")
print("=" * 80)

for _, row in wilcox_df.iterrows():

    level = int(row["cascade_level"])

    print(
        f"Level 0 vs Level {level}: "
        f"mean = {row['mean_similarity']:.6f}, "
        f"degradation = {row['mean_degradation']:.6f}, "
        f"W = {row['wilcoxon_W']:.1f}, "
        f"raw p = {row['raw_p']:.6g}, "
        f"Holm p = {row['holm_p']:.6g}, "
        f"RBC = {row['rank_biserial_effect']:.3f}, "
        f"{int(row['n_models_degraded'])}/12 degraded"
    )

print()
print(f"Saved corrected table to:")
print(output_path)
print("=" * 80)


PAIRED WILCOXON TESTS — EACH RANDOMIZED LEVEL vs ORIGINAL


,cascade_level,comparison,n_models,mean_similarity,median_similarity,mean_degradation,median_degradation,wilcoxon_W,raw_p,holm_p,rank_biserial_effect,n_models_degraded,n_models_equal,n_models_higher,significant_holm_0.05
0,1,level_0_vs_level_1,12,0.738948,0.738875,0.261052,0.261125,0.0,0.000244,0.001465,1.0,12,0,0,True
1,2,level_0_vs_level_2,12,0.631960,0.632408,0.368040,0.367592,0.0,0.000244,0.001465,1.0,12,0,0,True
2,3,level_0_vs_level_3,12,0.582623,0.582525,0.417377,0.417475,0.0,0.000244,0.001465,1.0,12,0,0,True
3,4,level_0_vs_level_4,12,0.433492,0.432628,0.566508,0.567372,0.0,0.000244,0.001465,1.0,12,0,0,True
4,5,level_0_vs_level_5,12,0.490359,0.491128,0.509641,0.508872,0.0,0.000244,0.001465,1.0,12,0,0,True
5,6,level_0_vs_level_6,12,0.458819,0.458518,0.541181,0.541482,0.0,0.000244,0.001465,1.0,12,0,0,True



CORRECTED WILCOXON SUMMARY
Level 0 vs Level 1: mean = 0.738948, degradation = 0.261052, W = 0.0, raw p = 0.000244141, Holm p = 0.00146484, RBC = 1.000, 12/12 degraded
Level 0 vs Level 2: mean = 0.631960, degradation = 0.368040, W = 0.0, raw p = 0.000244141, Holm p = 0.00146484, RBC = 1.000, 12/12 degraded
Level 0 vs Level 3: mean = 0.582623, degradation = 0.417377, W = 0.0, raw p = 0.000244141, Holm p = 0.00146484, RBC = 1.000, 12/12 degraded
Level 0 vs Level 4: mean = 0.433492, degradation = 0.566508, W = 0.0, raw p = 0.000244141, Holm p = 0.00146484, RBC = 1.000, 12/12 degraded
Level 0 vs Level 5: mean = 0.490359, degradation = 0.509641, W = 0.0, raw p = 0.000244141, Holm p = 0.00146484, RBC = 1.000, 12/12 degraded
Level 0 vs Level 6: mean = 0.458819, degradation = 0.541181, W = 0.0, raw p = 0.000244141, Holm p = 0.00146484, RBC = 1.000, 12/12 degraded

Saved corrected table to:
/kaggle/working/T1_4a_parameter_randomization_12x100/T1_4A_STATISTICAL_AUDIT/T1_4A_MODEL_LEVEL_WILCOXON_H

# T1.4a — Inferential Statistical Interpretation

## Model-level inferential evidence

The parameter-randomization sanity check was evaluated at the **model level** (`n = 12` models), rather than treating individual image-level observations as independent experimental units.

A Friedman test across the seven cascade levels showed a strong overall difference in attribution-map similarity:

- **Friedman statistic:** 70.8214
- **p-value:** 2.77 × 10⁻¹³

Pairwise one-sided Wilcoxon signed-rank tests comparing each randomized cascade level against the original, unrandomized model showed consistent degradation at every cascade level.

| Randomization level | Mean similarity | Mean degradation | Models degraded | Holm-adjusted p | Rank-biserial effect |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.738948 | 0.261052 | 12/12 | 0.001465 | 1.000 |
| 2 | 0.631960 | 0.368040 | 12/12 | 0.001465 | 1.000 |
| 3 | 0.582623 | 0.417377 | 12/12 | 0.001465 | 1.000 |
| 4 | 0.433492 | 0.566508 | 12/12 | 0.001465 | 1.000 |
| 5 | 0.490359 | 0.509641 | 12/12 | 0.001465 | 1.000 |
| 6 | 0.458819 | 0.541181 | 12/12 | 0.001465 | 1.000 |

For every randomized level:

- Wilcoxon statistic `W = 0`
- raw `p = 0.000244`
- Holm-adjusted `p = 0.001465`
- rank-biserial effect size = `1.000`
- all 12/12 model trajectories showed lower similarity than the original model.

The result therefore shows **complete directional consistency at the model level**: every production model exhibited lower attribution-map similarity after each parameter-randomization intervention.

## Interpretation

The parameter-randomization experiment provides evidence that the observed Integrated Gradients attribution maps are dependent on the learned model parameters. Progressive destruction of the learned parameters substantially disrupts similarity to the attribution maps generated by the corresponding trained model.

The mean similarity decreased from `1.000000` for the original model to `0.738948` after the first randomization level and to `0.458819` after the final randomization level.

The trajectory is **not strictly monotonic at every individual cascade step**: similarity increases from level 4 (`0.433492`) to level 5 (`0.490359`) before decreasing again at level 6 (`0.458819`). Therefore, the result is described as a **strong overall degradation across the randomization cascade**, rather than a strictly monotonic decrease.

## Completeness sensitivity

A separate completeness sensitivity analysis identified `508 / 7200` randomized observations (`7.06%`) with completeness failures. These failures were confined to the first randomized cascade level.

The failed and completeness-passing level-1 observations showed no statistically significant difference in similarity (`Mann–Whitney p = 0.1303`). Excluding the completeness-failed observations did not alter the downstream cascade results or any of the 12 model-level level-6 means.

Therefore, the parameter-randomization finding is not dependent on the completeness-failed observations.

## Scope of the conclusion

This sanity check supports **model-parameter dependence of the attribution maps** and therefore strengthens the interpretation that the maps are responsive to learned model structure.

It does **not**, by itself, establish that the attribution maps identify the biologically correct or causal image features. The complementary null-mask and baseline-sensitivity controls are therefore retained as separate Tier-1 XAI sanity checks.

**T1.4a status: COMPLETE — LOCK FOR MANUSCRIPT ANALYSIS.**